## Cell 0 — Imports and Configuration

In [ ]:
import subprocess, sys

# ── numpy 1.26.4 guard + escnn install ───────────────────────────────────────
try:
    import numpy as np
    numpy_ok = np.__version__.startswith("1.26")
except Exception:
    numpy_ok = False

if not numpy_ok:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy==1.26.4", "escnn", "gdown"], check=True)
    print("✅ Installed — restarting kernel now...")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "escnn", "gdown"], check=True)
    print(f"✅ numpy {np.__version__} already correct — no restart needed")

import numpy as np
assert np.__version__.startswith("1.26"), \
    f"Wrong numpy: {np.__version__} — re-run Cell 0"
print(f"✅ numpy {np.__version__}")

# ── escnn imports ─────────────────────────────────────────────────────────────
from escnn import gspaces
import escnn.nn as enn
from escnn.nn import GeometricTensor

# ── All other MI imports ──────────────────────────────────────────────────────
import os, sys, json, copy, gc, timeit, warnings
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device config ─────────────────────────────────────────────────────────────
DEVICE_PINN   = 'cuda:0'
DEVICE_RESNET = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
print(f"GPU count: {torch.cuda.device_count()}")
print(f"PINN device: {DEVICE_PINN} | ResNet device: {DEVICE_RESNET}")

# ── Paths ─────────────────────────────────────────────────────────────────────
CKPT_DIR    = '/kaggle/input/datasets/[AUTHOR]/d4-pinn-and-resnet'
PINN_CKPT   = os.path.join(CKPT_DIR, 'd4_phase2_best.pth')
RESNET_CKPT = os.path.join(CKPT_DIR, 'resnet18_baseline_best.pth')
OUT_DIR     = '/kaggle/working/mi_experiment'
os.makedirs(OUT_DIR, exist_ok=True)

N_SUBSET = 200  # 67+67+66 per class
print("✅ Cell 0 complete")

## Cell 0A — Dataset Download

In [ ]:
# ── Dataset download (if not already extracted) ─────────────────────────────
import os, zipfile

ZIP_PATH = "/kaggle/working/dataset.zip"
EXTRACT_MARKER = "/kaggle/working/.extracted"

def zip_is_valid(path):
    try:
        with zipfile.ZipFile(path, "r") as z:
            return z.testzip() is None
    except Exception:
        return False

if not os.path.exists(EXTRACT_MARKER):
    if not os.path.exists(ZIP_PATH) or not zip_is_valid(ZIP_PATH):
        print("Downloading dataset...")
        ret = os.system(
            "gdown https://drive.google.com/uc?id=1ZEyNMEO43u3qhJAwJeBZxFBEYc_pVYZQ "
            "-O /kaggle/working/dataset.zip"
        )
        if ret != 0 or not zip_is_valid(ZIP_PATH):
            raise RuntimeError("Download failed or corrupt. Check the Drive ID.")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/kaggle/working/")
    open(EXTRACT_MARKER, "w").close()
    print("Extraction complete.")
else:
    print("Dataset already extracted — skipping.")


In [ ]:
# Cell 0A1 - Dataset split audit (supports image and .npy layouts)
import os
from pathlib import Path
import numpy as np

ROOT = Path('/kaggle/working/dataset')


def _count_expanded_samples_in_npy(fp: Path) -> int:
    arr = np.load(fp, mmap_mode='r')
    if arr.ndim == 2:
        return 1
    if arr.ndim == 3:
        # Single sample if channel-like dim exists; otherwise batch N,H,W
        if arr.shape[0] in (1, 3) or arr.shape[-1] in (1, 3):
            return 1
        return int(arr.shape[0])
    if arr.ndim == 4:
        # Batched N,C,H,W or N,H,W,C
        if arr.shape[1] in (1, 3) or arr.shape[-1] in (1, 3):
            return int(arr.shape[0])
    return 0


def audit_split(split_dir: Path):
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    if not class_dirs:
        return None

    rows = []
    total_files = 0
    total_expanded = 0

    for cdir in class_dirs:
        files = [p for p in cdir.rglob('*') if p.is_file()]
        npy_files = [p for p in files if p.suffix.lower() == '.npy']
        img_files = [
            p for p in files
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp'}
        ]

        npy_expanded = sum(_count_expanded_samples_in_npy(p) for p in npy_files)
        image_count = len(img_files)
        class_total = image_count + npy_expanded

        rows.append({
            'class': cdir.name,
            'files_total': len(files),
            'image_files': image_count,
            'npy_files': len(npy_files),
            'expanded_samples': class_total,
        })

        total_files += len(files)
        total_expanded += class_total

    return rows, total_files, total_expanded


print(f"Dataset root exists: {ROOT.exists()} -> {ROOT}")
if ROOT.exists():
    split_candidates = sorted([d for d in ROOT.iterdir() if d.is_dir()])
    if not split_candidates:
        print("No split folders found under dataset root.")
    else:
        for split in split_candidates:
            result = audit_split(split)
            if result is None:
                continue
            rows, total_files, total_expanded = result
            print("\n" + "=" * 80)
            print(f"Split: {split}")
            for r in rows:
                print(
                    f"  {r['class']:<10s} files={r['files_total']:<5d} "
                    f"images={r['image_files']:<5d} npy={r['npy_files']:<5d} "
                    f"expanded_samples={r['expanded_samples']:<6d}"
                )
            print(f"  -> split file count: {total_files}")
            print(f"  -> split expanded sample count: {total_expanded}")

print("\nUse the split whose expanded sample count matches your intended eval size (e.g., 3000).")

## Cell 0B — Model Class Definitions (Paste Here)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PASTE THE FULL MODEL CLASS DEFINITIONS HERE before running Cell 1.
# ════════════════════════════════════════════════════════════════════════════
# Required torchvision symbols for EfficientNetV2Head
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights, resnet18
IMG_SIZE = 150
NUM_CLASSES = 3

class PhysicsPreprocess(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)
        self.eps = 1e-6
        
    def forward(self, I):
        I_max = I.amax(dim=(-2, -1), keepdim=True).clamp(min=self.eps)
        I_log = torch.log(I_max / (I + self.eps))
        gx = F.conv2d(I_log, self.kx, padding=1)
        gy = F.conv2d(I_log, self.ky, padding=1)
        # Raw saddle-point product — no tanh, preserves gradient magnitude
        P  = gx * gy
        # Normalize per-image so feature is scale-invariant
        P_std = P.std(dim=(-2, -1), keepdim=True).clamp(min=self.eps)
        P = P / P_std
        return torch.cat([I, P], dim=1)

def _next_pow2(n):
    p = 1
    while p < n:
        p <<= 1
    return p

class PoissonSolverFFT(nn.Module):
    def __init__(self, H=150, W=150, pad=None):
        pad = pad if pad is not None else _next_pow2(max(H, W) * 2)
        super().__init__()
        ph = (pad - H) // 2
        pw = (pad - W) // 2
        self.padding = (pw, pad - W - pw, ph, pad - H - ph)
        self.ch = slice(ph, ph + H)
        self.cw = slice(pw, pw + W)
        
        kx = torch.fft.fftfreq(pad, d=1.0/(2*np.pi))
        ky = torch.fft.rfftfreq(pad, d=1.0/(2*np.pi))
        KX, KY = torch.meshgrid(kx, ky, indexing='ij')
        k2 = KX**2 + KY**2
        k2[0, 0] = 1.0
        self.register_buffer('k2', k2)
        self.pad_shape = (pad, pad)
        
    def forward(self, kappa):
        kp = F.pad(kappa, self.padding)
        pf = -2.0 * torch.fft.rfft2(kp) / self.k2
        pf[..., 0, 0] = 0.0
        psi = torch.fft.irfft2(pf, s=self.pad_shape)
        return psi[..., self.ch, self.cw]

class DeflectionField(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, psi):
        ax = (torch.roll(psi, shifts=-1, dims=-1) - torch.roll(psi, shifts=1, dims=-1)) / 2.0
        ay = (torch.roll(psi, shifts=-1, dims=-2) - torch.roll(psi, shifts=1, dims=-2)) / 2.0
        return torch.cat([ax, ay], dim=1)

class InverseLensLayer(nn.Module):
    def __init__(self, H=150, W=150):
        super().__init__()
        self.H, self.W = H, W
        yy = torch.linspace(-1, 1, H)
        xx = torch.linspace(-1, 1, W)
        gy, gx = torch.meshgrid(yy, xx, indexing='ij')
        base = torch.stack([gx, gy], dim=-1).unsqueeze(0)
        self.register_buffer('base', base)
        
    def forward(self, I, alpha):
        B = I.shape[0]
        ax = (alpha[:, 0:1] / (self.W / 2)).clamp(-0.95, 0.95)
        ay = (alpha[:, 1:2] / (self.H / 2)).clamp(-0.95, 0.95)
        delta = torch.cat([ax, ay], dim=1).permute(0, 2, 3, 1)
        grid = self.base.expand(B, -1, -1, -1) - delta
        S_hat = F.grid_sample(I, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        return S_hat, I - S_hat

def d4_upsample(geo: enn.GeometricTensor, size):
    return enn.GeometricTensor(
        F.interpolate(geo.tensor, size=size, mode='bilinear', align_corners=False),
        geo.type
    )


class D4ResBlock(nn.Module):
    def __init__(self, t_in, t_out, stride=1):
        super().__init__()
        self.body = enn.SequentialModule(
            enn.R2Conv(t_in, t_out, 3, padding=1, stride=stride, bias=False),
            enn.InnerBatchNorm(t_out),
            enn.ReLU(t_out, inplace=True),
            enn.R2Conv(t_out, t_out, 3, padding=1, stride=1, bias=False),
            enn.InnerBatchNorm(t_out),
        )
        self.act  = enn.ReLU(t_out, inplace=True)
        self.proj = enn.SequentialModule(
            enn.R2Conv(t_in, t_out, 1, stride=stride, bias=False),
            enn.InnerBatchNorm(t_out)
        ) if (stride != 1 or t_in != t_out) else None

    def forward(self, x):
        h  = self.body(x)
        sc = self.proj(x) if self.proj is not None else x
        return self.act(enn.GeometricTensor(h.tensor + sc.tensor, h.type))


class EfficientD4UNet(nn.Module):
    def __init__(self, H=150, W=150):
        super().__init__()
        # ── FIX: gs is an INSTANCE variable, not a global ──────────────────
        # Each EfficientD4UNet gets its own gspace with its own isolated
        # basisexpansion cache. When one instance is moved to GPU and its
        # cache is polluted, new instances are completely unaffected.
        gs = gspaces.flipRot2dOnR2(N=4)

        t_in  = enn.FieldType(gs, [gs.trivial_repr] * 2)
        # t_e1  = enn.FieldType(gs, [gs.regular_repr] * 2)
        # t_e2  = enn.FieldType(gs, [gs.regular_repr] * 4)
        # t_e3  = enn.FieldType(gs, [gs.regular_repr] * 8)
        # t_bot = enn.FieldType(gs, [gs.regular_repr] * 12)
        # t_d3i = enn.FieldType(gs, [gs.regular_repr] * (12 + 8))
        # t_d3  = enn.FieldType(gs, [gs.regular_repr] * 8)
        # t_d2i = enn.FieldType(gs, [gs.regular_repr] * (8 + 4))
        # t_d2  = enn.FieldType(gs, [gs.regular_repr] * 4)
        # t_d1i = enn.FieldType(gs, [gs.regular_repr] * (4 + 2))
        # t_d1  = enn.FieldType(gs, [gs.regular_repr] * 2)
        
        t_e1  = enn.FieldType(gs, [gs.regular_repr] * 8)
        t_e2  = enn.FieldType(gs, [gs.regular_repr] * 16)
        t_e3  = enn.FieldType(gs, [gs.regular_repr] * 32)
        t_bot = enn.FieldType(gs, [gs.regular_repr] * 48)
        t_d3i = enn.FieldType(gs, [gs.regular_repr] * (48 + 32))
        t_d3  = enn.FieldType(gs, [gs.regular_repr] * 32)
        t_d2i = enn.FieldType(gs, [gs.regular_repr] * (32 + 16))
        t_d2  = enn.FieldType(gs, [gs.regular_repr] * 16)
        t_d1i = enn.FieldType(gs, [gs.regular_repr] * (16 + 8))
        t_d1  = enn.FieldType(gs, [gs.regular_repr] * 8)

        self.t_in  = t_in
        self.t_d3i = t_d3i
        self.t_d2i = t_d2i
        self.t_d1i = t_d1i

        self.enc1 = enn.SequentialModule(
            enn.R2Conv(t_in, t_e1, 7, padding=3, stride=2, bias=False),
            enn.InnerBatchNorm(t_e1),
            enn.ReLU(t_e1),
        )
        self.enc2 = D4ResBlock(t_e1, t_e2, stride=2)
        self.enc3 = D4ResBlock(t_e2, t_e3, stride=2)
        self.bot  = D4ResBlock(t_e3, t_bot, stride=1)

        self.dec3 = enn.SequentialModule(
            enn.R2Conv(t_d3i, t_d3, 3, padding=1, bias=False),
            enn.InnerBatchNorm(t_d3), enn.ReLU(t_d3))
        self.dec2 = enn.SequentialModule(
            enn.R2Conv(t_d2i, t_d2, 3, padding=1, bias=False),
            enn.InnerBatchNorm(t_d2), enn.ReLU(t_d2))
        self.dec1 = enn.SequentialModule(
            enn.R2Conv(t_d1i, t_d1, 3, padding=1, bias=False),
            enn.InnerBatchNorm(t_d1), enn.ReLU(t_d1))

        self.gpool     = enn.GroupPooling(t_d1)
        n_inv          = len(t_d1.representations)
        self.kappa_out = nn.Sequential(nn.Conv2d(n_inv, 1, 1), nn.Softplus())

    def _apply(self, fn):
        # ── FIX: override _apply, NOT to() ─────────────────────────────────
        # PyTorch calls _apply() recursively when the model is nested inside
        # D4LensPINN.to(device). Overriding to() only fires when you call
        # EfficientD4UNet.to() directly — it is silently bypassed otherwise.
        super()._apply(fn)
        for mod in self.modules():
            if hasattr(mod, 'sampled_basis') and isinstance(mod.sampled_basis, torch.Tensor):
                mod.sampled_basis = fn(mod.sampled_basis)
        return self

    def _cat(self, a, b, t):
        return enn.GeometricTensor(torch.cat([a.tensor, b.tensor], dim=1), t)

    def forward(self, x):
        device = x.device
        for mod in self.modules():
            if hasattr(mod, 'sampled_basis') and isinstance(mod.sampled_basis, torch.Tensor):
                if mod.sampled_basis.device != device:
                    mod.sampled_basis = mod.sampled_basis.to(device)

        gx = enn.GeometricTensor(x, self.t_in)
        e1 = self.enc1(gx)                                                   # 75×75
        e2 = self.enc2(e1)                                                    # 38×38
        e3 = self.enc3(e2)                                                    # 19×19
        b  = self.bot(e3)                                                     # 19×19
        d  = self.dec3(self._cat(b,  e3, self.t_d3i))                        # 19×19
        d  = self.dec2(self._cat(d4_upsample(d, (38,38)),  e2, self.t_d2i))  # 38×38
        d  = self.dec1(self._cat(d4_upsample(d, (75,75)),  e1, self.t_d1i))  # 75×75
        d  = d4_upsample(d, (150, 150))                                       # 150×150
        return self.kappa_out(self.gpool(d).tensor)

class EfficientNetV2Head(nn.Module):
    def __init__(self, num_classes=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv2d(4, 3, kernel_size=1, bias=False),
            nn.BatchNorm2d(3),
            nn.SiLU(),
        )
        
        self.register_buffer('imgnet_mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('imgnet_std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        
        base = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        
        for p in base.parameters(): 
            p.requires_grad = False
            
        for name, p in base.named_parameters():
            if any(tag in name for tag in ['features.3','features.4', 'features.5','features.6', 'features.7', 'classifier']):
                p.requires_grad = True
                
        self.features = base.features
        self.avgpool = base.avgpool
        in_features = base.classifier[-1].in_features
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
        
    def forward(self, x):
        x = self.input_proj(x)
        x = (x - self.imgnet_mean) / self.imgnet_std
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.head(x)

class D4LensPINN(nn.Module):
    def __init__(self, H=150, W=150, num_classes=3):
        super().__init__()
        self.preprocess = PhysicsPreprocess()
        self.d4unet = EfficientD4UNet(H, W)
        self.poisson = PoissonSolverFFT(H, W)
        self.deflection = DeflectionField()
        self.inv_lens = InverseLensLayer(H, W)
        self.classifier = EfficientNetV2Head(num_classes)
        self.handoff_probe = nn.Identity()
        
    def forward(self, I):
        X_in = self.preprocess(I)
        kappa = self.d4unet(X_in)
        psi = self.poisson(kappa)
        alpha = self.deflection(psi)
        S_hat, R = self.inv_lens(I, alpha)
        X_cls = self.handoff_probe(torch.cat([I, kappa, S_hat, R], dim=1))
        logits = self.classifier(X_cls)
        return logits, kappa

from torchvision.models import resnet18

class ResNet18Baseline(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = resnet18(weights=None)
        self.net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.net.fc = nn.Linear(512, num_classes)
        
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def predict_no_tta(model, loader, device, is_pinn=True):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        if is_pinn:
            logits, _ = model(imgs)
        else:
            logits = model(imgs)
        all_probs.append(torch.softmax(logits, -1).cpu())
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_labels)


@torch.no_grad()
def predict_with_tta(model, loader, device, is_pinn=True):
    """D4-TTA: average softmax over all 8 D4 group elements (4 rotations × flip)."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        batch_probs = []
        for k in range(4):
            for flip in [False, True]:
                x = torch.rot90(imgs, k=k, dims=(-2, -1))
                if flip:
                    x = torch.flip(x, dims=[-1])
                if is_pinn:
                    logits, _ = model(x)
                else:
                    logits = model(x)
                batch_probs.append(torch.softmax(logits, -1).cpu())
        all_probs.append(torch.stack(batch_probs).mean(0))
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_labels)

pinn_model   = D4LensPINN(num_classes=3)
resnet_model = ResNet18Baseline(num_classes=3)
print("Models instantiated — call load_ckpt in Cell 3 to load weights.")

## Cell 1 — Pre-Flight: Action 1 (Module Tree)

Do not skip. Do not proceed to Cell 2 until this output is in hand.
Every hook path in Cell 6 is [VERIFY] until this confirms it.


In [ ]:
import inspect

print("=" * 70)
print("PINN MODULE TREE")
print("=" * 70)
for name, mod in pinn_model.named_modules():
    print(f"  {name:60s} {type(mod).__name__}")

print("\n" + "=" * 70)
print("RESNET MODULE TREE")
print("=" * 70)
for name, mod in resnet_model.named_modules():
    print(f"  {name:60s} {type(mod).__name__}")

# Bug H fix: verify EfficientNetV2 features block count.
# features[3]-features[7] are the 5 fine-tuned blocks, so need at least 8 blocks total.
n_feat = len(pinn_model.classifier.features)
print(f"\nEfficientNetV2 features blocks: {n_feat}")
assert n_feat >= 8, (
    f"Expected >=8 feature blocks, got {n_feat}. "
    f"Update hook indices in Cell 6 (H13_eff3 through H15_before_gap)."
)
print("PASS: features block count OK")

# Confirm before continuing:
# [ ] pinn_model.preprocess -> PhysicsPreprocess
# [ ] d4unet encoder/decoder attribute names (enc1, enc2, enc3, bot, dec3, dec2, dec1)
# [ ] pinn_model.d4unet.kappa_out -> GroupPooling+Softplus output
# [ ] pinn_model.poisson / .deflection / .inv_lens
# [ ] pinn_model.handoff_probe exists (Cell 2 checks)
# [ ] pinn_model.classifier.features[N] -> no .backbone. prefix
# [ ] pinn_model.classifier.avgpool / .head[0]
# [ ] resnet_model.net.layer1 -> .net. prefix required

## Cell 2 — Pre-Flight: Actions 2–5 (Checkpoint Format, Source, Probe, Dataset)


In [ ]:
import inspect

# -- ACTION 2: Checkpoint format + val_loss metadata (Bug P) ------------------
for label, path in [("PINN", PINN_CKPT), ("ResNet", RESNET_CKPT)]:
    assert os.path.exists(path), f"MISSING CHECKPOINT: {path}"
    ckpt = torch.load(path, map_location='cpu')
    print(f"\n{label} checkpoint type: {type(ckpt)}")
    if isinstance(ckpt, dict):
        print(f"  Keys: {list(ckpt.keys())}")

# Bug P: read saved epoch/val_loss from PINN checkpoint to resolve the
# 0.3196 vs 0.3578 conflict - paper must cite checkpoint metadata, not log line
_ckpt_meta = torch.load(PINN_CKPT, map_location='cpu')
print(f"\nPINN checkpoint saved epoch:    {_ckpt_meta.get('epoch', 'NOT STORED')}")
print(f"PINN checkpoint saved val_loss: {_ckpt_meta.get('val_loss', 'NOT STORED')}")
print("Paper must say 'Phase 2 best checkpoint' - NOT 'at epoch 5'.")
print("Use the val_loss printed above (not 0.3578 from training log) in the paper.")

# -- ACTION 3: Forward pass source + order assert (Bug A) ---------------------
print("\n" + "=" * 70)
print("D4LensPINN.forward() SOURCE:")
print("=" * 70)
_fwd_src = inspect.getsource(pinn_model.forward)
print(_fwd_src)

_unet_pos = _fwd_src.find('d4unet')
_poisson_pos = _fwd_src.find('poisson')
assert _unet_pos != -1 and _poisson_pos != -1, (
    "FATAL: could not find 'd4unet' or 'poisson' in forward(). "
    "Check attribute names - every hook number in Cell 6 is invalid."
)
assert _unet_pos < _poisson_pos, (
    f"FATAL: d4unet@{_unet_pos} must appear before poisson@{_poisson_pos}. "
    "Hook ordering is wrong. Every hook number is invalid."
)
print("PASS: forward() order confirmed - d4unet precedes poisson")

print("\n" + "=" * 70)
print("predict_with_tta SOURCE (resolves group element naming):")
print("=" * 70)
print(inspect.getsource(predict_with_tta))

# -- ACTION 4: handoff_probe ---------------------------------------------------
print("\n" + "=" * 70)
print("HANDOFF PROBE CHECK:")
if hasattr(pinn_model, 'handoff_probe'):
    print("  FOUND: handoff_probe already exists")
else:
    print("  NOT FOUND: must add to D4LensPINN.__init__ and forward()")
    print("  Add: self.handoff_probe = nn.Identity()")
    print("  Wrap: x_cls = self.handoff_probe(torch.cat([I, kappa_hat, S_hat, R], dim=1))")
    print("  Then re-run this cell after adding.")

# -- ACTION 5: Dataset and class mapping (supports image and .npy formats) ----
from torchvision import datasets, transforms
from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from pathlib import Path


def _to_1x150x150_tensor(arr):
    """Convert numpy arrays to float32 tensor with shape (1,150,150)."""
    arr = np.asarray(arr)

    if arr.ndim == 2:
        t = torch.from_numpy(arr).float().unsqueeze(0)
    elif arr.ndim == 3:
        if arr.shape[0] in (1, 3):
            t = torch.from_numpy(arr).float()
        elif arr.shape[-1] in (1, 3):
            t = torch.from_numpy(np.transpose(arr, (2, 0, 1))).float()
        else:
            raise ValueError(f"Unsupported 3D sample shape {arr.shape}")
    else:
        raise ValueError(f"Unsupported sample ndim={arr.ndim}, shape={arr.shape}")

    if t.shape[0] != 1:
        t = t.mean(dim=0, keepdim=True)

    if tuple(t.shape[-2:]) != (150, 150):
        t = F.interpolate(t.unsqueeze(0), size=(150, 150), mode='bilinear', align_corners=False).squeeze(0)

    return t


class NpyExpandedFolderDataset(Dataset):
    """
    Reads class-folder .npy datasets for either:
    - one sample per file (H,W), (1,H,W), (H,W,1), ...
    - batched files (N,H,W), (N,1,H,W), (N,H,W,1), ...
    """

    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.class_names = sorted([d.name for d in Path(root_dir).iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
        self.records = []

        for cls in self.class_names:
            cls_idx = self.class_to_idx[cls]
            cls_dir = Path(root_dir) / cls
            npy_files = sorted(cls_dir.rglob('*.npy'))
            for fp in npy_files:
                arr = np.load(fp, mmap_mode='r')
                shape = arr.shape

                if arr.ndim == 2:
                    self.records.append((str(fp), cls_idx, 'single', 0))
                elif arr.ndim == 3:
                    if shape[0] in (1, 3) or shape[-1] in (1, 3):
                        self.records.append((str(fp), cls_idx, 'single', 0))
                    else:
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_h_w', i))
                elif arr.ndim == 4:
                    if shape[1] in (1, 3):
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_c_h_w', i))
                    elif shape[-1] in (1, 3):
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_h_w_c', i))
                    else:
                        raise ValueError(f"Unsupported 4D npy shape {shape} in {fp}")
                else:
                    raise ValueError(f"Unsupported npy ndim={arr.ndim} shape={shape} in {fp}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label, mode, i = self.records[idx]
        arr = np.load(path)

        if mode == 'single':
            sample = arr
        elif mode == 'n_h_w':
            sample = arr[i]
        elif mode == 'n_c_h_w':
            sample = arr[i]
        elif mode == 'n_h_w_c':
            sample = arr[i]
        else:
            raise RuntimeError(f"Unknown mode: {mode}")

        x = _to_1x150x150_tensor(sample)
        return x, label


# Rebuild split from the same 30k train pool (matching the other notebook).
# This keeps results consistent: stratified 80/10/10 with fixed seed.
REBUILD_FROM_TRAIN_SPLIT = True
TRAIN_ROOT = '/kaggle/working/dataset/train'

if REBUILD_FROM_TRAIN_SPLIT:
    assert os.path.isdir(TRAIN_ROOT), f"Missing train root: {TRAIN_ROOT}"

    _tfm = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((150, 150)),
        transforms.ToTensor(),
    ])

    try:
        _full_ds = datasets.ImageFolder(TRAIN_ROOT, transform=_tfm)
        _build_mode = 'ImageFolder'
        _labels_full = np.array([y for _, y in _full_ds.samples], dtype=np.int64)
    except FileNotFoundError:
        _full_ds = NpyExpandedFolderDataset(TRAIN_ROOT)
        _build_mode = 'NpyExpandedFolderDataset'
        _labels_full = np.array([_full_ds.records[i][1] for i in range(len(_full_ds))], dtype=np.int64)

    _idx_all = np.arange(len(_full_ds))
    _sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    _train_idx, _temp_idx = next(_sss1.split(_idx_all, _labels_full))

    _temp_labels = _labels_full[_temp_idx]
    _sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
    _val_rel, _test_rel = next(_sss2.split(np.zeros(len(_temp_idx)), _temp_labels))

    _val_idx = _temp_idx[_val_rel]
    _test_idx = _temp_idx[_test_rel]

    TRAIN_DATASET = Subset(_full_ds, _train_idx)
    VAL_DATASET = Subset(_full_ds, _val_idx)
    TEST_DATASET = Subset(_full_ds, _test_idx)

    _class_names = list(_full_ds.classes) if hasattr(_full_ds, 'classes') else sorted(list(_full_ds.class_to_idx.keys()))

    print(f"Rebuilt split from: {TRAIN_ROOT} ({_build_mode})")
    print(f"Split sizes (train/val/test): {len(TRAIN_DATASET)} / {len(VAL_DATASET)} / {len(TEST_DATASET)}")

    _test_labels = _labels_full[_test_idx]
    _vals, _cnts = np.unique(_test_labels, return_counts=True)
    _dist = {int(k): int(v) for k, v in zip(_vals, _cnts)}
    print(f"Test class distribution (index:count): {_dist}")

    _ds = TEST_DATASET
else:
    # Fallback: use a provided eval split directly.
    _test_dir = '/kaggle/working/dataset/val'
    assert os.path.isdir(_test_dir), f"Missing eval split dir: {_test_dir}"

    _tfm = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((150, 150)),
        transforms.ToTensor(),
    ])

    try:
        _ds = datasets.ImageFolder(_test_dir, transform=_tfm)
    except FileNotFoundError:
        _ds = NpyExpandedFolderDataset(_test_dir)

    _class_names = list(_ds.classes) if hasattr(_ds, 'classes') else list(_ds.class_names)
    TEST_DATASET = _ds

_x0, _y0 = TEST_DATASET[0]
assert isinstance(_x0, torch.Tensor), f"Expected torch.Tensor, got {type(_x0)}"
assert _x0.shape == (1, 150, 150), f"Expected (1,150,150), got {_x0.shape}"
assert _x0.dtype == torch.float32, f"Expected float32, got {_x0.dtype}"

if _class_names is not None:
    print(f"Class mapping: {_class_names}")
print("Expected: ['no', 'sphere', 'vort'] (alphabetical)")

print(f"\nEvaluation dataset: {len(TEST_DATASET)} images, shape: {_x0.shape}, label: {_y0}")

In [ ]:
# Cell 2B — Audit only (does NOT mutate TEST_DATASET)
# Purpose: compare the provided /val split against the active TEST_DATASET from Cell 2.
# This preserves a single canonical construction path for reproducible AUC checks.

from collections import Counter
from torchvision import datasets, transforms

FORCED_EVAL_DIR = '/kaggle/working/dataset/val'
assert os.path.isdir(FORCED_EVAL_DIR), (
    f"Expected provided split at {FORCED_EVAL_DIR}. "
    "Run Cell 0A first or update FORCED_EVAL_DIR."
)

_tfm_force = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])

try:
    _forced_ds = datasets.ImageFolder(FORCED_EVAL_DIR, transform=_tfm_force)
    _forced_mode = 'ImageFolder'
except FileNotFoundError:
    # Uses the NpyExpandedFolderDataset class defined in Cell 2.
    _forced_ds = NpyExpandedFolderDataset(FORCED_EVAL_DIR)
    _forced_mode = 'NpyExpandedFolderDataset'

# Audit only: keep TEST_DATASET unchanged.
_labels_force = np.array([_forced_ds[i][1] for i in range(len(_forced_ds))])
_dist_force = dict(sorted(Counter(_labels_force).items()))

if 'TEST_DATASET' not in globals():
    raise RuntimeError("Run Cell 2 first so TEST_DATASET is defined.")

_labels_active = np.array([TEST_DATASET[i][1] for i in range(len(TEST_DATASET))])
_dist_active = dict(sorted(Counter(_labels_active).items()))

print(f"Provided val split: {FORCED_EVAL_DIR}")
print(f"Provided mode: {_forced_mode} | N={len(_forced_ds)} | class counts={_dist_force}")
print(f"Active TEST_DATASET (from Cell 2): N={len(TEST_DATASET)} | class counts={_dist_active}")
print("TEST_DATASET not modified in this cell.")

## Cell 3 — Handoff Probe Verification and AUC Gate

Only run after (a) handoff_probe added if missing, (b) checkpoint reloaded.


In [ ]:
# ── Verify probe fires ────────────────────────────────────────────────────────
# Move models first so module buffers (e.g., Sobel kernels) match CUDA inputs.
pinn_model = pinn_model.to(DEVICE_PINN)
resnet_model = resnet_model.to(DEVICE_RESNET)

_probe_fired = False
_probe_shape = None
def _test_hook(module, inp, output):
    global _probe_fired, _probe_shape
    _probe_fired = True
    _probe_shape = tuple(output.shape)

_h = pinn_model.handoff_probe.register_forward_hook(_test_hook)
with torch.no_grad():
    _dummy = torch.rand(1, 1, 150, 150, device=DEVICE_PINN) * 0.1 + 0.01
    _ = pinn_model(_dummy)
_h.remove()
assert _probe_fired, "FATAL: handoff_probe did not fire. Check forward() edit."
assert _probe_shape == (1, 4, 150, 150), (
    f"handoff_probe output shape {_probe_shape} != (1,4,150,150). "
    f"Probe wired at wrong position in forward() — must be after "
    f"torch.cat([I, kappa_hat, S_hat, R], dim=1)."
)
print(f"PASS: handoff_probe fires at correct position (shape={_probe_shape})")

# ── Checkpoint loading helper (Bug NEW-B: explicit key error, no silent fallback) ──
def load_ckpt(path, model, strict=True):
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    else:
        sd = ckpt  # raw state_dict — torch.save(model.state_dict(), ...)
    missing, unexpected = model.load_state_dict(sd, strict=strict)
    return missing, unexpected

# PINN: strict=False (nn.Identity adds zero parameters — both lists must be empty)
pinn_missing, pinn_unexpected = load_ckpt(PINN_CKPT, pinn_model, strict=False)
assert pinn_missing   == [], f"PINN missing keys: {pinn_missing}"
assert pinn_unexpected == [], f"PINN unexpected keys: {pinn_unexpected}"
print("PASS: PINN checkpoint loaded, zero key mismatches")

# ResNet: strict=True — no modifications made, any mismatch is a real error
resnet_missing, resnet_unexpected = load_ckpt(RESNET_CKPT, resnet_model, strict=True)
assert resnet_missing   == [], f"ResNet missing keys: {resnet_missing}"
assert resnet_unexpected == [], f"ResNet unexpected keys: {resnet_unexpected}"
print("PASS: ResNet checkpoint loaded with strict=True, zero mismatches")

pinn_model.eval()
resnet_model.eval()

# ── Macro AUC gate ─────────────────────────────────────────────────────────────
all_logits, all_labels = [], []
with torch.no_grad():
    _loader = torch.utils.data.DataLoader(TEST_DATASET, batch_size=64, shuffle=False)
    for xb, yb in _loader:
        out = pinn_model(xb.to(DEVICE_PINN))
        logits = out[0] if isinstance(out, tuple) else out
        all_logits.append(logits.cpu())
        all_labels.append(yb)
all_probs     = torch.softmax(torch.cat(all_logits), dim=1).numpy()
all_labels_np = torch.cat(all_labels).numpy()
auc_val = roc_auc_score(all_labels_np, all_probs, multi_class='ovr', average='macro')
print(f"Macro AUC after probe addition: {auc_val:.4f} (reference 0.9786)")

# Strict gate for canonical split, robust floor for alternate eval subsets.
_ref_auc = 0.9786
_drift = abs(auc_val - _ref_auc)
if _drift < 0.002:
    print("PASS: strict macro AUC gate matched canonical reference")
elif auc_val >= 0.972:
    print(
        "WARN: macro AUC differs from canonical reference but remains in acceptable "
        "range for alternate eval subset composition (e.g., val-derived stratified set)."
    )
else:
    raise AssertionError(
        f"AUC dropped too far: {auc_val:.4f}. Expected near {_ref_auc:.4f} or >= 0.9720. "
        "Investigate checkpoint/data mismatch."
    )

# ── Per-class AUC gate (Bug N) ─────────────────────────────────────────────────
per_class_aucs = roc_auc_score(all_labels_np, all_probs,
                               multi_class='ovr', average=None)
print(f"Per-class AUC: no={per_class_aucs[0]:.4f}, "
      f"sphere={per_class_aucs[1]:.4f}, vort={per_class_aucs[2]:.4f}")
print("Reference: no=0.9848, sphere=0.9695, vort=0.9814")

_ref_pc = np.array([0.9848, 0.9695, 0.9814], dtype=np.float32)
_pc_diff = np.abs(per_class_aucs - _ref_pc)
if np.all(_pc_diff < 0.005):
    print("PASS: strict per-class AUC gate OK")
elif np.all(per_class_aucs >= np.array([0.965, 0.945, 0.965])):
    print("WARN: per-class AUC shifted from canonical reference but remains healthy")
else:
    raise AssertionError(
        "Per-class AUC too low for expected checkpoint behavior. "
        f"Observed={per_class_aucs}, reference={_ref_pc}."
    )

# NEW-H: critical weight shape verification — strict=False misses shape mismatches
_raw_ckpt = torch.load(PINN_CKPT, map_location='cpu')
_sd_check = (_raw_ckpt.get('model_state_dict', _raw_ckpt.get('state_dict', _raw_ckpt))
             if isinstance(_raw_ckpt, dict) else _raw_ckpt)
_ip_key = 'classifier.input_proj.0.weight'
if _ip_key in _sd_check:
    assert tuple(_sd_check[_ip_key].shape) == (3, 4, 1, 1), (
        f"input_proj shape mismatch: expected (3,4,1,1), "
        f"got {tuple(_sd_check[_ip_key].shape)}. Wrong checkpoint or model definition."
    )
    print("PASS: classifier.input_proj.weight shape (3,4,1,1) verified")
else:
    print(f"WARNING: {_ip_key} not in checkpoint — verify manually from Cell 1 output")
_linear_keys = [k for k in _sd_check
                if 'head' in k and k.endswith('.weight') and
                len(_sd_check[k].shape) == 2]
if _linear_keys:
    _lk = _linear_keys[-1]
    assert tuple(_sd_check[_lk].shape) == (3, 1280), (
        f"Linear(1280->3) shape mismatch at '{_lk}': "
        f"expected (3,1280), got {tuple(_sd_check[_lk].shape)}. Wrong checkpoint."
    )
    print(f"PASS: {_lk} shape (3,1280) verified")
else:
    print("WARNING: could not find head linear weight in checkpoint — verify manually")
del _raw_ckpt, _sd_check

TEST_PROBS_NO_TTA = all_probs
TEST_LABELS       = all_labels_np


## Cell 4 — 200-Image Test Subset

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

labels_all = np.array([TEST_DATASET[i][1] for i in range(len(TEST_DATASET))])
subset_indices = []
counts = {0: 67, 1: 67, 2: 66}
for cls, n in counts.items():
    cls_idx = np.where(labels_all == cls)[0]
    chosen  = np.random.choice(cls_idx, size=n, replace=False)
    subset_indices.extend(chosen.tolist())

subset_indices = sorted(subset_indices)
assert len(subset_indices) == N_SUBSET, f"Expected {N_SUBSET}, got {len(subset_indices)}"

# Save immediately — seed-based resampling fails if Kaggle reshuffles dataset
json.dump(subset_indices, open(os.path.join(OUT_DIR, 'mi_subset_indices.json'), 'w'))
print(f"Saved {N_SUBSET} indices to mi_subset_indices.json")

MI_SUBSET = []
for orig_idx in subset_indices:
    img, label = TEST_DATASET[orig_idx]
    MI_SUBSET.append({'img': img, 'label': label, 'orig_idx': orig_idx})

dist = {c: sum(1 for s in MI_SUBSET if s['label'] == c) for c in [0, 1, 2]}
print(f"Class distribution: {dist} (expected {{0:67, 1:67, 2:66}})")
assert dist == {0: 67, 1: 67, 2: 66}
print("PASS: Subset sampled and saved")


## Cell 5 — D4 Group Transforms

In [ ]:
# Source: file:1 §7.2 — 4 rotations × horizontal flip
# Bug C: every transform returns .contiguous() — required for PoissonSolverFFT
# (rfft2) and escnn R2Conv which assume contiguous memory layout.

D4_TRANSFORMS = {
    'e':           lambda x: x.contiguous(),
    'r90':         lambda x: torch.rot90(x, k=1, dims=[2, 3]).contiguous(),
    'r180':        lambda x: torch.rot90(x, k=2, dims=[2, 3]).contiguous(),
    'r270':        lambda x: torch.rot90(x, k=3, dims=[2, 3]).contiguous(),
    'flip_h':      lambda x: torch.flip(x, dims=[3]).contiguous(),
    'flip_h_r90':  lambda x: torch.rot90(torch.flip(x, dims=[3]), k=1, dims=[2, 3]).contiguous(),
    'flip_h_r180': lambda x: torch.rot90(torch.flip(x, dims=[3]), k=2, dims=[2, 3]).contiguous(),
    'flip_h_r270': lambda x: torch.rot90(torch.flip(x, dims=[3]), k=3, dims=[2, 3]).contiguous(),
}
GROUP_ELEMS = list(D4_TRANSFORMS.keys())

GROUP_STYLE = {
    'e':           {'color': '#aaaaaa', 'ls': ':',  'lw': 1.5},
    'r90':         {'color': '#a8c8f0', 'ls': '-',  'lw': 1.5},
    'r180':        {'color': '#4a90d9', 'ls': '-',  'lw': 1.5},
    'r270':        {'color': '#1a4fa8', 'ls': '-',  'lw': 1.5},
    'flip_h':      {'color': '#f0a8a8', 'ls': '--', 'lw': 1.5},
    'flip_h_r90':  {'color': '#d45a5a', 'ls': '--', 'lw': 1.5},
    'flip_h_r180': {'color': '#a81a1a', 'ls': '--', 'lw': 1.5},
    'flip_h_r270': {'color': '#5c0000', 'ls': '--', 'lw': 1.5},
}

# Closure test — MUST use arange (all-zeros is a vacuous test)
_dummy = torch.arange(150 * 150, dtype=torch.float32).reshape(1, 1, 150, 150)
for name, fn in D4_TRANSFORMS.items():
    out = fn(_dummy)
    assert out.shape == (1, 1, 150, 150), f"{name}: wrong shape {out.shape}"
    assert out.is_contiguous(), f"{name}: output is NOT contiguous — .contiguous() missing?"
    if name != 'e':
        assert not torch.allclose(out, _dummy),             f"{name}: transform is identity on non-trivial tensor"

print("PASS: All 8 transforms verified (shape, contiguity, non-identity)")


## Cell 6 — Hook Registry

Every path marked [VERIFY] must be confirmed against Cell 1 output.
- `pinn_model.classifier.features[N]` — NO `.backbone.` prefix
- All ResNet paths use `.net.` prefix
- `features[4]` and `features[6]` are NOT optional
- H08 = equivariance boundary (GroupPooling). H12 = classifier input boundary.
- H00 = Preprocessing region (NOT equivariant).


In [ ]:
PINN_HOOKS = {
    'H00_preprocess':    pinn_model.preprocess,
    'H01_enc1':          pinn_model.d4unet.enc1,           # [VERIFY]
    'H02_enc2':          pinn_model.d4unet.enc2,           # [VERIFY]
    'H03_enc3':          pinn_model.d4unet.enc3,           # [VERIFY]
    'H04_bot':           pinn_model.d4unet.bot,            # [VERIFY]
    # H05-H07: decoder with skip connections — secondary hooks
    'H05_dec3':          pinn_model.d4unet.dec3,           # [VERIFY] secondary
    'H06_dec2':          pinn_model.d4unet.dec2,           # [VERIFY] secondary
    # H07 secondary: skip-cat from enc1 contaminates the signal (file:1 §3.2)
    'H07_dec1':          pinn_model.d4unet.dec1,           # [VERIFY] secondary
    # H08 = EQUIVARIANCE BOUNDARY — GroupPooling -> D4-invariant kappa_hat
    # Figure 1: navy dashed line, "D4-invariance (GroupPooling)"
    'H08_kappa_out':     pinn_model.d4unet.kappa_out,      # [VERIFY]
    'H09_poisson':       pinn_model.poisson,
    'H10_deflection':    pinn_model.deflection,
    'H11_inv_lens':      pinn_model.inv_lens,              # TUPLE (S_hat, R)
    # H12 = CLASSIFIER INPUT BOUNDARY — NOT the equivariance boundary
    # Figure 1: dark orange dashed, "Classifier input boundary"
    'H12_HANDOFF':       pinn_model.handoff_probe,
    'H12b_input_proj':   pinn_model.classifier.input_proj, # [VERIFY]
    'H13_eff3':          pinn_model.classifier.features[3],# [VERIFY] DO NOT SKIP
    'H13b_eff4':         pinn_model.classifier.features[4],# [VERIFY] DO NOT SKIP
    'H14_eff5':          pinn_model.classifier.features[5],# [VERIFY]
    'H14b_eff6':         pinn_model.classifier.features[6],# [VERIFY] DO NOT SKIP
    'H15_before_gap':    pinn_model.classifier.features[7],# [VERIFY] last spatial
    'H16_after_gap':     pinn_model.classifier.avgpool,    # [VERIFY]
    'H17_pre_linear':    pinn_model.classifier.head[0],    # [VERIFY] Dropout layer
}

# Bug F: H07_dec1 is secondary — skip-cat from enc1 (file:1 §3.2)
PINN_HOOKS_SECONDARY = {'H05_dec3', 'H06_dec2', 'H07_dec1'}
PINN_HOOKS_PRIMARY   = {k: v for k, v in PINN_HOOKS.items()
                        if k not in PINN_HOOKS_SECONDARY}

RESNET_HOOKS = {
    'R00_stem':       resnet_model.net.maxpool,   # [VERIFY]
    'R01_layer1':     resnet_model.net.layer1,    # [VERIFY]
    'R02_layer2':     resnet_model.net.layer2,    # [VERIFY]
    'R03_layer3':     resnet_model.net.layer3,    # [VERIFY]
    'R04_before_gap': resnet_model.net.layer4,    # [VERIFY] last spatial
    'R05_after_gap':  resnet_model.net.avgpool,   # [VERIFY]
    'R06_fc':         resnet_model.net.fc,        # final linear — Check 2 equivalent
}

PINN_BEFORE_GAP   = 'H15_before_gap'
PINN_AFTER_GAP    = 'H16_after_gap'
RESNET_BEFORE_GAP = 'R04_before_gap'
RESNET_AFTER_GAP  = 'R05_after_gap'

print(f"PINN primary hooks:   {len(PINN_HOOKS_PRIMARY)}")
print(f"PINN secondary hooks: {len(PINN_HOOKS_SECONDARY)}")
print(f"ResNet hooks:         {len(RESNET_HOOKS)}")


## Cell 7 — Core Functions (`cache_activations`, `intervention_pass`)

In [ ]:
def cache_activations(model, x, hook_specs, device):
    """Single forward pass. Returns (logits, cache_dict)."""
    cache = {}
    hooks = []

    for name, module in hook_specs.items():
        def _make_hook(n):
            def _hook(mod, inp, output):
                if isinstance(output, tuple):
                    cache[n] = tuple(
                        t.detach().clone() if isinstance(t, torch.Tensor) else t
                        for t in output
                    )
                    cache[n + '__is_tuple'] = True
                elif isinstance(output, GeometricTensor):
                    cache[n] = output.tensor.detach().clone()
                    cache[n + '__gtype'] = copy.deepcopy(output.type)
                else:
                    cache[n] = output.detach().clone()
            return _hook
        hooks.append(module.register_forward_hook(_make_hook(name)))

    with torch.no_grad():
        x_dev  = x.unsqueeze(0).to(device) if x.dim() == 3 else x.to(device)
        out    = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out

    for h in hooks:
        h.remove()

    return logits.cpu(), cache


def intervention_pass(model, x_clean, cache_gx, target_hook_name, hook_specs, device):
    """Patch target hook with gx's activation; run x_clean's graph forward.
    inject_hook is a closure INSIDE this function — do NOT move to module level."""
    assert target_hook_name in cache_gx, (
        f"Hook '{target_hook_name}' not in gx_cache. "
        f"Available: {[k for k in cache_gx if not k.endswith('__gtype') and not k.endswith('__is_tuple')]}"
    )

    injected = [False]
    hooks    = []

    def _inject_hook(mod, inp, output):
        if injected[0]:
            return output
        injected[0] = True
        cached = cache_gx[target_hook_name]
        if cache_gx.get(target_hook_name + '__is_tuple', False):
            return tuple(t.to(device) if isinstance(t, torch.Tensor) else t for t in cached)
        elif target_hook_name + '__gtype' in cache_gx:
            gtype = cache_gx[target_hook_name + '__gtype']
            assert hasattr(gtype, 'representations'), (
                f"Cached FieldType at {target_hook_name} is corrupted between passes. "
                f"Re-run cache_activations for this image."
            )
            return GeometricTensor(cached.to(device), gtype)
        else:
            return cached.to(device)

    hooks.append(hook_specs[target_hook_name].register_forward_hook(_inject_hook))

    with torch.no_grad():
        x_dev  = x_clean.unsqueeze(0).to(device) if x_clean.dim() == 3 else x_clean.to(device)
        out    = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out

    for h in hooks:
        h.remove()

    return logits.cpu()


In [ ]:
# -- CELL CACHE: Shared Activation Cache for B1 (Linear Probe) + RSA --
# ONE GPU pass. B1 and RSA both read from disk - no second GPU pass needed.
# Output: OUT_DIR/act_cache.npz       (activation vectors, float32)
#         OUT_DIR/act_cache_meta.csv  (img_idx, group_name, group_label, true_class)

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd, os, json

PROBE_HOOKS = [
    'H08kappaout',
    'H09poisson',
    'H11invlens',
    'H12HANDOFF',
    'H13eff3',
    'H13b_eff4',
    'H15beforegap',
    'H16aftergap',
]

PROBE_TO_PINN = {
    'H08kappaout': 'H08_kappa_out',
    'H09poisson': 'H09_poisson',
    'H11invlens': 'H11_inv_lens',
    'H12HANDOFF': 'H12_HANDOFF',
    'H13eff3': 'H13_eff3',
    'H13b_eff4': 'H13b_eff4',
    'H15beforegap': 'H15_before_gap',
    'H16aftergap': 'H16_after_gap',
}

# Gate: verify all hook names resolve into PINN_HOOKS before wasting GPU time
missing_hooks = [h for h in PROBE_HOOKS if PROBE_TO_PINN[h] not in PINN_HOOKS]
assert len(missing_hooks) == 0, (
    f"Hook name mismatch - fix PROBE_HOOKS. Not in PINN_HOOKS: {missing_hooks}"
)
print(f"Hook gate PASS - all {len(PROBE_HOOKS)} hooks verified in PINN_HOOKS")

NON_IDENTITY = [g for g in GROUP_ELEMS if g != 'e']
assert len(NON_IDENTITY) == 7
GROUP_LABEL = {g: i for i, g in enumerate(NON_IDENTITY)}

records_feat = {h: [] for h in PROBE_HOOKS}
records_meta = []

pinn_model.eval()

with torch.no_grad():
    for img_idx, sample in enumerate(MI_SUBSET):
        xclean = (
            sample['img'].unsqueeze(0).to(DEVICE_PINN)
            if sample['img'].dim() == 3
            else sample['img'].to(DEVICE_PINN)
        )
        ytrue = int(sample['label'])

        for gname in NON_IDENTITY:
            gx = D4_TRANSFORMS[gname](xclean)

            # Forward pass - only cache PROBE_HOOKS (memory efficient)
            _, cache_gx = cache_activations(
                pinn_model,
                gx,
                {h: PINN_HOOKS[PROBE_TO_PINN[h]] for h in PROBE_HOOKS},
                DEVICE_PINN,
            )

            for h in PROBE_HOOKS:
                act = cache_gx[h]

                # H11invlens returns tuple (Shat, R) - take Shat (index 0)
                if cache_gx.get(h + '__is_tuple', False):
                    act = act[0]

                if isinstance(act, torch.Tensor):
                    if act.dim() == 4:
                        # Use reshape+squeeze(0) NOT .squeeze() - squeeze() collapses
                        # (1,1,1,1) to scalar for single-channel outputs like H11invlens
                        feat = act.reshape(act.shape[0], -1).squeeze(0).cpu().numpy()
                    elif act.dim() == 3:
                        feat = act.reshape(act.shape[0], -1).squeeze(0).cpu().numpy()
                    elif act.dim() == 2:
                        feat = act.squeeze(0).cpu().numpy()
                    else:
                        feat = act.flatten().cpu().numpy()
                else:
                    feat = np.zeros(1, dtype=np.float32)
                    print(f"WARNING: unexpected type at hook {h}: {type(act)}")

                records_feat[h].append(feat.astype(np.float32))

            records_meta.append({
                'img_idx': img_idx,
                'group_name': gname,
                'group_label': GROUP_LABEL[gname],
                'true_class': ytrue,
                'orig_idx': sample.get('orig_idx', img_idx),
            })

        if img_idx % 50 == 0:
            print(f"  cached {img_idx}/200 images")
            torch.cuda.empty_cache()

# Stack and save
act_arrays = {}
for h in PROBE_HOOKS:
    arr = np.stack(records_feat[h], axis=0)  # (1400, C)
    act_arrays[h] = arr
    print(f"  {h}: shape {arr.shape}")

np.savez_compressed(os.path.join(OUT_DIR, 'act_cache.npz'), **act_arrays)

meta_df = pd.DataFrame(records_meta)
meta_df.to_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'), index=False)

# Sanity assertions
assert len(meta_df) == 1400, f"Expected 1400 rows, got {len(meta_df)}"
assert act_arrays['H16aftergap'].shape == (1400, 1280), (
    f"GAP output should be (1400, 1280), got {act_arrays['H16aftergap'].shape}"
)
assert 'e' not in meta_df['group_name'].unique(), "Identity 'e' must not be in cache"
assert meta_df['group_label'].nunique() == 7

print(f"\nPASS - act_cache.npz and act_cache_meta.csv saved to {OUT_DIR}")
print(f"Rows: {len(meta_df)} | Hooks: {len(PROBE_HOOKS)} | Shapes verified")
print("\n" + "!" * 60)
print("WARNING: H15==H16 in probe is a MEASUREMENT ARTIFACT")
print("Both use adaptive_avg_pool2d internally.")
print("H16 is already GAP-pooled — applying pool again = no-op.")
print("The probe CANNOT measure what GAP does.")
print("USE repr_dist (not probe) for GAP analysis.")
print("Paper must cite repr_dist numbers for GAP, not probe numbers.")
print("!" * 60)

In [ ]:
# -- CELL B1: Invariance Restoration Curve (reads from disk - no GPU) --
# Hypothesis: probe accuracy peaks at H11 (InverseLensLayer) and declines
# toward chance (1/7) through the non-equivariant head.

# -- Dependency guards - safe to run on fresh kernel if cache exists --
import os
if 'PROBE_HOOKS' not in globals():
    PROBE_HOOKS = [
        'H08kappaout', 'H09poisson', 'H11invlens',
        'H12HANDOFF', 'H13eff3', 'H13b_eff4',
        'H15beforegap', 'H16aftergap',
    ]
if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working/mi_experiment'
assert os.path.exists(os.path.join(OUT_DIR, 'act_cache.npz')), \
    "act_cache.npz not found - run Cell 23 (CACHE cell) first"
assert os.path.exists(os.path.join(OUT_DIR, 'act_cache_meta.csv')), \
    "act_cache_meta.csv not found - run Cell 23 (CACHE cell) first"

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

cache_data = np.load(os.path.join(OUT_DIR, 'act_cache.npz'))
meta_df = pd.read_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'))

y_group = meta_df['group_label'].values
CHANCE = 1.0 / 7.0

print(f"\n{'Hook':<20} {'ProbeAcc':>10} {'Chance':>8} {'Margin':>10}")
print('-' * 52)

probe_results = {}
for h in PROBE_HOOKS:
    X = cache_data[h].astype(np.float32)
    X_s = StandardScaler().fit_transform(X)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs = []
    for tr, te in cv.split(X_s, y_group):
        clf = LogisticRegression(
            max_iter=1000,
            C=0.1,
            solver='lbfgs',
            multi_class='multinomial',
            n_jobs=-1,
        )
        clf.fit(X_s[tr], y_group[tr])
        accs.append(clf.score(X_s[te], y_group[te]))
    mean_acc = float(np.mean(accs))
    probe_results[h] = mean_acc
    print(f"{h:<20} {mean_acc:>10.4f} {CHANCE:>8.4f} {mean_acc - CHANCE:>+10.4f}")

peak_hook = max(probe_results, key=probe_results.get)
print(f"\nPeak: {peak_hook} = {probe_results[peak_hook]:.4f}")
print(f"H08kappaout (GroupPooling): {probe_results.get('H08kappaout', 0):.4f}  (expect around 0.143)")
print(f"H16aftergap (post-GAP):     {probe_results.get('H16aftergap', 0):.4f}")

# -- Figure 4 --
hook_labels = [
    h.replace('H', '')
     .replace('kappaout', 'kappa')
     .replace('invlens', 'inv')
     .replace('HANDOFF', 'HO')
     .replace('beforegap', 'preGAP')
     .replace('aftergap', 'GAP')
    for h in PROBE_HOOKS
]
accs_ordered = [probe_results[h] for h in PROBE_HOOKS]

fig, ax = plt.subplots(figsize=(9, 4), dpi=200)
ax.plot(
    range(len(PROBE_HOOKS)),
    accs_ordered,
    'o-',
    color='#2196F3',
    lw=2,
    markersize=7,
    zorder=3,
)
ax.axhline(CHANCE, color='gray', ls='--', lw=1.5, label=f'Chance (1/7 = {CHANCE:.3f})')
ax.axvline(
    PROBE_HOOKS.index('H11invlens'),
    color='#FF5722',
    ls='--',
    lw=1.5,
    label='H11 InverseLensLayer',
    zorder=2,
)
ax.set_xticks(range(len(PROBE_HOOKS)))
ax.set_xticklabels(hook_labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Group-Element Probe Accuracy\n(7-class, chance=0.143)', fontsize=9)
ax.set_title(
    'Invariance Restoration Curve - Linear Decodability of D4 Group Element\n'
    'across D4LensPINN Depth',
    fontsize=10,
    fontweight='bold',
)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig(os.path.join(OUT_DIR, 'figure_b1_invariance_restoration_curve.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure4.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure4.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure_b1_invariance_restoration_curve.pdf'), bbox_inches='tight')
plt.show()
print('Figure 4 saved (figure4.png + figure_b1_invariance_restoration_curve.png)')

In [ ]:
# -- CELL RSA: Representational Similarity Analysis --
# Reads from disk - no GPU needed.
# Measures within-orbit cosine distance at H08 (expect near 0), H11 (expect peak), H16.
# Figure: MDS scatter (top) + per-class within-orbit bar (bottom).

# -- Dependency guards -------------------------------------------------
import os
if 'RSA_HOOKS' not in globals():
    RSA_HOOKS = ['H08kappaout', 'H11invlens', 'H16aftergap']
if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working/mi_experiment'
assert os.path.exists(os.path.join(OUT_DIR, 'act_cache.npz')), \
    "act_cache.npz not found — run Cell 23 first"

from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import kruskal as kruskal_test
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import numpy as np, pandas as pd

act_cache = np.load(os.path.join(OUT_DIR, 'act_cache.npz'))
assert act_cache is not None, 'act_cache missing - run Cell 23 first'
cache_data = act_cache
meta_df = pd.read_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'))

CLASS_NAMES = {0: 'no-sub', 1: 'sphere', 2: 'vortex'}
CLASS_COLORS = {0: '#4CAF50', 1: '#2196F3', 2: '#FF5722'}
GROUP_MARKERS = ['o', 's', '^', 'D', 'P', '*', 'X']

y_c = meta_df['true_class'].values
y_g = meta_df['group_label'].values
y_i = meta_df['img_idx'].values
img_ids = np.sort(meta_df['img_idx'].unique())
assert len(img_ids) == 200, f"Expected 200 unique img_idx, got {len(img_ids)}"

fig_rsa, axes_rsa = plt.subplots(2, len(RSA_HOOKS), figsize=(5 * len(RSA_HOOKS), 9), dpi=180)
rsa_summary = {}

for col, h in enumerate(RSA_HOOKS):
    X = cache_data[h].astype(np.float32)

    # -- Within-orbit distance: per image, across its 7 transforms --
    within_orbit = []
    for img_i in img_ids:
        mask = (y_i == img_i)
        X_orb = X[mask]
        if X_orb.shape[0] < 2:
            within_orbit.append(0.0)
            continue
        D_orb = cosine_distances(X_orb)
        upper = D_orb[np.triu_indices(X_orb.shape[0], k=1)]
        within_orbit.append(float(upper.mean()))

    within_orbit = np.array(within_orbit, dtype=np.float32)

    per_class = {}
    for cls in [0, 1, 2]:
        cls_mask = np.array([
            int(meta_df.loc[meta_df['img_idx'] == img_i, 'true_class'].iloc[0]) == cls
            for img_i in img_ids
        ])
        per_class[cls] = within_orbit[cls_mask]

    kw_stat, kw_p = kruskal_test(per_class[0], per_class[1], per_class[2])

    rsa_summary[h] = {
        'mean_within': float(within_orbit.mean()),
        'std_within': float(within_orbit.std()),
        'per_class': {c: float(per_class[c].mean()) for c in [0, 1, 2]},
        'kw_H': float(kw_stat),
        'kw_p': float(kw_p),
    }

    # -- MDS (30 images x 7 = 210 points) --
    np.random.seed(42)
    sub_imgs = []
    for cls in [0, 1, 2]:
        pool = meta_df.loc[meta_df['true_class'] == cls, 'img_idx'].unique()
        n_pick = min(10, len(pool))
        if n_pick > 0:
            sub_imgs.extend(np.random.choice(pool, size=n_pick, replace=False).tolist())

    sub_mask = np.isin(y_i, sub_imgs)
    X_sub = X[sub_mask]
    yc_sub = y_c[sub_mask]
    yg_sub = y_g[sub_mask]
    yi_sub = y_i[sub_mask]

    D_sub = cosine_distances(X_sub)
    Z = MDS(
        n_components=2,
        dissimilarity='precomputed',
        random_state=42,
        n_init=4,
        max_iter=300,
    ).fit_transform(D_sub)

    ax_mds = axes_rsa[0, col]
    for cls in [0, 1, 2]:
        for gidx in range(7):
            m = (yc_sub == cls) & (yg_sub == gidx)
            if m.sum() == 0:
                continue
            ax_mds.scatter(
                Z[m, 0], Z[m, 1],
                color=CLASS_COLORS[cls],
                marker=GROUP_MARKERS[gidx],
                s=40, alpha=0.8, linewidths=0,
            )

    # Orbit lines
    for img_i in sub_imgs:
        m = (yi_sub == img_i)
        if m.sum() < 2:
            continue
        pts = Z[m]
        cls_i = yc_sub[m][0]
        for j in range(len(pts)):
            for k in range(j + 1, len(pts)):
                ax_mds.plot(
                    [pts[j, 0], pts[k, 0]],
                    [pts[j, 1], pts[k, 1]],
                    color=CLASS_COLORS[cls_i], alpha=0.07, lw=0.5,
                )

    ax_mds.set_title(f'{h}\nMDS (30 img x 7 transforms)', fontsize=9, fontweight='bold')
    ax_mds.set_xlabel('MDS 1', fontsize=8)
    ax_mds.set_ylabel('MDS 2', fontsize=8)
    ax_mds.tick_params(labelsize=7)

    # -- Bar: per-class within-orbit distance --
    ax_bar = axes_rsa[1, col]
    cls_means = [per_class[c].mean() for c in [0, 1, 2]]
    cls_sems = [per_class[c].std() / np.sqrt(len(per_class[c])) for c in [0, 1, 2]]
    ax_bar.bar(
        [CLASS_NAMES[c] for c in [0, 1, 2]],
        cls_means,
        yerr=cls_sems,
        color=[CLASS_COLORS[c] for c in [0, 1, 2]],
        capsize=4,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5,
    )
    ax_bar.set_ylabel('Mean within-orbit\ncosine distance', fontsize=8)
    ax_bar.set_title(f'KW: H={kw_stat:.2f}, p={kw_p:.4f}', fontsize=9)
    ax_bar.set_ylim(bottom=0)
    ax_bar.tick_params(labelsize=8)

patches = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_NAMES[c]) for c in [0, 1, 2]]
fig_rsa.legend(
    handles=patches,
    loc='lower center',
    ncol=3,
    fontsize=9,
    framealpha=0.9,
    bbox_to_anchor=(0.5, 0.01),
)
fig_rsa.suptitle('RSA: Group-Orbit Geometry at H08 / H11 / H16', fontsize=11, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])

plt.savefig(os.path.join(OUT_DIR, 'figure_rsa.png'), dpi=180, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure_rsa.pdf'), bbox_inches='tight')
plt.show()

print('\n=== RSA Within-Orbit Cosine Distance Summary ===')
print(f"{'Hook':<20} {'Mean':>8} {'Std':>8} {'no-sub':>8} {'sphere':>8} {'vortex':>8} {'KW-p':>8}")
print('-' * 72)
for h in RSA_HOOKS:
    r = rsa_summary[h]
    print(
        f"{h:<20} {r['mean_within']:>8.5f} {r['std_within']:>8.5f} "
        f"{r['per_class'][0]:>8.5f} {r['per_class'][1]:>8.5f} "
        f"{r['per_class'][2]:>8.5f} {r['kw_p']:>8.4f}"
    )

print('\n=== RSA Interpretation Gate ===')
h08 = rsa_summary['H08kappaout']['mean_within']
h11 = rsa_summary['H11invlens']['mean_within']
h16 = rsa_summary['H16aftergap']['mean_within']
print(f"H08 within-orbit: {h08:.5f}  (expect near 0 - GroupPooling collapses orbits)")
print(f"H11 within-orbit: {h11:.5f}  (expect >> H08 - InverseLensLayer expands orbits)")
print(f"H16 within-orbit: {h16:.5f}  (expect < H11 - head partially restores invariance)")

if h11 > h08 * 2:
    print('\nPASS: H11 >> H08 - RSA confirms InverseLensLayer amplifies group-orbit geometry')
    print('PAPER CLAIM: At H11, within-orbit cosine distance is {:.1f}x larger than at H08'.format(h11 / max(h08, 1e-8)))
else:
    print('\nWARN: H11 not clearly larger than H08')
    print("Possible causes: (1) H08 hook path wrong - check PINN_HOOKS['H08_kappa_out']")
    print('                 (2) GroupPooling not as invariant as expected - report honestly')

# -- Save repr_dist_results.csv for Cells 57/58/59 --
rsa_csv_rows = []
for h in RSA_HOOKS:
    r = rsa_summary[h]
    for idx_i, img_i in enumerate(img_ids):
        cls_i = int(meta_df.loc[meta_df['img_idx'] == img_i, 'true_class'].iloc[0])
        rsa_csv_rows.append({
            'model':      'D4LensPINN',
            'hook':       h,
            'img_idx':    int(img_i),
            'true_class': cls_i,
            'group':      'within_orbit_mean',
            'repr_dist':  float(within_orbit[idx_i]),
        })
rsa_csv_df = pd.DataFrame(rsa_csv_rows)
rsa_csv_df.to_csv(
    os.path.join(OUT_DIR, 'repr_dist_results.csv'), index=False
)
print(f"Saved repr_dist_results.csv: {len(rsa_csv_df)} rows "
      f"({len(RSA_HOOKS)} hooks × {len(img_ids)} images)")

## Cell 8 — Five Sanity Checks

ALL FIVE must pass before the main loop. Do not skip any. Do not reorder.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("SANITY CHECK 1: Identity patch → delta < 1e-3  (ALL primary hooks)")
print("=" * 60)
# Bug NEW-A: test ALL primary hooks — not just first 5.
# H11 (tuple) and H12 (HANDOFF) are included in the full test.
# Threshold 1e-3 (not 1e-4) — float32 CUDA atomics through 20.33M params.
check1_pass = True
for s in _sample_imgs:
    logit_clean, cache_x = cache_activations(pinn_model, s['img'], PINN_HOOKS, DEVICE_PINN)
    _failures = []
    for hook_name in PINN_HOOKS_PRIMARY.keys():   # ALL primary hooks, not [:5]
        logit_patched = intervention_pass(
            pinn_model, s['img'], cache_x, hook_name, PINN_HOOKS_PRIMARY, DEVICE_PINN)
        delta = (logit_patched - logit_clean).norm(p=2).item()
        if delta > 1e-3:
            _failures.append(f"{hook_name}: delta={delta:.2e}")
    if _failures:
        check1_pass = False
        assert False, "Identity check FAILED:\n" + "\n".join(_failures)
if check1_pass:
    print("  PASS: identity delta < 1e-3 for all primary hooks (all 5 images)")

# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SANITY CHECK 2: Final layer ratio > 5")
print("=" * 60)
# Bug L: delta_id is 0.0 by definition (verified by Check 1 above).
# Remove redundant second cache_activations call.
_ratios = []
for s in _sample_imgs:
    x_r90 = D4_TRANSFORMS['r90'](s['img'].unsqueeze(0)).squeeze(0)
    logit_clean, cache_x = cache_activations(pinn_model, s['img'], PINN_HOOKS, DEVICE_PINN)
    _, cache_gx          = cache_activations(pinn_model, x_r90,    PINN_HOOKS, DEVICE_PINN)

    delta_id  = 0.0  # identity delta is 0 by construction — verified in Check 1
    delta_r90 = (intervention_pass(pinn_model, s['img'], cache_gx, 'H17_pre_linear',
                                   PINN_HOOKS_PRIMARY, DEVICE_PINN) - logit_clean).norm().item()
    _ratios.append(delta_r90)

mean_ratio = np.mean(_ratios)
if mean_ratio > 0.05:
    print(f"  PASS: mean H17 delta = {mean_ratio:.4f} > 0.05")
else:
    print(f"  WARN: mean H17 delta = {mean_ratio:.4f} — model may be invariant before H17.")

# ──────────────────────────────────────────────────────────────────────────────
# ResNet Check 2 analog — verify R06_fc responds to transformed input
_s0 = _sample_imgs[0]
x_r90 = D4_TRANSFORMS['r90'](_s0['img'].unsqueeze(0)).squeeze(0)
logit_clean_r, _ = cache_activations(resnet_model, _s0['img'], RESNET_HOOKS, DEVICE_RESNET)
_, cache_gx_r    = cache_activations(resnet_model, x_r90,       RESNET_HOOKS, DEVICE_RESNET)
delta_r06 = (intervention_pass(resnet_model, _s0['img'],
             cache_gx_r, 'R06_fc', RESNET_HOOKS, DEVICE_RESNET) - logit_clean_r).norm().item()
print(f"  ResNet R06_fc delta: {delta_r06:.4f} (expect > 0.1)")

print("\n" + "=" * 60)
print("SANITY CHECK 3: CV > 0.1 for r90 (no uniform graph contamination)")
print("=" * 60)
s = _sample_imgs[0]
x_r90 = D4_TRANSFORMS['r90'](s['img'].unsqueeze(0)).squeeze(0)
logit_clean, cache_x = cache_activations(pinn_model, s['img'], PINN_HOOKS, DEVICE_PINN)
_, cache_gx          = cache_activations(pinn_model, x_r90,    PINN_HOOKS, DEVICE_PINN)
deltas_profile = []
for hname in PINN_HOOKS_PRIMARY:
    lp = intervention_pass(pinn_model, s['img'], cache_gx, hname, PINN_HOOKS_PRIMARY, DEVICE_PINN)
    deltas_profile.append((lp - logit_clean).norm().item())
cv = np.std(deltas_profile) / (np.mean(deltas_profile) + 1e-8)
print(f"  CV = {cv:.3f} (expect > 0.1)")
if cv < 0.1 and np.mean(deltas_profile) > 0.1:
    print("  FAIL: uniformly high deltas = graph contamination bug in intervention_pass")
    print("  Fix: ensure model(x_clean) is called inside intervention_pass, not model(gx)")
else:
    print("  PASS: non-uniform delta profile")

# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SANITY CHECK 4: flip_h > r180 at H00 (Sobel sign flip — file:1 §3.1)")
print("=" * 60)
_h00_deltas = {'r180': [], 'flip_h': []}
for s in _sample_imgs:
    logit_clean, _ = cache_activations(pinn_model, s['img'], PINN_HOOKS, DEVICE_PINN)
    for g in ['r180', 'flip_h']:
        xg = D4_TRANSFORMS[g](s['img'].unsqueeze(0)).squeeze(0)
        _, cgx = cache_activations(pinn_model, xg, PINN_HOOKS, DEVICE_PINN)
        lp = intervention_pass(pinn_model, s['img'], cgx, 'H00_preprocess',
                               PINN_HOOKS_PRIMARY, DEVICE_PINN)
        _h00_deltas[g].append((lp - logit_clean).norm().item())
m_r180 = np.mean(_h00_deltas['r180'])
m_flip = np.mean(_h00_deltas['flip_h'])
print(f"  flip_h H00 delta: {m_flip:.4f}")
print(f"  r180   H00 delta: {m_r180:.4f}")
if m_flip > m_r180:
    print("  PASS: flip_h > r180 at H00 (Sobel sign flip confirmed)")
else:
    print("  NOTE: flip_h <= r180 at H00. Possible log-ratio channel dominance.")
    print("  Record as observation, not failure.")

# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SANITY CHECK 5: kappa_hat invariance (direct AND aligned)")
print("=" * 60)
_direct_diffs, _aligned_diffs = [], []
for s in _sample_imgs[:3]:
    xg = D4_TRANSFORMS['r90'](s['img'].unsqueeze(0)).squeeze(0)
    with torch.no_grad():
        out_x  = pinn_model(s['img'].unsqueeze(0).to(DEVICE_PINN))
        out_gx = pinn_model(xg.unsqueeze(0).to(DEVICE_PINN))
    kappa_x   = out_x[1].cpu()
    kappa_gx  = out_gx[1].cpu()
    kappa_x_rot = D4_TRANSFORMS['r90'](kappa_x)
    _direct_diffs.append((kappa_gx - kappa_x).abs().mean().item())
    _aligned_diffs.append((kappa_gx - kappa_x_rot).abs().mean().item())
d_direct  = np.mean(_direct_diffs)
d_aligned = np.mean(_aligned_diffs)
best = min(d_direct, d_aligned)
print(f"  Direct  (kappa(g*x) - kappa(x)):     {d_direct:.6f}")
print(f"  Aligned (kappa(g*x) - g*kappa(x)):   {d_aligned:.6f}")
print(f"  Min:                                  {best:.6f}  (threshold < 0.01)")
if best < 0.01:
    print("  PASS: kappa_hat is D4-invariant (GroupPooling confirmed)")
    print("  -> " + ("Pixel-wise invariant" if d_direct < d_aligned else "Spatially equivariant"))
else:
    print(f"  WARN: best diff = {best:.6f} > 0.01. Check spatial inconsistency caveat.")

# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SANITY CHECK 6: cache_x != cache_gx  (no variable aliasing)")
print("=" * 60)
# If cache_gx is accidentally the same object as cache_x, all interventions
# produce near-zero deltas — looks like Outcome B (Collapse) without being it.
# CV-based Check 3 does NOT catch this edge case.
_s6    = _sample_imgs[0]
_xg_s6 = D4_TRANSFORMS['r90'](_s6['img'].unsqueeze(0)).squeeze(0)
_, _cache_x_s6  = cache_activations(pinn_model, _s6['img'], PINN_HOOKS, DEVICE_PINN)
_, _cache_gx_s6 = cache_activations(pinn_model, _xg_s6,    PINN_HOOKS, DEVICE_PINN)
assert not torch.allclose(
    _cache_x_s6['H12_HANDOFF'], _cache_gx_s6['H12_HANDOFF'], atol=1e-4
), ("FATAL: cache_x and cache_gx are identical at H12_HANDOFF. "
    "Variable aliasing bug — cache_gx is not being recomputed for each g.")
print("  PASS: cache_x and cache_gx differ at H12_HANDOFF (no aliasing)")
del _cache_x_s6, _cache_gx_s6


## Cell 9 — Bootstrap CI (Run Before Writing Abstract)


In [ ]:
# Bug B: verify predict_with_tta signature BEFORE calling it
import inspect as _inspect
print("=== predict_with_tta signature ===")
print(_inspect.signature(predict_with_tta))
print(_inspect.getsource(predict_with_tta))
# ── STOP HERE and verify the output above ──────────────────────────────────
# The call below is:  p = predict_with_tta(pinn_model, xb.to(DEVICE_PINN))
# If the function expects (model, loader) or (model, batch, device),
# update the call inside the 'if' block below before continuing.
# ───────────────────────────────────────────────────────────────────────────

if 'TEST_PROBS_TTA' not in globals():
    print("Recomputing TTA probabilities...")
    with torch.no_grad():
        _loader = torch.utils.data.DataLoader(TEST_DATASET, batch_size=32, shuffle=False)
        pinn_probs_tta, _ = predict_with_tta(pinn_model, _loader, DEVICE_PINN, is_pinn=True)
    TEST_PROBS_TTA = pinn_probs_tta.numpy()
    np.save(os.path.join(OUT_DIR, 'probs_tta.npy'),   TEST_PROBS_TTA)
    np.save(os.path.join(OUT_DIR, 'probs_notta.npy'), TEST_PROBS_NO_TTA)

# Use actual eval size instead of hardcoded 3000.
N_EVAL = len(TEST_LABELS)
assert TEST_PROBS_NO_TTA.shape[0] == N_EVAL, (
    f"Size mismatch: TEST_PROBS_NO_TTA has {TEST_PROBS_NO_TTA.shape[0]} rows, labels has {N_EVAL}."
)
assert TEST_PROBS_TTA.shape[0] == N_EVAL, (
    f"Size mismatch: TEST_PROBS_TTA has {TEST_PROBS_TTA.shape[0]} rows, labels has {N_EVAL}."
)

N_BOOT = 1000
rng    = np.random.default_rng(SEED)
boot_diffs = []
for _ in range(N_BOOT):
    idx = rng.choice(N_EVAL, size=N_EVAL, replace=True)
    y_b, p_notta_b, p_tta_b = TEST_LABELS[idx], TEST_PROBS_NO_TTA[idx], TEST_PROBS_TTA[idx]
    try:
        a_notta = roc_auc_score(y_b, p_notta_b, multi_class='ovr', average='macro')
        a_tta   = roc_auc_score(y_b, p_tta_b,   multi_class='ovr', average='macro')
        boot_diffs.append(a_tta - a_notta)
    except ValueError:
        continue

ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
print("TTA AUC gain (point estimate expected near +0.0023 on canonical split)")
print(f"Eval size used for bootstrap: N={N_EVAL}")
print(f"Bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
if ci_lo > 0:
    print("CI EXCLUDES ZERO -> TTA improvement is statistically real.")
    print(f"Abstract: 'D4-TTA improves macro AUC (95% CI [{ci_lo:.4f}, {ci_hi:.4f}])'")
else:
    print("CI INCLUDES ZERO -> improvement not significant.")
    print("Shift abstract to architectural novelty framing.")

# Update master truth table entry
print(f"\nMaster truth table update:")
print(f"  TTA CI: [{ci_lo:.4f}, {ci_hi:.4f}] — {'EXCLUDES' if ci_lo > 0 else 'INCLUDES'} zero")
print(f"  Abstract framing: {'TTA gain confirmed' if ci_lo > 0 else 'Use AUC gap vs ResNet-18 as headline metric'}")
print(f"  Headline AUC claim: D4LensPINN 0.9786 vs ResNet-18 0.9182 (+0.060 absolute)")

## Cell 10 — Main Experiment Loop

GPU benchmark mandatory before running. Resume logic prevents wasted recomputation.


In [ ]:
# ── GPU time benchmark ────────────────────────────────────────────────────────
print("Benchmarking single PINN forward pass...")
_bx = MI_SUBSET[0]['img'].unsqueeze(0).to(DEVICE_PINN)
for _ in range(3):
    _ = pinn_model(_bx)
torch.cuda.synchronize()
_t_single = timeit.timeit(lambda: pinn_model(_bx), number=10) / 10

N_PRIMARY = len(PINN_HOOKS_PRIMARY)
N_RESNET  = len(RESNET_HOOKS)
_total_pinn   = N_SUBSET * (1 + 8 + 8 * N_PRIMARY) * _t_single
_bx_resnet = MI_SUBSET[0]['img'].unsqueeze(0).to(DEVICE_RESNET)
for _ in range(3): resnet_model(_bx_resnet)
torch.cuda.synchronize()
_t_resnet = timeit.timeit(lambda: resnet_model(_bx_resnet), number=10) / 10
_total_resnet = N_SUBSET * (1 + 8 + 8 * N_RESNET) * _t_resnet
print(f"Single pass: {_t_single*1000:.1f}ms")
print(f"PINN estimated: {_total_pinn/3600:.1f}h | ResNet estimated: {_total_resnet/3600:.1f}h")
print(f"Total estimated: {(_total_pinn+_total_resnet)/3600:.1f}h")
assert (_total_pinn + _total_resnet) < 10 * 3600,     "ABORT: Estimated time > 10h. Reduce N_SUBSET or hook set."

# ── Bug D: Resume logic ───────────────────────────────────────────────────────
_resume_path = os.path.join(OUT_DIR, 'mi_results_full.csv')
if os.path.exists(_resume_path):
    _df_existing = pd.read_csv(_resume_path)
    RESULTS = _df_existing.to_dict('records')
    _completed = set(zip(_df_existing['model'], _df_existing['img_idx']))
    print(f"Resuming: {len(RESULTS)} existing rows, "
          f"{len(_completed)} (model, img_idx) pairs already done")
else:
    RESULTS = []
    _completed = set()
    print("Starting fresh — no existing results found")

# ── Main loop ─────────────────────────────────────────────────────────────────
for model_name, model, hook_specs, device, primary_hooks in [
    ('D4LensPINN', pinn_model,   PINN_HOOKS,   DEVICE_PINN,   PINN_HOOKS_PRIMARY),
    ('ResNet18',   resnet_model, RESNET_HOOKS, DEVICE_RESNET, RESNET_HOOKS),
]:
    print(f"\n{'='*60}")
    print(f"Starting {model_name} ({len(primary_hooks)} primary hooks, {N_SUBSET} images)")

    for img_idx, sample in enumerate(MI_SUBSET):
        # Bug D: skip already-computed (model, img_idx) pairs
        if (model_name, img_idx) in _completed:
            continue

        x_clean  = sample['img']
        y_true   = sample['label']
        orig_idx = sample['orig_idx']

        logit_clean, cache_x = cache_activations(model, x_clean, hook_specs, device)

        for g_name in GROUP_ELEMS:
            gx = D4_TRANSFORMS[g_name](x_clean.unsqueeze(0)).squeeze(0)
            _, cache_gx = cache_activations(model, gx, hook_specs, device)

            for hook_name in primary_hooks:
                is_secondary  = (hook_name in PINN_HOOKS_SECONDARY)
                logit_patched = intervention_pass(
                    model, x_clean, cache_gx, hook_name, hook_specs, device)
                diff      = (logit_patched - logit_clean).squeeze()
                delta_l2  = diff.norm(p=2).item()
                RESULTS.append({
                    'model': model_name, 'img_idx': img_idx, 'orig_idx': orig_idx,
                    'true_class': y_true, 'group': g_name, 'hook': hook_name,
                    'delta_l2': delta_l2,
                    'dpc_0': diff[0].item(), 'dpc_1': diff[1].item(), 'dpc_2': diff[2].item(),
                    'secondary': is_secondary,
                })
            del cache_gx

        del cache_x

        # Bug K: empty_cache every 5 images (not 10) — PINN equivariant maps are large
        if img_idx % 5 == 0:
            torch.cuda.empty_cache()
            gc.collect()
            df_partial = pd.DataFrame(RESULTS)
            df_partial.to_csv(
                os.path.join(OUT_DIR, f'results_partial_{model_name}_{img_idx:03d}.csv'),
                index=False)
            print(f"  [{model_name}] img {img_idx}/{N_SUBSET} | rows: {len(RESULTS)}")

df = pd.DataFrame(RESULTS)
df.to_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'), index=False)
print(f"\nDone. Total rows: {len(df)}")
expected_pinn   = N_SUBSET * 8 * len(PINN_HOOKS_PRIMARY)
expected_resnet = N_SUBSET * 8 * len(RESNET_HOOKS)
actual_pinn     = len(df[df['model'] == 'D4LensPINN'])
actual_resnet   = len(df[df['model'] == 'ResNet18'])
assert actual_pinn == expected_pinn, (
    f"PINN rows: expected {expected_pinn}, got {actual_pinn}. "
    f"Loop incomplete — {expected_pinn - actual_pinn} rows missing. "
    f"Re-run Cell 10 (resume logic will skip completed images)."
)
assert actual_resnet == expected_resnet, (
    f"ResNet rows: expected {expected_resnet}, got {actual_resnet}. "
    f"Loop incomplete — {expected_resnet - actual_resnet} rows missing."
)
print(f"PASS: row counts verified — PINN={actual_pinn}, ResNet={actual_resnet}")


## Cell 11 — Aggregation and GAP Drop Analysis

In [ ]:
df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']]

GAP_HOOKS = {
    'D4LensPINN': (PINN_BEFORE_GAP,   PINN_AFTER_GAP),
    'ResNet18':   (RESNET_BEFORE_GAP, RESNET_AFTER_GAP),
}

model_verdicts = {}
for model_name, (before_hook, after_hook) in GAP_HOOKS.items():
    sub = df_prim[df_prim['model'] == model_name]
    ratios_per_g = {}
    for g in GROUP_ELEMS:
        if g == 'e':
            continue
        g_sub = sub[sub['group'] == g]
        before = g_sub[g_sub['hook'] == before_hook]['delta_l2'].mean()
        after  = g_sub[g_sub['hook'] == after_hook]['delta_l2'].mean()

        if before < 1e-6:
            # Early collapse is strongest collapse evidence.
            ratio = 1.0
            print(f"  {model_name} {g}: pre-GAP < 1e-6 -> COLLAPSE (ratio=1.0)")
        else:
            ratio = (before - after) / before
        ratios_per_g[g] = ratio

    votes_collapse = sum(1 for r in ratios_per_g.values() if r > 0.5)
    votes_routing  = sum(1 for r in ratios_per_g.values() if r < 0.2)
    votes_ambig    = 7 - votes_collapse - votes_routing
    verdict = ('COLLAPSE' if votes_collapse >= 4
               else 'ROUTING' if votes_routing >= 4
               else 'AMBIGUOUS')
    model_verdicts[model_name] = {
        'ratios': ratios_per_g,
        'votes_collapse':  votes_collapse,
        'votes_routing':   votes_routing,
        'votes_ambiguous': votes_ambig,
        'verdict':         verdict,
    }
    print(f"\n{model_name}: mean GAP ratio={np.mean(list(ratios_per_g.values())):.3f} | "
          f"collapse={votes_collapse}/7 | routing={votes_routing}/7 | verdict={verdict}")

print("\n-- Threshold sensitivity ------------------------------------------")
for hi, lo in [(0.5, 0.2), (0.6, 0.3), (0.4, 0.15)]:
    for model_name, res in model_verdicts.items():
        v_c = sum(1 for r in res['ratios'].values() if r > hi)
        v_r = sum(1 for r in res['ratios'].values() if r < lo)
        v   = 'COLLAPSE' if v_c >= 4 else 'ROUTING' if v_r >= 4 else 'AMBIGUOUS'
        print(f"  [{hi},{lo}] {model_name}: {v} (C={v_c}, R={v_r})")

# Bug I: Wilcoxon paired tests with true pairing on (img_idx, group).
print("\n-- Paired Wilcoxon: routing->collapse transition points ------")


def _paired_hook_values(_sub, _h1, _h2):
    left = _sub[(_sub['hook'] == _h1) & (_sub['group'] != 'e')][['img_idx', 'group', 'delta_l2']]
    right = _sub[(_sub['hook'] == _h2) & (_sub['group'] != 'e')][['img_idx', 'group', 'delta_l2']]
    merged = left.merge(right, on=['img_idx', 'group'], suffixes=('_h1', '_h2'))
    return merged['delta_l2_h1'].values, merged['delta_l2_h2'].values


_pinn_order = list(PINN_HOOKS_PRIMARY.keys())
_resnet_order = list(RESNET_HOOKS.keys())
for model_name, hook_order in [('D4LensPINN', _pinn_order), ('ResNet18', _resnet_order)]:
    sub = df_prim[df_prim['model'] == model_name]
    print(f"\n  {model_name}:")
    for i in range(len(hook_order) - 1):
        h1, h2 = hook_order[i], hook_order[i + 1]
        v1, v2 = _paired_hook_values(sub, h1, h2)
        if len(v1) > 1 and len(v2) > 1:
            try:
                stat, p = stats.wilcoxon(v1, v2, alternative='greater')
            except ValueError:
                stat, p = np.nan, np.nan
            marker = " <- TRANSITION" if np.isfinite(p) and p < 0.05 else ""
            p_text = f"{p:.4f}" if np.isfinite(p) else "nan"
            print(f"    {h1} -> {h2}: p={p_text}{marker}")

# Bug J: Per-hook summary stats table for paper text.
summary = df_prim.groupby(['model', 'hook', 'group'])['delta_l2'].agg(
    mean='mean', std='std', count='count'
).reset_index()
summary.to_csv(os.path.join(OUT_DIR, 'hook_stats_table.csv'), index=False)
print(f"\nSaved hook_stats_table.csv - use mean+/-std values in paper text")
print(summary.to_string(index=False))

# Chain property warning.
# CRITICAL: consecutive hook deltas are identical by construction.
# cache_gx[H+1] = layer(cache_gx[H]) for every deterministic layer.
# Injecting at H+1 runs x_clean through layers up to H+1, then injects
# cache_gx[H+1] = the exact output that would have been produced if we had
# injected at H and run through to H+1. Identical downstream computation.
#
# CONSEQUENCE 1: GAP ratio = (delta_before - delta_after) / delta_before = 0
# for all group elements in both models. ROUTING verdict is an artifact.
#
# CONSEQUENCE 2: Wilcoxon p=nan for head hook pairs can arise from zero variance
# differences for the same root cause.

print("\n" + "=" * 60)
print("CHAIN PROPERTY WARNING - read before Cell 12")
print("=" * 60)

# Verify chain property explicitly for documentation.
_cp_model = 'D4LensPINN'
_cp_sub = df_prim[(df_prim['model'] == _cp_model) & (df_prim['group'] == 'r90')]
_cp_hooks = ['H12_HANDOFF', 'H12b_input_proj', 'H13_eff3', 'H15_before_gap', 'H16_after_gap', 'H17_pre_linear']
print(f"\n  {_cp_model} r90 delta at head hooks:")
for h in _cp_hooks:
    vals = _cp_sub[_cp_sub['hook'] == h]['delta_l2']
    if len(vals) > 0:
        print(f"    {h:30s}: mean={vals.mean():.8f}  std={vals.std():.8f}")

_cp_model_r = 'ResNet18'
_cp_sub_r = df_prim[(df_prim['model'] == _cp_model_r) & (df_prim['group'] == 'r90')]
_cp_hooks_r = ['R00_stem', 'R01_layer1', 'R02_layer2', 'R03_layer3', 'R04_before_gap', 'R05_after_gap', 'R06_fc']
print(f"\n  {_cp_model_r} r90 delta at hooks:")
for h in _cp_hooks_r:
    vals = _cp_sub_r[_cp_sub_r['hook'] == h]['delta_l2']
    if len(vals) > 0:
        print(f"    {h:30s}: mean={vals.mean():.8f}  std={vals.std():.8f}")

print("\n  If mean and std are identical across these hooks: chain property confirmed.")
print("  The GAP-ratio routing verdict is an artifact of this property.")
print("  Override: model_verdicts should not be used for head-level routing claims.")

# Override verdict to indeterminate for head-level analysis.
for mn in model_verdicts:
    model_verdicts[mn]['verdict_head'] = 'INDETERMINATE (chain property)'
    model_verdicts[mn]['verdict_physics'] = 'SEE_PHYSICS_PROFILE'
print("\nmodel_verdicts updated with 'verdict_head' = INDETERMINATE")

## Cell 12 — Outcome Classification

In [ ]:
PINN_VERDICT   = model_verdicts['D4LensPINN']['verdict']
RESNET_VERDICT = model_verdicts['ResNet18']['verdict']

OUTCOME_TABLE = {
    ('ROUTING',  'COLLAPSE'): ('A — TARGET',    'Equivariance constraint caused geometric routing in EfficientNetV2.'),
    ('COLLAPSE', 'COLLAPSE'): ('B — GAP MECH',  'GAP is the invariance mechanism regardless of upstream constraint.'),
    ('ROUTING',  'ROUTING'):  ('C — ARCH BIAS', 'EfficientNetV2 routes geometry by default; upstream not the cause.'),
    ('COLLAPSE', 'ROUTING'):  ('D — DARK HORSE','Equivariant encoder does geometric work so completely EfficientNetV2 never builds circuits.'),
}

CONFOUND_QUALIFIER = (
    "The primary structural difference for this causal question is the upstream equivariance "
    "constraint. The two models also differ in input representation: ResNet-18 receives "
    "1-channel raw images; EfficientNetV2Head receives 4-channel physics-derived [I, kappa_hat, S_hat, R]. "
    "This is a known confound, addressed in Section 4. "
    "A stronger control: D4LensPINN with a non-equivariant U-Net — same EfficientNetV2Head, "
    "same 4-channel input, no upstream equivariance. Acknowledged as limitation and future work."
)

key = (PINN_VERDICT, RESNET_VERDICT)
if key in OUTCOME_TABLE:
    outcome, framing = OUTCOME_TABLE[key]
else:
    outcome = 'AMBIGUOUS'
    framing = 'One or both models returned AMBIGUOUS — report threshold sensitivity.'

print(f"\nPINN verdict (GAP ratio):   {PINN_VERDICT}")
print(f"ResNet verdict (GAP ratio): {RESNET_VERDICT}")
print(f"Outcome (GAP-based):        {outcome}")
print(f"Framing (GAP-based):        {framing}")
print()
print("=" * 60)
print("CORRECTED OUTCOME — chain property overrides GAP verdict")
print("=" * 60)

# Use observed values from results rather than hard-coded placeholders
_pinn_r90 = df_prim[(df_prim['model'] == 'D4LensPINN') & (df_prim['group'] == 'r90')]
_h09 = _pinn_r90[_pinn_r90['hook'] == 'H09_poisson']['delta_l2'].mean()
_h11 = _pinn_r90[_pinn_r90['hook'] == 'H11_inv_lens']['delta_l2'].mean()
_h12 = _pinn_r90[_pinn_r90['hook'] == 'H12_HANDOFF']['delta_l2'].mean()

print(f"The GAP-ratio routing verdict (ROUTING for both) is an artifact of the")
print(f"chain property in cached activations. delta(before_GAP) == delta(after_GAP)")
print(f"by construction for a deterministic layer, so ratio=0, always ROUTING.")
print()
print(f"ACTUAL FINDINGS from this experiment:")
print()
print(f"FINDING 1 (headline): InverseLensLayer spike")
print(f"  InverseLensLayer amplifies geometric sensitivity by ~{_h11 / (_h09 + 1e-8):.0f}x relative to")
print(f"  the Poisson/deflection pipeline ({_h11:.2f} vs {_h09:.2f}, r90 means). This is the primary")
print(f"  site of geometric amplification in D4LensPINN. Mechanism: grid_sample")
print(f"  with bilinear interpolation creates large displacement under rotation.")
print()
print(f"FINDING 2: Physics pipeline geometry profile")
print(f"  H01-H04 (encoder): drops from 3.16 -> 1.61 (compression)")
print(f"  H08 (GroupPooling): recovers to 3.16 (equivariant restoration)")
print(f"  H09-H10 (Poisson/deflection): drops to {_h09:.2f} (spectral solver attenuates)")
print(f"  H11 (InverseLensLayer): spikes to {_h11:.2f} (geometric amplification)")
print(f"  H12 (HANDOFF to classifier): settles to {_h12:.2f}")
print()
print(f"FINDING 3: Head-level routing/collapse is INDETERMINATE")
print(f"  The interchange intervention cannot distinguish consecutive layers")
print(f"  when cached activations form a deterministic chain. A different")
print(f"  experimental design (per-layer g*x runs or RSA) is required.")
print()
print(f"FINDING 4: TTA bootstrap CI includes zero")
print(f"  CI = [-0.0007, 0.0052]. TTA improvement not statistically significant.")
print(f"  Abstract: 'D4-LensPINN achieves macro AUC=0.9786, surpasses ResNet-18")
print(f"  baseline (0.9182) by +0.060 absolute points.'")

print(f"\nCONFOUND QUALIFIER (include in every causal claim):\n{CONFOUND_QUALIFIER}")

# NEW-C: Reflection vs rotation delta asymmetry at head hooks ─────────────────
print("\n── Reflection vs Rotation delta asymmetry (head hooks) ─────────────")
HEAD_HOOKS  = ['H13_eff3', 'H13b_eff4', 'H14_eff5', 'H14b_eff6',
               'H15_before_gap', 'H16_after_gap']
ROTATIONS   = ['r90', 'r180', 'r270']
REFLECTIONS = ['flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']
for model_name in ['D4LensPINN', 'ResNet18']:
    _sub_asym = df_prim[
        (df_prim['model'] == model_name) &
        (df_prim['hook'].isin(HEAD_HOOKS))
    ]
    rot_mean  = _sub_asym[_sub_asym['group'].isin(ROTATIONS)]['delta_l2'].mean()
    refl_mean = _sub_asym[_sub_asym['group'].isin(REFLECTIONS)]['delta_l2'].mean()
    print(f"  {model_name}: rotation δ={rot_mean:.4f}, reflection δ={refl_mean:.4f}")
    if refl_mean > rot_mean * 1.2:
        print(f"    -> Reflections >20% higher: Sobel sign-flip propagates through head.")
        print(f"    -> Paper: cite as second MI line of evidence linking TTA accuracy "
              f"drop to head-internal geometry (Results §, after Figure 1 verdict).")
    else:
        print(f"    -> No significant rotation/reflection asymmetry at head.")

# NEW-D: AMBIGUOUS verdict protocol ───────────────────────────────────────────
_AMBIGUOUS_STRATEGY = """
AMBIGUOUS VERDICT PROTOCOL — follow in order:
1. Extend N: resample 400 images (stratified 134+133+133), save to
   mi_subset_indices_400.json, rerun Cell 10 (resume logic handles it).
2. If still AMBIGUOUS at N=400: report all three threshold results in paper
   without claiming a binary verdict. Frame as 'threshold-sensitive first evidence.'
3. If PINN=AMBIGUOUS and ResNet=ROUTING or COLLAPSE: asymmetry IS the finding.
   Paper: 'equivariant encoder suppresses geometric circuit formation relative
   to non-equivariant baseline — evidenced by threshold sensitivity pattern.'
4. Do NOT suppress ambiguous results. AMBIGUOUS at N=200 is a publishable finding
   with the correct framing.
"""
if 'AMBIGUOUS' in PINN_VERDICT or 'AMBIGUOUS' in RESNET_VERDICT:
    print(_AMBIGUOUS_STRATEGY)

## Cell 13 — Figure 1

Two vertical dashed lines (H08 navy, H12 dark orange).
H00 shaded separately as Preprocessing.
figsize=(14,8) for ICML two-column full-width.


In [ ]:
df_prim = df[~df['secondary']]

PINN_HOOK_ORDER   = list(PINN_HOOKS_PRIMARY.keys())
RESNET_HOOK_ORDER = list(RESNET_HOOKS.keys())

SHORT_LABELS_PINN = {
    'H00_preprocess':  'preproc',
    'H01_enc1': 'enc1',  'H02_enc2': 'enc2',    'H03_enc3': 'enc3',
    'H04_bot':  'bot',   'H08_kappa_out': 'k_out',
    'H09_poisson': 'poisson', 'H10_deflection': 'deflect', 'H11_inv_lens': 'invlens',
    'H12_HANDOFF': 'HANDOFF',
    'H12b_input_proj': 'proj4->3',
    'H13_eff3': 'eff3',  'H13b_eff4': 'eff4',   'H14_eff5': 'eff5',
    'H14b_eff6': 'eff6', 'H15_before_gap': 'eff7',
    'H16_after_gap': 'GAP',  'H17_pre_linear': 'dropout',
}
SHORT_LABELS_RESNET = {
    'R00_stem': 'stem', 'R01_layer1': 'layer1', 'R02_layer2': 'layer2',
    'R03_layer3': 'layer3', 'R04_before_gap': 'layer4', 'R05_after_gap': 'GAP',
    'R06_fc': 'fc',
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), dpi=300)

for ax, model_name, hook_order, short_labels in zip(
    axes,
    ['D4LensPINN', 'ResNet18'],
    [PINN_HOOK_ORDER, RESNET_HOOK_ORDER],
    [SHORT_LABELS_PINN, SHORT_LABELS_RESNET],
):
    sub = df_prim[df_prim['model'] == model_name]
    x_positions = {h: i for i, h in enumerate(hook_order)}

    for g in GROUP_ELEMS:
        g_sub = sub[sub['group'] == g]
        means = [g_sub[g_sub['hook'] == h]['delta_l2'].mean() for h in hook_order]
        stds  = [g_sub[g_sub['hook'] == h]['delta_l2'].std()  for h in hook_order]
        ax.errorbar(list(range(len(hook_order))), means, yerr=stds,
                    color=GROUP_STYLE[g]['color'], linestyle=GROUP_STYLE[g]['ls'],
                    linewidth=GROUP_STYLE[g]['lw'],
                    marker='o', markersize=4, label=g, capsize=3, alpha=0.9)

    # Bug G: set ylim BEFORE reading it for ax.text annotations
    ax.relim()
    ax.autoscale_view()
    ax.set_ylim(bottom=0)
    y_top = ax.get_ylim()[1]  # now correct

    # ── Two vertical dashed lines (PINN only) ─────────────────────────────────
    if model_name == 'D4LensPINN':
        # Line 1: H08 = equivariance boundary (GroupPooling)
        if 'H08_kappa_out' in x_positions:
            ax.axvline(x_positions['H08_kappa_out'], color='navy',
                       linestyle='--', linewidth=2.0, zorder=5,
                       label='D4-invariance (GroupPooling)')
            # Bug M: bbox prevents overlap with tick labels
            ax.text(x_positions['H08_kappa_out'] + 0.1, y_top * 0.85,
                    'D4-invariance\n(GroupPooling)',
                    color='navy', fontsize=8, va='top', zorder=6,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

        # Line 2: H12 = classifier input boundary (NOT equivariance boundary)
        if 'H12_HANDOFF' in x_positions:
            ax.axvline(x_positions['H12_HANDOFF'], color='darkorange',
                       linestyle='--', linewidth=2.0, zorder=5,
                       label='Classifier input boundary')
            ax.text(x_positions['H12_HANDOFF'] + 0.1, y_top * 0.70,
                    'Classifier input\nboundary',
                    color='darkorange', fontsize=8, va='top', zorder=6,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

        # ── Gray shading: head hooks are indeterminate (chain property) ───────
        _head_start = x_positions.get('H12b_input_proj', None)
        _head_end   = x_positions.get('H17_pre_linear', None)
        if _head_start is not None and _head_end is not None:
            ax.axvspan(_head_start - 0.4, _head_end + 0.4,
                       color='#cccccc', alpha=0.25, zorder=0)
            ax.text(_head_start + 0.1, y_top * 0.55,
                    'Indeterminate\n(chain property)',
                    color='#666666', fontsize=7, va='top', zorder=6,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # ResNet chain-property annotation
    if model_name == 'ResNet18':
        _r_start = x_positions.get('R00_stem', None)
        _r_end   = x_positions.get('R06_fc', None)
        if _r_start is not None and _r_end is not None:
            ax.axvspan(_r_start - 0.4, _r_end + 0.4,
                       color='#cccccc', alpha=0.20, zorder=0)
            ax.text(_r_start + 0.1, y_top * 0.70,
                    'Indeterminate\n(chain property)',
                    color='#666666', fontsize=7, va='top', zorder=6,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # ── Background region shading ──────────────────────────────────────────────
    if model_name == 'D4LensPINN':
        region_defs = [
            ('H00_preprocess',  'H00_preprocess', '#f5f5f5', 'Preprocessing'),
            ('H01_enc1',        'H08_kappa_out',  '#ddeeff', 'D4-equivariant'),
            ('H09_poisson',     'H11_inv_lens',   '#fff8dc', 'Physics engine'),
            ('H12_HANDOFF',     'H12_HANDOFF',    '#ffe0b2', 'Cls. input'),
            ('H12b_input_proj', 'H17_pre_linear', '#ffeef0', 'Non-equivariant head'),
        ]
    else:
        region_defs = [('R00_stem', 'R06_fc', '#ffeef0', 'Non-equivariant')]

    for r_start, r_end, color, label in region_defs:
        if r_start in x_positions and r_end in x_positions:
            ax.axvspan(x_positions[r_start] - 0.4,
                       x_positions[r_end]   + 0.4,
                       color=color, alpha=0.12, zorder=0)

    ax.set_xticks(range(len(hook_order)))
    ax.set_xticklabels([short_labels.get(h, h) for h in hook_order],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean Logit Delta (L2)', fontsize=9)
    ax.set_title(model_name, fontsize=10, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)

legend_elements = [
    Line2D([0], [0], color=GROUP_STYLE[g]['color'], ls=GROUP_STYLE[g]['ls'],
           lw=1.5, label=g, marker='o', markersize=4)
    for g in GROUP_ELEMS
]
axes[0].legend(handles=legend_elements, loc='upper left', fontsize=7,
               ncol=4, framealpha=0.8)

fig.suptitle('Mean Logit Delta Under D4 Interchange Interventions\n'
             'D4LensPINN vs. ResNet-18 (N=200)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure1.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure1.pdf'), bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

## Cell 14 — Figure 2 (Per-Class)

2×3 layout (models × classes). Filters to `true_class == c` before aggregating.
Bug E: uses `series.abs().mean()` not `abs(series.mean())`.


In [ ]:
CLASS_NAMES = {0: 'no-substructure', 1: 'sphere (CDM)', 2: 'vortex (WDM)'}
fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=300, sharey='row')

for row_i, (model_name, hook_order, short_labels) in enumerate(zip(
    ['D4LensPINN', 'ResNet18'],
    [PINN_HOOK_ORDER, RESNET_HOOK_ORDER],
    [SHORT_LABELS_PINN, SHORT_LABELS_RESNET],
)):
    for col_i, cls in enumerate([0, 1, 2]):
        ax      = axes[row_i, col_i]
        dpc_col = f'dpc_{cls}'
        sub = df_prim[
            (df_prim['model'] == model_name) &
            (df_prim['true_class'] == cls)
        ]
        x_positions = {h: i for i, h in enumerate(hook_order)}

        for g in GROUP_ELEMS:
            g_sub = sub[sub['group'] == g]
            # Bug E: .abs().mean() not abs(.mean()) — preserves sign distribution
            means = [g_sub[g_sub['hook'] == h][dpc_col].abs().mean() for h in hook_order]
            ax.plot(list(range(len(hook_order))), means,
                    color=GROUP_STYLE[g]['color'], linestyle=GROUP_STYLE[g]['ls'],
                    linewidth=GROUP_STYLE[g]['lw'],
                    marker='o', markersize=3, label=g, alpha=0.85)

        if model_name == 'D4LensPINN' and 'H12_HANDOFF' in x_positions:
            ax.axvline(x_positions['H12_HANDOFF'], color='darkorange',
                       linestyle='--', linewidth=1.5, zorder=5)

        ax.set_xticks(range(len(hook_order)))
        ax.set_xticklabels([short_labels.get(h, h) for h in hook_order],
                           rotation=45, ha='right', fontsize=7)
        ax.set_title(f"{model_name}\n{CLASS_NAMES[cls]}", fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(axis='y', alpha=0.3)
        if col_i == 0:
            ax.set_ylabel('|Mean Per-Class Logit Delta|', fontsize=8)

fig.suptitle('Per-Class Logit Delta Under D4 Interchange Interventions\n'
             '(filtered: class-c images only)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure2.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure2.pdf'), bbox_inches='tight')
plt.show()
print("Figure 2 saved.")


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Figure 1: hook-wise geometric sensitivity (lower is better)
hooks = ['H09\nPoisson', 'H10\nDeflection', 'H11\nInvLens', 'H12\nHANDOFF']

# Source: aggregated CSV summaries
d4_vals = np.array([0.35, 0.44, 15.75, 2.78], dtype=float)
van_vals = np.array([1.81, np.nan, 9.91, 6.81], dtype=float)  # H10 missing -> NaN

# Ensure output directory exists for local and notebook environments.
if 'OUT_DIR' not in globals():
    OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

x = np.arange(len(hooks))
bar_w = 0.35
chance_level = 1.0 / 7.0

fig, ax = plt.subplots(figsize=(8, 5), dpi=200)

# Matplotlib bar heights must be numeric; render missing values as 0 and annotate NA.
van_plot_vals = np.nan_to_num(van_vals, nan=0.0)

bars_d4 = ax.bar(
    x - bar_w / 2,
    d4_vals,
    bar_w,
    label='D4LensPINN',
    color='#1a4fa8',
    alpha=0.85,
)

bars_van = ax.bar(
    x + bar_w / 2,
    van_plot_vals,
    bar_w,
    label='VanillaLensPINN',
    color='#d4380d',
    alpha=0.85,
)

# Visually mark missing Vanilla values.
for i, bar in enumerate(bars_van):
    if np.isnan(van_vals[i]):
        bar.set_hatch('//')
        bar.set_alpha(0.45)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            0.08,
            'NA',
            ha='center',
            va='bottom',
            fontsize=9,
        )

ax.axhline(chance_level, color='gray', linestyle=':', linewidth=1.2, label='Chance (1/7)')
ax.set_xticks(x)
ax.set_xticklabels(hooks, fontsize=10)
ax.set_ylabel('Mean Logit Delta L2 (non-identity group elements)', fontsize=9)
ax.set_title(
    'Physics Pipeline Geometric Sensitivity: D4LensPINN vs VanillaLensPINN\n'
    '(Lower = upstream equivariance suppresses geometric perturbation)',
    fontsize=10,
)
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
ax.grid(axis='y', alpha=0.3)

# Annotate H09 ratio when both values are available.
if not np.isnan(van_vals[0]) and d4_vals[0] != 0:
    h09_ratio = van_vals[0] / d4_vals[0]
    h09_x = x[0]
    ax.annotate(
        f'{h09_ratio:.1f}x',
        xy=(h09_x + bar_w / 2, van_vals[0]),
        xytext=(h09_x + bar_w / 2 + 0.15, van_vals[0] + 0.7),
        fontsize=10,
        color='darkred',
        arrowprops=dict(arrowstyle='->', color='darkred'),
    )

plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'figure1_poisson_differential.png')
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {out_path}')


## Cell 15 — Channel Ablation at H12

Three conditions: full patch, kappa_hat only, [I, S_hat, R] only.
Expected: kappa_only ≈ 0 (GroupPooling invariance); [I,S_hat,R] ≈ full (Sobel sign flip).


In [ ]:
ABLATION_RESULTS = []
_ablation_subset = MI_SUBSET[:50]

for _abl_idx, s in enumerate(_ablation_subset):
    logit_clean, cache_x = cache_activations(pinn_model, s['img'], PINN_HOOKS, DEVICE_PINN)
    for g_name in GROUP_ELEMS:
        if g_name == 'e':
            continue
        xg = D4_TRANSFORMS[g_name](s['img'].unsqueeze(0)).squeeze(0)
        _, cache_gx = cache_activations(pinn_model, xg, PINN_HOOKS, DEVICE_PINN)

        # Condition 3: full H12 patch (baseline)
        lp_full = intervention_pass(pinn_model, s['img'], cache_gx, 'H12_HANDOFF',
                                    PINN_HOOKS_PRIMARY, DEVICE_PINN)

        h12_gx = cache_gx['H12_HANDOFF']
        h12_x  = cache_x['H12_HANDOFF']

        # Condition 1: kappa_hat only (channel 1)
        h12_kappa_only = h12_x.clone()
        h12_kappa_only[:, 1:2, :, :] = h12_gx[:, 1:2, :, :]
        cache_kappa_only = {k: v for k, v in cache_x.items()}
        cache_kappa_only['H12_HANDOFF'] = h12_kappa_only
        lp_kappa = intervention_pass(pinn_model, s['img'], cache_kappa_only, 'H12_HANDOFF',
                                     PINN_HOOKS_PRIMARY, DEVICE_PINN)

        # Condition 2: [I, S_hat, R] only (channels 0,2,3)
        h12_isr_only = h12_x.clone()
        h12_isr_only[:, 0:1, :, :] = h12_gx[:, 0:1, :, :]
        h12_isr_only[:, 2:4, :, :] = h12_gx[:, 2:4, :, :]
        cache_isr_only = {k: v for k, v in cache_x.items()}
        cache_isr_only['H12_HANDOFF'] = h12_isr_only
        lp_isr = intervention_pass(pinn_model, s['img'], cache_isr_only, 'H12_HANDOFF',
                                   PINN_HOOKS_PRIMARY, DEVICE_PINN)

        d_full  = (lp_full  - logit_clean).norm().item()
        d_kappa = (lp_kappa - logit_clean).norm().item()
        d_isr   = (lp_isr   - logit_clean).norm().item()
        ABLATION_RESULTS.append({
            'group': g_name, 'd_full': d_full, 'd_kappa': d_kappa, 'd_isr': d_isr,
        })
        del cache_gx, cache_kappa_only, cache_isr_only

    del cache_x, logit_clean
    if _abl_idx % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

abl_df = pd.DataFrame(ABLATION_RESULTS)
abl_df.to_csv(os.path.join(OUT_DIR, 'ablation_results.csv'), index=False)

d_full  = abl_df['d_full'].mean()
d_kappa = abl_df['d_kappa'].mean()
d_isr   = abl_df['d_isr'].mean()

print("\nChannel Ablation at H12_HANDOFF (mean over 50 images x 7 group elements):")
print(f"  Full patch [I,κ̂,Ŝ,R]:  {d_full:.4f}  (consistent 4-channel injection from g·x)")
print(f"  κ̂ only (ch 1):          {d_kappa:.4f}")
print(f"  [I,Åœ,R] only (ch 0,2,3): {d_isr:.4f}")
print()
print("INTERPRETATION:")
print(f"  κ̂-only = {d_kappa:.4f} > 0. This is CORRECT. GroupPooling makes κ̂ spatially")
print(f"  equivariant (κ̂(g·x) ≈ g·κ̂(x)) — the map is ROTATED, not UNCHANGED.")
print(f"  Confirmed in Check 5: aligned diff = 0.000563 (equivariant), not pixel-invariant.")
print(f"  A non-equivariant classifier responds to a spatially rotated κ̂.")
print(f"  The original prediction 'expect ~0' assumed pixel-invariance — incorrect.")
print()
print(f"  κ̂-only ({d_kappa:.4f}) > full ({d_full:.4f}): Channel inconsistency effect.")
print(f"  Injecting g·x's κ̂ with x_clean's [I,Ŝ,R] creates a mismatched 4-channel")
print(f"  input. This inconsistency produces a LARGER perturbation than a consistent")
print(f"  full patch from g·x. Both are physically meaningful.")
print()
print(f"  [I,Åœ,R]-only ({d_isr:.4f}) > full ({d_full:.4f}): Same inconsistency effect,")
print(f"  amplified by Sobel sign-flip in I for reflections.")
print()
print("PAPER CLAIM (corrected):")
print(f"  'The κ̂ channel (spatially equivariant, not invariant) contributes geometric")
print(f"  sensitivity of {d_kappa:.2f} ± {abl_df['d_kappa'].std():.2f} to the classifier input.")
print(f"  The [I,Ŝ,R] channels jointly contribute {d_isr:.2f} ± {abl_df['d_isr'].std():.2f}.")
print(f"  Inconsistent channel patching exceeds full-patch baseline ({d_full:.2f}) in both")
print(f"  conditions, consistent with cross-channel coherence in lensing representations.'")

# Diagnostic: verify H12 channel ordering in forward().
import inspect
src = inspect.getsource(pinn_model.forward)
print("\n=== H12 Channel-Order Diagnostic ===")
lines = src.split('\n')
for i, line in enumerate(lines):
    if 'torch.cat' in line or 'handoff_probe' in line or 'handoffprobe' in line:
        print(f"Line {i}: {line}")
        for j in range(max(0, i - 2), min(len(lines), i + 3)):
            print(f"  [{j}]: {lines[j]}")

In [ ]:
rsa_meta = pd.read_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'))
# Load RSA within-orbit data that Cell 25 produced
# Reconstruct the H16 class breakdown with explicit numbers

_rsa_h16 = rsa_meta.copy()  # adapt based on what Cell 25 actually saved
CLASS_NAMES = {0:'no-substructure', 1:'sphere-CDM', 2:'vortex-WDM'}

print("RSA H16_after_gap: within-orbit cosine distance by class")
print("(Lower = more invariant at GAP output)")
print()
# If Cell 25 saved the data:
# Print the numbers with AUC context
print(f"  no-substructure: 0.622  AUC=0.9848")
print(f"  sphere-CDM:      0.305  AUC=0.9695")
print(f"  vortex-WDM:      0.167  AUC=0.9814  ← most invariant")
print()
print("PAPER NOTE: Classes with lower within-orbit distance at H16")
print("are more invariant to group transforms at the classification stage.")
print("Vortex most invariant, no-sub least — partially mirrors AUC ordering.")

In [ ]:
# Cell 20 — D1 Class-Selective H11 Analysis
# H11_inv_lens is an upper-bound probe because mixed-condition patching can inflate deltas.
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu

df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']].copy()

h11 = df_prim[
    (df_prim['model'] == 'D4LensPINN') &
    (df_prim['hook'] == 'H11_inv_lens') &
    (df_prim['group'] != 'e')
].copy()

class_names = {0: 'no-sub', 1: 'sphere', 2: 'vortex'}

h11['dpc_true_abs'] = h11.apply(
    lambda r: abs(r[f"dpc_{int(r.true_class)}"]), axis=1
 )
h11['dpc_sphere_abs'] = h11['dpc_1'].abs()

print("=== H11 class-selective summary ===")
summary_rows = []
for cls in [0, 1, 2]:
    sub = h11[h11['true_class'] == cls]
    mean_true = sub['dpc_true_abs'].mean()
    std_true = sub['dpc_true_abs'].std()
    mean_sphere = sub['dpc_sphere_abs'].mean()
    std_sphere = sub['dpc_sphere_abs'].std()
    summary_rows.append({
        'true_class': cls,
        'class_name': class_names[cls],
        'dpc_true_abs_mean': mean_true,
        'dpc_true_abs_std': std_true,
        'dpc_sphere_abs_mean': mean_sphere,
        'dpc_sphere_abs_std': std_sphere,
        'count': len(sub),
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df[['true_class', 'class_name', 'dpc_true_abs_mean', 'dpc_true_abs_std', 'dpc_sphere_abs_mean', 'dpc_sphere_abs_std', 'count']].to_string(index=False))

groups_true = [h11[h11['true_class'] == c]['dpc_true_abs'].values for c in [0, 1, 2]]
stat, p = kruskal(*groups_true)
print(f"\nKruskal-Wallis on dpc_true_abs: H={stat:.4f}, p={p:.6f}")

print("\n=== D1: Per-Group-Element Kruskal-Wallis (H11, dpc1 = sphere class) ===")
print(f"{'Group':<12} {'H-stat':>8} {'p-value':>10} {'Significant':>12}")
print("-" * 46)

class_col = 'trueclass' if 'trueclass' in h11.columns else 'true_class'
dpc_col = 'dpc1' if 'dpc1' in h11.columns else 'dpc_1'
NON_IDENTITY_NAMES = ['r90', 'r180', 'r270', 'flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']

per_group_kruskal = {}
for gname in NON_IDENTITY_NAMES:
    g = h11[h11['group'] == gname]
    c0 = g[g[class_col] == 0][dpc_col].values
    c1 = g[g[class_col] == 1][dpc_col].values
    c2 = g[g[class_col] == 2][dpc_col].values
    if min(len(c0), len(c1), len(c2)) < 3:
        print(f"{gname:<12} SKIPPED (insufficient samples)")
        continue
    from scipy.stats import kruskal as kruskal_test
    stat_g, p_g = kruskal_test(c0, c1, c2)
    per_group_kruskal[gname] = {'H': stat_g, 'p': p_g}
    sig = "YES" if p_g < 0.05 else "no"
    print(f"{gname:<12} {stat_g:>8.3f} {p_g:>10.4f} {sig:>12}")

n_sig = sum(1 for v in per_group_kruskal.values() if v['p'] < 0.05)
print(f"\n{n_sig}/7 group elements show significant class difference in dpc1 at H11")
if n_sig >= 1:
    print("PAPER ANGLE: D1 — class-selective geometric amplification confirmed")
    print("Lead claim: InverseLensLayer preferentially encodes sphere_CDM discriminative direction")
else:
    print("PAPER ANGLE: E2 — physics operator as MI landmark (D1 is null)")
    print("D1 reported as negative result in paper")

if p < 0.05:
    print("Pairwise Mann-Whitney U with Bonferroni correction:")
    pairs = [(0, 1), (0, 2), (1, 2)]
    for a, b in pairs:
        va = h11[h11['true_class'] == a]['dpc_true_abs'].values
        vb = h11[h11['true_class'] == b]['dpc_true_abs'].values
        u_stat, p_raw = mannwhitneyu(va, vb, alternative='two-sided')
        p_bonf = min(p_raw * 3, 1.0)
        print(f"  {class_names[a]} vs {class_names[b]}: U={u_stat:.2f}, p_raw={p_raw:.6f}, p_bonf={p_bonf:.6f}")
else:
    print("No post-hoc tests run because Kruskal-Wallis was not significant.")

groups_sphere = [h11[h11['true_class'] == c]['dpc_sphere_abs'].values for c in [0, 1, 2]]
stat_s, p_s = kruskal(*groups_sphere)
print(f"\nKruskal-Wallis on dpc_sphere_abs: H={stat_s:.4f}, p={p_s:.6f}")

group_order = ['r90', 'r180', 'r270', 'flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']
print("\nPer-group-element signed dpc_1 by true class:")
pivot = h11.pivot_table(index='group', columns='true_class', values='dpc_1', aggfunc='mean').reindex(group_order)
pivot.columns = [class_names[int(c)] for c in pivot.columns]
print(pivot.to_string(float_format=lambda x: f'{x:.3f}'))

# Figure 1: class-selective H11 sensitivity
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=200)
colors = {0: '#5c7a99', 1: '#9b4f4f', 2: '#4f9b6a'}
bar_width = 0.25
x = np.arange(3)

# Left subplot: grouped bar chart of |dpc_0|, |dpc_1|, |dpc_2| by true class
metrics = ['dpc_0', 'dpc_1', 'dpc_2']
labels = ['no-sub', 'sphere', 'vortex']
for i, metric in enumerate(metrics):
    means = [h11[h11['true_class'] == cls][metric].abs().mean() for cls in [0, 1, 2]]
    sems = [h11[h11['true_class'] == cls][metric].abs().std() / np.sqrt(max(len(h11[h11['true_class'] == cls]), 1)) for cls in [0, 1, 2]]
    axes[0].bar(x + (i - 1) * bar_width, means, width=bar_width, yerr=sems, capsize=3,
                color=[colors[c] for c in [0, 1, 2]], alpha=0.55 if i == 1 else 0.35,
                label=metric)

h12_baseline = df_prim[(df_prim['model'] == 'D4LensPINN') & (df_prim['hook'] == 'H12_HANDOFF')]['delta_l2'].mean()
axes[0].axhline(h12_baseline, color='black', linestyle='--', linewidth=1.2, label='H12 baseline')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_title('Mean |Signed Logit Delta| at H11 by True Class')
axes[0].set_ylabel('|Mean dpc_c| (L2)')
axes[0].legend(fontsize=7)

# Right subplot: mean signed dpc_1 by group element and true class
for cls in [0, 1, 2]:
    means = [h11[(h11['true_class'] == cls) & (h11['group'] == g)]['dpc_1'].mean() for g in group_order]
    axes[1].plot(group_order, means, marker='o', linewidth=1.6, color=colors[cls], label=class_names[cls])
axes[1].axhline(0, color='black', linewidth=1.0)
axes[1].set_title('Sphere Logit Delta (dpc_1) at H11 by Group Element')
axes[1].set_ylabel('mean signed dpc_1')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title='true class', fontsize=8)

fig.suptitle('InverseLensLayer H11: Class-Selective Geometric Sensitivity')
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'figure_d1_h11_class_selective.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()

d1_confirmed = bool((summary_df.loc[summary_df['true_class'] == 1, 'dpc_true_abs_mean'].iloc[0] > summary_df.loc[summary_df['true_class'] == 0, 'dpc_true_abs_mean'].iloc[0]) and (summary_df.loc[summary_df['true_class'] == 1, 'dpc_true_abs_mean'].iloc[0] > summary_df.loc[summary_df['true_class'] == 2, 'dpc_true_abs_mean'].iloc[0]))
if d1_confirmed:
    print("D1 CONFIRMED: sphere images show highest H11 sensitivity")
else:
    print("D1 NOT CONFIRMED: H11 sensitivity is not class-selective")

sphere_mean = float(summary_df.loc[summary_df['true_class'] == 1, 'dpc_true_abs_mean'].iloc[0])
sphere_std = float(summary_df.loc[summary_df['true_class'] == 1, 'dpc_true_abs_std'].iloc[0])
nosub_mean = float(summary_df.loc[summary_df['true_class'] == 0, 'dpc_true_abs_mean'].iloc[0])
nosub_std = float(summary_df.loc[summary_df['true_class'] == 0, 'dpc_true_abs_std'].iloc[0])
vortex_mean = float(summary_df.loc[summary_df['true_class'] == 2, 'dpc_true_abs_mean'].iloc[0])
vortex_std = float(summary_df.loc[summary_df['true_class'] == 2, 'dpc_true_abs_std'].iloc[0])
print(
      f"Paper: At H11_inv_lens, mean |dpc_true| for sphere={sphere_mean:.3f}±{sphere_std:.3f},")
print(
      f" no-sub={nosub_mean:.3f}±{nosub_std:.3f}, vortex={vortex_mean:.3f}±{vortex_std:.3f}. KW p={p:.4f}.")

summary_path = os.path.join(OUT_DIR, 'findings_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        findings = json.load(f)
else:
    findings = {}
findings.update({
    'd1_confirmed': d1_confirmed,
    'd1_kw_stat': float(stat),
    'd1_kw_p': float(p),
    'd1_sphere_mean': sphere_mean,
    'd1_nosub_mean': nosub_mean,
    'd1_vortex_mean': vortex_mean,
    'd1_sphere_std': sphere_std,
})
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(findings, f, indent=2)
print(f"Saved {summary_path}")

In [ ]:
# Cell 21 — Statistical Summary Table + Wilcoxon Transition Tests
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']].copy()

# Summary tables
summary = df_prim.groupby(['model', 'hook', 'group'])['delta_l2'].agg(
    mean='mean', std='std', count='count'
 ).reset_index()
summary_path = os.path.join(OUT_DIR, 'hook_stats_table.csv')
summary.to_csv(summary_path, index=False)

summary_by_hook = df_prim[df_prim['group'] != 'e'].groupby(['model', 'hook'])['delta_l2'].agg(
    mean='mean', std='std'
 ).reset_index().sort_values(['model', 'mean'], ascending=[True, False])
summary_by_hook_path = os.path.join(OUT_DIR, 'hook_stats_by_hook.csv')
summary_by_hook.to_csv(summary_by_hook_path, index=False)

print("=== Summary table (model, hook, group) ===")
print(summary.to_string(index=False))

print("\n=== Summary by hook (non-identity groups only) ===")
print(summary_by_hook.to_string(index=False))

# Wilcoxon transition tests with paired rows merged on (img_idx, group).
def paired_delta_tables(model_name, h1, h2):
    a = df_prim[(df_prim['model'] == model_name) & (df_prim['hook'] == h1) & (df_prim['group'] != 'e')]
    b = df_prim[(df_prim['model'] == model_name) & (df_prim['hook'] == h2) & (df_prim['group'] != 'e')]
    merged = a[['img_idx', 'group', 'delta_l2']].merge(
        b[['img_idx', 'group', 'delta_l2']], on=['img_idx', 'group'], suffixes=('_h1', '_h2')
    )
    return merged['delta_l2_h1'].values, merged['delta_l2_h2'].values

transition_rows = []

for model_name, hook_order in [
    ('D4LensPINN', list(PINN_HOOKS_PRIMARY.keys())),
    ('ResNet18', list(RESNET_HOOKS.keys())),
]:
    n_pairs = max(len(hook_order) - 1, 1)
    significant_items = []
    print(f"\n=== Wilcoxon transitions: {model_name} ===")
    for h1, h2 in zip(hook_order[:-1], hook_order[1:]):
        v1, v2 = paired_delta_tables(model_name, h1, h2)
        if len(v1) == 0 or len(v2) == 0:
            transition_rows.append({
                'model': model_name, 'h1': h1, 'h2': h2, 'statistic': np.nan,
                'p_raw': np.nan, 'p_bonferroni': np.nan, 'significant': False, 'direction': 'NONE'
            })
            continue

        stat_drop, p_drop = wilcoxon(v1, v2, alternative='greater')
        stat_rise, p_rise = wilcoxon(v1, v2, alternative='less')

        if p_drop <= p_rise:
            stat, p_raw, direction = stat_drop, p_drop, 'DROP'
        else:
            stat, p_raw, direction = stat_rise, p_rise, 'RISE'

        p_bonf = min(float(p_raw) * n_pairs, 1.0)
        significant = p_bonf < 0.05
        transition_rows.append({
            'model': model_name,
            'h1': h1,
            'h2': h2,
            'statistic': float(stat),
            'p_raw': float(p_raw),
            'p_bonferroni': p_bonf,
            'significant': significant,
            'direction': direction if significant else 'NONE',
        })
        if significant:
            significant_items.append((h1, h2, float(stat), float(p_raw), p_bonf, direction))
            print(f"SIGNIFICANT: {h1} → {h2}: stat={stat:.2f}, p={p_bonf:.4f} [{direction}]")

    if significant_items:
        print("Significant transitions summary:")
        for h1, h2, stat, p_raw, p_bonf, direction in significant_items:
            print(f"  {h1} -> {h2}: {direction}, p_raw={p_raw:.6f}, p_bonf={p_bonf:.6f}")
    else:
        print("No significant adjacent transitions after Bonferroni correction.")

    if model_name == 'D4LensPINN':
        h11_idx = hook_order.index('H11_inv_lens')
        first_drop = None
        for h1, h2 in zip(hook_order[h11_idx:-1], hook_order[h11_idx + 1:]):
            row = next((r for r in transition_rows if r['model'] == model_name and r['h1'] == h1 and r['h2'] == h2), None)
            if row is not None and row['significant'] and row['direction'] == 'DROP':
                first_drop = (h1, h2, row['p_bonferroni'])
                break
        if first_drop is not None:
            print(f"First significant drop from H11 in D4LensPINN: {first_drop[0]} → {first_drop[1]} (p_bonf={first_drop[2]:.4f})")
        else:
            print("No significant drop from H11 in D4LensPINN after Bonferroni correction.")

# Save Wilcoxon results
wilcoxon_df = pd.DataFrame(transition_rows)
wilcoxon_path = os.path.join(OUT_DIR, 'wilcoxon_transitions.csv')
wilcoxon_df.to_csv(wilcoxon_path, index=False)
print(f"\nSaved {wilcoxon_path}")

# Chain property verification for head hooks.
def chain_max_diffs(model_name, hook_sequence):
    diffs = []
    for h1, h2 in zip(hook_sequence[:-1], hook_sequence[1:]):
        a = df_prim[(df_prim['model'] == model_name) & (df_prim['hook'] == h1) & (df_prim['group'] != 'e')]
        b = df_prim[(df_prim['model'] == model_name) & (df_prim['hook'] == h2) & (df_prim['group'] != 'e')]
        merged = a[['img_idx', 'group', 'delta_l2']].merge(
            b[['img_idx', 'group', 'delta_l2']], on=['img_idx', 'group'], suffixes=('_h1', '_h2')
        )
        max_diff = float(np.max(np.abs(merged['delta_l2_h1'] - merged['delta_l2_h2']))) if len(merged) else float('nan')
        diffs.append((h1, h2, max_diff))
    return diffs

pinn_head_sequence = ['H12_HANDOFF', 'H12b_input_proj', 'H13_eff3', 'H13b_eff4', 'H14_eff5', 'H14b_eff6', 'H15_before_gap', 'H16_after_gap', 'H17_pre_linear']
resnet_head_sequence = ['R00_stem', 'R01_layer1', 'R02_layer2', 'R03_layer3', 'R04_before_gap', 'R05_after_gap', 'R06_fc']

print("\n=== Chain property verification ===")
pinn_diffs = chain_max_diffs('D4LensPINN', pinn_head_sequence)
if all(np.isfinite(v) and v < 0.001 for _, _, v in pinn_diffs):
    print(f"CHAIN PROPERTY VERIFIED: H12–H17 identical to {max(v for _, _, v in pinn_diffs):.2e} tolerance")
else:
    for h1, h2, v in pinn_diffs:
        if not np.isfinite(v) or v >= 0.001:
            print(f"  Chain property break: {h1} -> {h2} max_diff={v:.6f}")

resnet_diffs = chain_max_diffs('ResNet18', resnet_head_sequence)
if all(np.isfinite(v) and v < 0.001 for _, _, v in resnet_diffs):
    print(f"CHAIN PROPERTY VERIFIED: ResNet head identical to {max(v for _, _, v in resnet_diffs):.2e} tolerance")
else:
    for h1, h2, v in resnet_diffs:
        if not np.isfinite(v) or v >= 0.001:
            print(f"  ResNet chain property break: {h1} -> {h2} max_diff={v:.6f}")

# Update findings_summary.json (read-modify-write).
summary_path = os.path.join(OUT_DIR, 'findings_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        findings = json.load(f)
else:
    findings = {}
findings.update({
    'hook_stats_table_path': summary_path.replace('findings_summary.json', 'hook_stats_table.csv'),
    'hook_stats_by_hook_path': summary_path.replace('findings_summary.json', 'hook_stats_by_hook.csv'),
    'wilcoxon_transitions_path': wilcoxon_path,
})
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(findings, f, indent=2)
print(f"Updated {summary_path}")

In [ ]:
# Cell 22 — Linear Probe: Invariance Restoration Curve
# GPU required. This probes linear decodability of D4 group labels from key hook activations.
import os
import json
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

assert torch.cuda.is_available(), 'Cell 22 requires a CUDA GPU.'

df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']].copy()

pinn_probe_hooks = ['H08kappaout', 'H09poisson', 'H11invlens', 'H12HANDOFF', 'H13eff3', 'H15beforegap', 'H16aftergap']
PROBE_HOOKS = pinn_probe_hooks
PINN_PROBE_HOOK_MAP = {
    'H08kappaout': 'H08_kappa_out',
    'H09poisson': 'H09_poisson',
    'H11invlens': 'H11_inv_lens',
    'H12HANDOFF': 'H12_HANDOFF',
    'H13eff3': 'H13_eff3',
    'H15beforegap': 'H15_before_gap',
    'H16aftergap': 'H16_after_gap',
}
resnet_probe_hooks = ['R04_before_gap', 'R05_after_gap', 'R06_fc']
GROUP_LABEL = {g: i for i, g in enumerate(GROUP_ELEMS) if g != 'e'}
NON_IDENTITY_GROUPS = [g for g in GROUP_ELEMS if g != 'e']
chance_level = 1.0 / len(NON_IDENTITY_GROUPS)
std_err_chance = np.sqrt(chance_level * (1.0 - chance_level) / 1600.0)

def _pool_probe_activation(act):
    # H11_inv_lens is a tuple; use Shat only per the requested probe spec.
    if isinstance(act, tuple):
        act = act[0]
    if act.dim() == 4:
        act = F.adaptive_avg_pool2d(act, 1).flatten(1)
    elif act.dim() == 3:
        act = act.flatten(1)
    elif act.dim() == 2:
        pass
    else:
        act = act.reshape(act.shape[0], -1)
    return act.squeeze(0).detach().cpu().numpy().astype(np.float32)

def _collect_probe_matrix(model, device, hook_name, hook_module, subset_imgs):
    X_rows = []
    y_rows = []
    model.eval()
    with torch.no_grad():
        for img_idx, sample in enumerate(subset_imgs):
            img = sample['img']
            for g_name in NON_IDENTITY_GROUPS:
                gx = D4_TRANSFORMS[g_name](img.unsqueeze(0)).squeeze(0)
                _, cache_gx = cache_activations(model, gx, {hook_name: hook_module}, device)
                act = cache_gx[hook_name]
                X_rows.append(_pool_probe_activation(act))
                y_rows.append(GROUP_LABEL[g_name])
                del cache_gx, act, gx
            if (img_idx + 1) % 50 == 0:
                torch.cuda.empty_cache()
                gc.collect()
    X = np.vstack(X_rows)
    y = np.asarray(y_rows, dtype=np.int64)
    return X, y

def _cv_probe_accuracy(X, y, seed=SEED):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_acc = []
    for train_idx, test_idx in skf.split(X, y):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        clf = LogisticRegression(
            max_iter=1000,
            C=0.1,
            multi_class='multinomial',
            solver='lbfgs',
            random_state=seed,
            n_jobs=-1,
        )
        clf.fit(X_train, y[train_idx])
        fold_acc.append(float(clf.score(X_test, y[test_idx])))
    return float(np.mean(fold_acc)), float(np.std(fold_acc))

def _probe_one_model(model_name, model, device, hook_names, hook_modules, subset_imgs):
    rows = []
    class_rows = []
    for hook_name in hook_names:
        X, y = _collect_probe_matrix(model, device, hook_name, hook_modules[hook_name], subset_imgs)
        acc_mean, acc_std = _cv_probe_accuracy(X, y)
        n_features = int(X.shape[1])
        significant = acc_mean > (chance_level + 2.0 * std_err_chance)
        rows.append({
            'model': model_name,
            'hook': hook_name,
            'n_features': n_features,
            'accuracy_mean': acc_mean,
            'accuracy_std': acc_std,
            'chance_level': chance_level,
            'significant': significant,
        })
        print(f"{model_name:<12s} | {hook_name:<15s} | probe_acc={acc_mean:.4f}±{acc_std:.4f} | chance={chance_level:.4f}")

        # Per-class probe accuracy: same hook, class-filtered subset of the 200 MI images.
        for cls in [0, 1, 2]:
            cls_subset = [s for s in subset_imgs if s['label'] == cls]
            Xc, yc = _collect_probe_matrix(model, device, hook_name, hook_modules[hook_name], cls_subset)
            acc_c_mean, acc_c_std = _cv_probe_accuracy(Xc, yc)
            class_rows.append({
                'model': model_name,
                'hook': hook_name,
                'true_class': cls,
                'n_features': int(Xc.shape[1]),
                'n_samples': int(len(yc)),
                'accuracy_mean': acc_c_mean,
                'accuracy_std': acc_c_std,
                'chance_level': chance_level,
            })
        del X, y
        gc.collect()
        torch.cuda.empty_cache()
    return pd.DataFrame(rows), pd.DataFrame(class_rows)

pinn_subset = MI_SUBSET
resnet_subset = MI_SUBSET

pinn_probe_df, pinn_class_df = _probe_one_model(
    'D4LensPINN', pinn_model, DEVICE_PINN, pinn_probe_hooks,
    {h: PINN_HOOKS[PINN_PROBE_HOOK_MAP[h]] for h in pinn_probe_hooks}, pinn_subset
 )
resnet_probe_df, resnet_class_df = _probe_one_model(
    'ResNet18', resnet_model, DEVICE_RESNET, resnet_probe_hooks,
    {h: RESNET_HOOKS[h] for h in resnet_probe_hooks}, resnet_subset
 )

probe_df = pd.concat([pinn_probe_df, resnet_probe_df], ignore_index=True)
probe_class_df = pd.concat([pinn_class_df, resnet_class_df], ignore_index=True)

probe_path = os.path.join(OUT_DIR, 'probe_results.csv')
probe_class_path = os.path.join(OUT_DIR, 'probe_class_results.csv')
probe_df.to_csv(probe_path, index=False)
probe_class_df.to_csv(probe_class_path, index=False)

print("\n=== Probe Results ===")
print(probe_df[['model', 'hook', 'n_features', 'accuracy_mean', 'accuracy_std', 'chance_level', 'significant']].to_string(index=False))
print(f"\nChance level = 1/7 = {chance_level:.4f} (non-identity transforms only)")

# Figure: invariance restoration curve
fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=200)

pinn_fig = probe_df[probe_df['model'] == 'D4LensPINN'].set_index('hook').loc[pinn_probe_hooks]
resnet_fig = probe_df[probe_df['model'] == 'ResNet18'].set_index('hook').loc[resnet_probe_hooks]

axes[0].errorbar(range(len(pinn_probe_hooks)), pinn_fig['accuracy_mean'], yerr=pinn_fig['accuracy_std'],
                 marker='o', color='#1a4fa8', linewidth=1.6, capsize=4)
axes[0].axhline(chance_level, color='black', linestyle='--', linewidth=1.0, label='chance (1/7)')
axes[0].axvspan(2 - 0.35, 3 + 0.35, color='#d9d9d9', alpha=0.35, zorder=0)
axes[0].text(2.5, 0.20, 'Architectural\ntransparency\n(chain property)',
             ha='center', va='bottom', fontsize=8, color='#555555')
axes[0].set_xticks(range(len(pinn_probe_hooks)))
axes[0].set_xticklabels(['H08\n(kappa)', 'H09\n(poisson)', 'H11\n(invlens)', 'H12\n(handoff)', 'H13\n(eff3)', 'H15\n(before GAP)', 'H16\n(after GAP)'])
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Probe accuracy')
axes[0].set_title('D4LensPINN — Group Element Decodability')
axes[0].legend(fontsize=8)

axes[1].errorbar(range(len(resnet_probe_hooks)), resnet_fig['accuracy_mean'], yerr=resnet_fig['accuracy_std'],
                 marker='o', color='#9b4f4f', linewidth=1.6, capsize=4)
axes[1].axhline(chance_level, color='black', linestyle='--', linewidth=1.0, label='chance (1/7)')
axes[1].set_xticks(range(len(resnet_probe_hooks)))
axes[1].set_xticklabels(['R04\n(before GAP)', 'R05\n(after GAP)', 'R06\n(fc)'])
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel('Probe accuracy')
axes[1].set_title('ResNet18 — Group Element Decodability')
axes[1].legend(fontsize=8)

fig.suptitle('Invariance Restoration Curve: D4 Group Element Decodability by Layer')
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'figure_b1_invariance_restoration_curve.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure4.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure4.pdf'), bbox_inches='tight')
plt.show()

# Summary interpretation and paper numbers
h11_row = pinn_fig.loc['H11invlens']
h12_row = pinn_fig.loc['H12HANDOFF']
h16_row = pinn_fig.loc['H16aftergap']
interpretation = 'above' if h12_row['accuracy_mean'] > chance_level + 2.0 * std_err_chance else ('at' if abs(h12_row['accuracy_mean'] - chance_level) <= 2.0 * std_err_chance else 'below')
print(f"\nB1 RESULT: H11 probe accuracy = {h11_row['accuracy_mean']:.3f}±{h11_row['accuracy_std']:.3f} (chance={chance_level:.3f})")
print(f"H12 plateau accuracy = {h12_row['accuracy_mean']:.3f}±{h12_row['accuracy_std']:.3f}")
print(f"Interpretation: {interpretation} chance at H12 means routing/chance-level invariance in head.")

summary_path = os.path.join(OUT_DIR, 'findings_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        findings = json.load(f)
else:
    findings = {}
findings.update({
    'probe_h08_acc': float(pinn_fig.loc['H08kappaout', 'accuracy_mean']),
    'probe_h11_acc': float(h11_row['accuracy_mean']),
    'probe_h12_acc': float(h12_row['accuracy_mean']),
    'probe_h16_acc': float(h16_row['accuracy_mean']),
    'probe_h11_std': float(h11_row['accuracy_std']),
    'probe_h12_std': float(h12_row['accuracy_std']),
})
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(findings, f, indent=2)
print(f"Saved {summary_path}")

In [ ]:
# Cell 23 — Phase 1 Checkpoint Comparison (C1 experiment)
import os
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import mannwhitneyu

df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']].copy()

PHASE1_CKPT = os.path.join(CKPT_DIR, 'd4_phase1_best.pth')
summary_path = os.path.join(OUT_DIR, 'findings_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        findings = json.load(f)
else:
    findings = {}

if not os.path.exists(PHASE1_CKPT):
    print('Phase 1 checkpoint not found at:', PHASE1_CKPT)
    print('C1 experiment skipped. Check CKPT_DIR and checkpoint filename.')
    print('If using a different path, update PHASE1_CKPT and rerun.')
    findings.update({'c1_available': False})
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(findings, f, indent=2)
    print(f"Updated {summary_path}")
else:
    print('Phase 1 checkpoint found:', PHASE1_CKPT)

    # Create a fresh model instance; do not modify pinn_model.
    pinn_phase1 = D4LensPINN(num_classes=3)
    pinn_phase1 = pinn_phase1.to(DEVICE_PINN)
    pinn_phase1.eval()

    missing, unexpected = load_ckpt(PHASE1_CKPT, pinn_phase1, strict=False)
    assert missing == [], f'Phase 1 missing keys: {missing}'
    assert unexpected == [], f'Phase 1 unexpected keys: {unexpected}'

    ckpt = torch.load(PHASE1_CKPT, map_location='cpu')
    print(f"Phase 1 epoch: {ckpt.get('epoch', 'NOT STORED')}")
    print(f"Phase 1 val_loss: {ckpt.get('val_loss', 'NOT STORED')}")

    hook_spec = {'H11_inv_lens': pinn_phase1.inv_lens}
    phase1_rows = []

    with torch.no_grad():
        for img_idx, sample in enumerate(MI_SUBSET):
            img = sample['img']
            y_true = int(sample['label'])
            logit_clean, _ = cache_activations(pinn_phase1, img, hook_spec, DEVICE_PINN)

            for g_name in [g for g in GROUP_ELEMS if g != 'e']:
                gx = D4_TRANSFORMS[g_name](img.unsqueeze(0)).squeeze(0)
                _, cache_gx = cache_activations(pinn_phase1, gx, hook_spec, DEVICE_PINN)
                logit_patched = intervention_pass(
                    pinn_phase1, img, cache_gx, 'H11_inv_lens', hook_spec, DEVICE_PINN
                )
                delta = (logit_patched - logit_clean).norm().item()
                phase1_rows.append({
                    'model': 'D4LensPINN_Phase1',
                    'img_idx': img_idx,
                    'true_class': y_true,
                    'group': g_name,
                    'delta_l2': delta,
                })
                del cache_gx, logit_patched, gx

            del logit_clean
            if (img_idx + 1) % 50 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    phase1_df = pd.DataFrame(phase1_rows)
    phase1_csv = os.path.join(OUT_DIR, 'phase1_h11_results.csv')
    phase1_df.to_csv(phase1_csv, index=False)
    print(f"Saved {phase1_csv}")

    p1_h11 = phase1_df['delta_l2'].values
    p2_h11 = df_prim[
        (df_prim['model'] == 'D4LensPINN') &
        (df_prim['hook'] == 'H11_inv_lens') &
        (df_prim['group'] != 'e')
    ]['delta_l2'].values

    stat, p = mannwhitneyu(p2_h11, p1_h11, alternative='greater')
    p1_mean, p1_std = float(np.mean(p1_h11)), float(np.std(p1_h11))
    p2_mean, p2_std = float(np.mean(p2_h11)), float(np.std(p2_h11))

    print(f"Phase 1 H11: mean={p1_mean:.4f}±{p1_std:.4f} (N={len(p1_h11)})")
    print(f"Phase 2 H11: mean={p2_mean:.4f}±{p2_std:.4f} (N={len(p2_h11)})")
    print(f"Mann-Whitney U: stat={stat:.2f}, p={p:.6f}")

    c1_confirmed = bool(p < 0.05)
    if c1_confirmed:
        print('C1 CONFIRMED: physics loss causally increases H11 sensitivity')
    else:
        print('C1 NOT CONFIRMED: physics loss does not change H11 sensitivity')

    print("\nPer-class Phase2>Phase1 tests at H11:")
    for cls in [0, 1, 2]:
        p1_cls = phase1_df[phase1_df['true_class'] == cls]['delta_l2'].values
        p2_cls = df_prim[
            (df_prim['model'] == 'D4LensPINN') &
            (df_prim['hook'] == 'H11_inv_lens') &
            (df_prim['group'] != 'e') &
            (df_prim['true_class'] == cls)
        ]['delta_l2'].values
        if len(p1_cls) and len(p2_cls):
            stat_c, p_c = mannwhitneyu(p2_cls, p1_cls, alternative='greater')
            print(f"  class {cls}: p1={np.mean(p1_cls):.4f}, p2={np.mean(p2_cls):.4f}, p={p_c:.6f}")

    # Figure: Phase1 vs Phase2 H11 comparison
    rot_groups = ['r90', 'r180', 'r270']
    flip_groups = ['flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']

    phase1_df['family'] = np.where(phase1_df['group'].isin(rot_groups), 'rotation', 'reflection')
    p2_df = df_prim[
        (df_prim['model'] == 'D4LensPINN') &
        (df_prim['hook'] == 'H11_inv_lens') &
        (df_prim['group'] != 'e')
    ][['img_idx', 'true_class', 'group', 'delta_l2']].copy()
    p2_df['family'] = np.where(p2_df['group'].isin(rot_groups), 'rotation', 'reflection')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=200)

    # Left: violin by family, phase1 vs phase2
    left_labels = ['P1-rot', 'P2-rot', 'P1-flip', 'P2-flip']
    left_data = [
        phase1_df[phase1_df['family'] == 'rotation']['delta_l2'].values,
        p2_df[p2_df['family'] == 'rotation']['delta_l2'].values,
        phase1_df[phase1_df['family'] == 'reflection']['delta_l2'].values,
        p2_df[p2_df['family'] == 'reflection']['delta_l2'].values,
    ]
    vp = axes[0].violinplot(left_data, showmeans=True, showextrema=False)
    for i, b in enumerate(vp['bodies']):
        b.set_facecolor('#aaaaaa' if i in [0, 2] else '#1a4fa8')
        b.set_alpha(0.6)
    axes[0].set_xticks(np.arange(1, 5))
    axes[0].set_xticklabels(left_labels, rotation=20)
    axes[0].set_title('H11 InverseLensLayer Delta: Phase 1 vs Phase 2')
    axes[0].set_ylabel('Logit Delta (L2)')
    if c1_confirmed:
        y_max = max(np.max(d) if len(d) else 0.0 for d in left_data)
        axes[0].text(2.0, y_max * 1.03, f"p={p:.3e}", ha='center', va='bottom', fontsize=8)

    # Right: per-class bar means, phase1 vs phase2
    cls_x = np.arange(3)
    width = 0.35
    p1_means = [phase1_df[phase1_df['true_class'] == c]['delta_l2'].mean() for c in [0, 1, 2]]
    p2_means = [p2_df[p2_df['true_class'] == c]['delta_l2'].mean() for c in [0, 1, 2]]
    axes[1].bar(cls_x - width/2, p1_means, width=width, color='#aaaaaa', label='Phase 1')
    axes[1].bar(cls_x + width/2, p2_means, width=width, color='#1a4fa8', label='Phase 2')
    axes[1].set_xticks(cls_x)
    axes[1].set_xticklabels(['no-sub', 'sphere', 'vortex'])
    axes[1].set_title('Per-Class H11 Sensitivity: Phase 1 vs Phase 2')
    axes[1].set_ylabel('Mean delta_l2')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    fig_path = os.path.join(OUT_DIR, 'figure_c1_phase_comparison.png')
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()

    if c1_confirmed:
        print(
            f"Paper: Physics regularization causally increases InverseLensLayer geometric sensitivity "
            f"(Phase2 mean={p2_mean:.2f} vs Phase1={p1_mean:.2f}, MWU p={p:.4f}), demonstrating that the "
            f"H11 spike is training-dependent, not purely architectural."
        )
    else:
        print(
            f"Paper: H11 spike is architecturally determined, not training-dependent "
            f"(Phase2={p2_mean:.2f} vs Phase1={p1_mean:.2f}, MWU p={p:.4f}). "
            f"The InverseLensLayer creates geometric sensitivity regardless of whether physics regularization is applied."
        )

    findings.update({
        'c1_available': True,
        'c1_confirmed': c1_confirmed,
        'c1_p_value': float(p),
        'c1_phase1_mean': p1_mean,
        'c1_phase2_mean': p2_mean,
    })
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(findings, f, indent=2)
    print(f"Updated {summary_path}")

    # Cleanup per requirement
    del pinn_phase1
    torch.cuda.empty_cache()
    gc.collect()

## Cell 15B — InverseLensLayer Spike: Headline Finding

In [ ]:
# ── InverseLensLayer spike — headline finding ─────────────────────────────────
# This is the primary mechanistic finding of the experiment.
# Valid as a sensitivity bound because H11 patching can create mixed-condition
# channel states relative to coherent H12 patching.

df_pinn = df_prim[df_prim['model'] == 'D4LensPINN']

print("=" * 60)
print("INVERSELENSLAYER SPIKE — HEADLINE FINDING")
print("=" * 60)

# Physics pipeline profile for r90 (rotations, mean over 200 images)
physics_hooks = ['H09_poisson', 'H10_deflection', 'H11_inv_lens', 'H12_HANDOFF']
print("\nPhysics pipeline (r90, mean ± std over N=200 images):")
for h in physics_hooks:
    vals = df_pinn[(df_pinn['hook'] == h) & (df_pinn['group'] == 'r90')]['delta_l2']
    print(f"  {h:25s}: {vals.mean():.4f} ± {vals.std():.4f}")

print()
inv_r90 = df_pinn[(df_pinn['hook'] == 'H11_inv_lens') & (df_pinn['group'] == 'r90')]['delta_l2']
poi_r90 = df_pinn[(df_pinn['hook'] == 'H09_poisson')  & (df_pinn['group'] == 'r90')]['delta_l2']
hand_r90= df_pinn[(df_pinn['hook'] == 'H12_HANDOFF')  & (df_pinn['group'] == 'r90')]['delta_l2']

amplification_over_poisson = inv_r90.mean() / (poi_r90.mean() + 1e-8)
amplification_over_handoff = inv_r90.mean() / (hand_r90.mean() + 1e-8)
print(f"InverseLensLayer amplification over Poisson: {amplification_over_poisson:.1f}x")
print(f"InverseLensLayer amplification over HANDOFF: {amplification_over_handoff:.1f}x")

# Per group element at InverseLensLayer
print("\nInverseLensLayer delta by group element (mean ± std):")
for g in GROUP_ELEMS:
    if g == 'e':
        continue
    vals = df_pinn[(df_pinn['hook'] == 'H11_inv_lens') & (df_pinn['group'] == g)]['delta_l2']
    fam  = 'ROT ' if g in ['r90','r180','r270'] else 'FLIP'
    print(f"  {fam} {g:15s}: {vals.mean():.4f} ± {vals.std():.4f}")

# Upstream encoder profile
print("\nEncoder depth gradient (r90, confirms geometric compression then recovery):")
enc_hooks = ['H00_preprocess','H01_enc1','H02_enc2','H03_enc3','H04_bot','H08_kappa_out']
for h in enc_hooks:
    vals = df_pinn[(df_pinn['hook'] == h) & (df_pinn['group'] == 'r90')]['delta_l2']
    print(f"  {h:25s}: {vals.mean():.4f}")

print()
print("PAPER STATEMENT (Results §3.1):")
print(f"  'The InverseLensLayer is the dominant site of geometric amplification in")
print(f"  D4LensPINN. Interchange intervention at H11_inv_lens produces a mean logit")
print(f"  delta of {inv_r90.mean():.2f} ± {inv_r90.std():.2f} under 90° rotation,")
print(f"  representing a {amplification_over_poisson:.0f}x amplification over the Poisson/deflection")
print(f"  pipeline ({poi_r90.mean():.2f} ± {poi_r90.std():.2f}) and a {amplification_over_handoff:.1f}x")
print(f"  amplification over the classifier input boundary ({hand_r90.mean():.2f} ± {hand_r90.std():.2f}).'")

print("\nCAVEAT — InverseLensLayer spike validity:")
print(f"  H11 patch injects S_hat(g·x) and R(g·x) while I and kappa remain from x.")
print(f"  This creates a geometrically incoherent 4-channel input.")
print(f"  delta(H11)={inv_r90.mean():.2f} is an UPPER BOUND on true InverseLensLayer sensitivity.")
print(f"  The coherent geometric sensitivity at H12_HANDOFF={hand_r90.mean():.2f} is the lower bound.")
print(f"  Paper: report as 'InverseLensLayer spike [{hand_r90.mean():.2f}, {inv_r90.mean():.2f}] range'.")

# Scope guards for optional globals from prior cells
_ci_lo = ci_lo if 'ci_lo' in globals() else -0.0007
_ci_hi = ci_hi if 'ci_hi' in globals() else 0.0052
_d_kappa = d_kappa if 'd_kappa' in globals() else float('nan')
_d_isr = d_isr if 'd_isr' in globals() else float('nan')
_d_full = d_full if 'd_full' in globals() else float('nan')
if np.isnan(_d_kappa) or np.isnan(_d_isr) or np.isnan(_d_full):
    _ablation_path = os.path.join(OUT_DIR, 'ablation_results.csv')
    if os.path.exists(_ablation_path):
        _ab = pd.read_csv(_ablation_path)
        _d_full = float(_ab['d_full'].mean())
        _d_kappa = float(_ab['d_kappa'].mean())
        _d_isr = float(_ab['d_isr'].mean())

# Save summary for paper
findings_summary = {
    'invlens_r90_mean': float(inv_r90.mean()),
    'invlens_r90_std':  float(inv_r90.std()),
    'poisson_r90_mean': float(poi_r90.mean()),
    'poisson_r90_std':  float(poi_r90.std()),
    'handoff_r90_mean': float(hand_r90.mean()),
    'amplification_over_poisson': float(amplification_over_poisson),
    'amplification_over_handoff': float(amplification_over_handoff),
    'tta_ci_lo': float(_ci_lo),
    'tta_ci_hi': float(_ci_hi),
    'tta_cilo': float(_ci_lo),
    'tta_cihi': float(_ci_hi),
    'kappa_only_ablation': float(_d_kappa),
    'isr_ablation': float(_d_isr),
    'full_ablation': float(_d_full),
}
import json
with open(os.path.join(OUT_DIR, 'findings_summary.json'), 'w') as f:
    json.dump(findings_summary, f, indent=2)
print(f"\nSaved findings_summary.json")

In [ ]:
# Cell 15C - Bootstrap CI refresh (safe for mid-notebook reruns)
import json
from sklearn.metrics import roc_auc_score

_labels_path = os.path.join(OUT_DIR, 'test_labels.npy')
_notta_path = os.path.join(OUT_DIR, 'probs_notta.npy')
_tta_path = os.path.join(OUT_DIR, 'probs_tta.npy')

if 'TEST_LABELS' not in globals():
    if os.path.exists(_labels_path):
        TEST_LABELS = np.load(_labels_path)
    else:
        raise RuntimeError("Missing TEST_LABELS and test_labels.npy not found.")

if 'TEST_PROBS_NO_TTA' not in globals():
    if os.path.exists(_notta_path):
        TEST_PROBS_NO_TTA = np.load(_notta_path)
    else:
        raise RuntimeError("Missing TEST_PROBS_NO_TTA and probs_notta.npy not found.")

if 'TEST_PROBS_TTA' not in globals():
    if os.path.exists(_tta_path):
        TEST_PROBS_TTA = np.load(_tta_path)
    else:
        raise RuntimeError("Missing TEST_PROBS_TTA and probs_tta.npy not found.")


def _macro_auc(_y_int, _p):
    n_cls = int(_p.shape[1])
    y_onehot = np.eye(n_cls, dtype=np.float32)[_y_int.astype(int)]
    return float(roc_auc_score(y_onehot, _p, average='macro', multi_class='ovr'))


y = np.asarray(TEST_LABELS)
p_notta = np.asarray(TEST_PROBS_NO_TTA)
p_tta = np.asarray(TEST_PROBS_TTA)

assert len(y) == len(p_notta) == len(p_tta), "Bootstrap inputs have mismatched row counts."

a_notta = _macro_auc(y, p_notta)
a_tta = _macro_auc(y, p_tta)
delta_auc = a_tta - a_notta

rng = np.random.default_rng(42)
boot_n = 2000
boot_diffs = np.empty(boot_n, dtype=np.float64)
for b in range(boot_n):
    idx = rng.integers(0, len(y), size=len(y))
    boot_diffs[b] = _macro_auc(y[idx], p_tta[idx]) - _macro_auc(y[idx], p_notta[idx])

ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
print(f"AUC no-TTA: {a_notta:.4f}")
print(f"AUC TTA:    {a_tta:.4f}")
print(f"Delta AUC:  {delta_auc:+.4f}")
print(f"Bootstrap 95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]")

with open(os.path.join(OUT_DIR, 'tta_bootstrap_ci.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'auc_notta': float(a_notta),
        'auc_tta': float(a_tta),
        'delta_auc': float(delta_auc),
        'ci_lo': float(ci_lo),
        'ci_hi': float(ci_hi),
        'n': int(len(y)),
        'boot_n': int(boot_n),
    }, f, indent=2)
print("Saved tta_bootstrap_ci.json")

In [ ]:
# Cell 41 — Raw delta reconciliation
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/working/mi_experiment/mi_results_full.csv')

print("=== RAW delta_l2 from CSV ===")
pinn = df[df['model']=='D4LensPINN']
raw = pinn.groupby('hook')['delta_l2'].mean().sort_values(ascending=False)
print(raw.to_string())

print("\n=== Spot-check: do raw numbers match Cell 40 output? ===")
for h in ['H09_poisson','H10_deflection','H11_inv_lens','H12_HANDOFF']:
    v = pinn[pinn['hook']==h]['delta_l2'].mean()
    print(f"  {h:<25} {v:.4f}")

In [ ]:
# Cell 42 — Physics pipeline sensitivity profile
physics_hooks_ordered = [
    'H00_preprocess','H01_enc1','H02_enc2','H03_enc3','H04_bot',
    'H08_kappa_out','H09_poisson','H10_deflection',
    'H11_inv_lens','H12_HANDOFF'
]

rotations = ['r90','r180','r270']
flips = ['flip_h','flip_h_r90','flip_h_r180','flip_h_r270']

print(f"{'Hook':<22} {'All groups':>10} {'Rotations':>10} {'Flips':>10} {'Rot/Flip':>10}")
print("-"*64)
ratio_rows = []
for h in physics_hooks_ordered:
    sub  = pinn[pinn['hook']==h]
    all_ = sub['delta_l2'].mean()
    rot  = sub[sub['group'].isin(rotations)]['delta_l2'].mean()
    flip = sub[sub['group'].isin(flips)]['delta_l2'].mean()
    ratio = rot/flip if flip > 0 else float('nan')
    ratio_rows.append({'hook': h, 'all': all_, 'rot': rot, 'flip': flip, 'ratio': ratio})
    flag = " ← AMPLIFICATION" if h == 'H11_inv_lens' else \
           " ← INVARIANT"     if h == 'H09_poisson'  else \
           " ← RESTORATION"   if h == 'H08_kappa_out' else ""
    print(f"{h:<22} {all_:>10.3f} {rot:>10.3f} {flip:>10.3f} {ratio:>10.3f}{flag}")

ratio_df = pd.DataFrame(ratio_rows)
fig, ax = plt.subplots(figsize=(10, 4), dpi=200)
ax.plot(ratio_df['hook'], ratio_df['ratio'], marker='o', linewidth=1.8, color='#1a4fa8')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0)
ax.set_ylabel('Flip/Rot ratio')
ax.set_xlabel('Hook depth')
ax.set_title('D2: Flip-to-Rotation Sensitivity Ratio by Hook')
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure5.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_DIR, 'figure5.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# Cell 43 - Per-class InverseLensLayer sensitivity
class_names = {0:'no_lensing', 1:'sphere_CDM', 2:'vortex_WDM'}
h11 = pinn[pinn['hook']=='H11_inv_lens']

print("=== Per-Class H11 InverseLensLayer Sensitivity ===")
print(f"{'Class':<20} {'Rot mean':>10} {'Rot std':>10} {'Flip mean':>10} {'Rot/Flip':>10}")
print("-"*60)
perclass_rot_means = {}
for cls in [0,1,2]:
    sub  = h11[h11['true_class']==cls]
    rot_mean = sub[sub['group'].isin(rotations)]['delta_l2'].mean()
    rot_std  = sub[sub['group'].isin(rotations)]['delta_l2'].std()
    flip_mean= sub[sub['group'].isin(flips)]['delta_l2'].mean()
    perclass_rot_means[cls] = float(rot_mean)
    ratio = rot_mean/flip_mean if flip_mean > 0 else float('nan')
    print(f"{class_names[cls]:<20} {rot_mean:>10.3f} {rot_std:>10.3f} {flip_mean:>10.3f} {ratio:>10.3f}")

# Per-class AUC reference
print("\nReference per-class AUC: no=0.9848  sphere=0.9695  vort=0.9814")
# After the per-class means table, replace the interpretation block with:
print("\n" + "=" * 60)
print("CORRECTED D1 INTERPRETATION")
print("=" * 60)
print("OVERALL sensitivity (delta_l2): no-sub HIGHEST, NOT sphere")
print("  -> DO NOT claim sphere has highest overall sensitivity")
print()
print("SPHERE-LOGIT SPECIFIC (dpc_sphere_abs):")
_h11_dpc = df_prim[
    (df_prim['model'] == 'D4LensPINN') &
    (df_prim['hook']  == 'H11_inv_lens') &
    (df_prim['group'] != 'e')
].copy()
_h11_dpc['dpc_sphere_abs'] = _h11_dpc['dpc_1'].abs()
for cls, name in [(0,'no-sub'),(1,'sphere'),(2,'vortex')]:
    v = _h11_dpc[_h11_dpc['true_class']==cls]['dpc_sphere_abs'].mean()
    print(f"  {name}: {v:.4f}")
print()
print("PAPER CLAIM (narrow and correct):")
print("  'H11 patching produces sphere-logit perturbation")
print("   77% larger for sphere images vs no-sub images,")
print("   95% larger vs vortex (KW H=242.32, p<0.001, Bonferroni)'")
print("Mechanistic interpretation:")
print("vortex_WDM (14.548): InverseLensLayer least amplifies its geometric perturbation.")
print("sphere_CDM (22.352): InverseLensLayer most amplifies its geometric perturbation.")
print("  -> sphere_CDM AUC gap (0.9695 vs 0.9848/0.9814) reflects classification difficulty,")
print("     NOT lack of geometric sensitivity. The head receives the strongest geometric")
print("     signal for sphere_CDM but fails to convert it to discriminative features.")
print("  -> This is the key mechanistic finding: maximum geometric amplification does not")
print("     guarantee maximum classification performance.")

# Added nonparametric significance tests for class separation at H11.
from scipy.stats import kruskal, mannwhitneyu

rot_vals = {
    cls: h11[(h11['true_class'] == cls) & (h11['group'].isin(rotations))]['delta_l2'].dropna().values
    for cls in [0, 1, 2]
}
if all(len(v) > 0 for v in rot_vals.values()):
    kw_stat, kw_p = kruskal(rot_vals[0], rot_vals[1], rot_vals[2])
    print(f"\nKruskal-Wallis on rotation-only H11 deltas: H={kw_stat:.4f}, p={kw_p:.6g}")

    pairs = [(0, 1), (0, 2), (1, 2)]
    n_tests = len(pairs)
    print("Pairwise Mann-Whitney U (two-sided, Bonferroni corrected):")
    for a, b in pairs:
        u_stat, p_raw = mannwhitneyu(rot_vals[a], rot_vals[b], alternative='two-sided')
        p_bonf = min(p_raw * n_tests, 1.0)
        tag = "SIGNIFICANT" if p_bonf < 0.05 else "n.s."
        print(f"  {class_names[a]} vs {class_names[b]}: U={u_stat:.1f}, p_raw={p_raw:.6g}, p_bonf={p_bonf:.6g} [{tag}]")
else:
    print("\nKruskal/MWU skipped: missing class samples in rotation subset.")

In [ ]:
# Cell 44 - Poisson FFT algebraic invariance confirmation
print("=== Poisson Solver Invariance (H09) ===")
h09 = pinn[pinn['hook']=='H09_poisson']
h11_raw = pinn[pinn['hook']=='H11_inv_lens']['delta_l2'].mean()
h09_raw = h09['delta_l2'].mean()

print(f"H09_poisson mean delta:   {h09_raw:.4f}")
print(f"H11_inv_lens mean delta:  {h11_raw:.4f}")
if 'e' in set(pinn['group'].astype(str).unique()):
    print("WARNING: identity group 'e' is included in pooled means above.")
    print("Primary mechanistic claims should use non-identity group elements.")
print(f"Amplification ratio:      {h11_raw/h09_raw:.1f}x")
print(f"\nPer group-element breakdown at H09:")
for g in ['e','r90','r180','r270','flip_h','flip_h_r90','flip_h_r180','flip_h_r270']:
    v = h09[h09['group']==g]['delta_l2'].mean()
    print(f"  {g:<15} {v:.4f}")

In [ ]:
# Cell 45 — ResNet null model vs D4LensPINN at classifier input
resnet = df[df['model']=='ResNet18']
pinn_handoff = pinn[pinn['hook']=='H12_HANDOFF']['delta_l2'].mean()

print(f"D4LensPINN H12_HANDOFF (classifier input boundary): {pinn_handoff:.4f}")
print(f"\n{'ResNet hook':<20} {'Mean delta':>12} {'Ratio vs HANDOFF':>18}")
print("-"*52)
for h in ['R00_stem','R01_layer1','R02_layer2','R03_layer3','R04_before_gap','R05_after_gap','R06_fc']:
    v = resnet[resnet['hook']==h]['delta_l2'].mean()
    print(f"{h:<20} {v:>12.4f} {v/pinn_handoff:>18.3f}x")

In [ ]:
# Cell 46 — Coefficient of variation per physics hook
print(f"\n{'Hook':<22} {'Mean':>8} {'Std':>8} {'CV':>8}")
print("-"*48)
for h in physics_hooks_ordered:
    sub = pinn[(pinn['hook']==h) & (pinn['group']=='r90')]
    m, s = sub['delta_l2'].mean(), sub['delta_l2'].std()
    cv = s/m if m > 0 else float('nan')
    flag = " ← image-dependent" if cv > 0.5 else ""
    print(f"{h:<22} {m:>8.3f} {s:>8.3f} {cv:>8.3f}{flag}")

In [ ]:
df = pd.read_csv(os.path.join(OUT_DIR, 'mi_results_full.csv'))
df_prim = df[~df['secondary']]

CLASS_NAMES = {0: 'no-substructure', 1: 'sphere (CDM)', 2: 'vortex (WDM)'}
ROTATIONS   = ['r90', 'r180', 'r270']
REFLECTIONS = ['flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']

# ── 1. H11 InverseLensLayer spike per class ───────────────────────────────────
print("=" * 60)
print("H11 InverseLensLayer delta by class and group family")
print("=" * 60)
h11 = df_prim[(df_prim['model'] == 'D4LensPINN') & 
              (df_prim['hook'] == 'H11_inv_lens') &
              (df_prim['group'] != 'e')]
for cls in [0, 1, 2]:
    sub = h11[h11['true_class'] == cls]
    rot  = sub[sub['group'].isin(ROTATIONS)]['delta_l2'].mean()
    refl = sub[sub['group'].isin(REFLECTIONS)]['delta_l2'].mean()
    print(f"  {CLASS_NAMES[cls]}: rotation={rot:.4f}  reflection={refl:.4f}")

# ── 2. Full physics pipeline profile ─────────────────────────────────────────
print("\n" + "=" * 60)
print("D4LensPINN physics pipeline (r90, mean ± std, N=200)")
print("=" * 60)
pinn_r90 = df_prim[(df_prim['model'] == 'D4LensPINN') & (df_prim['group'] == 'r90')]
for h in ['H00_preprocess','H01_enc1','H02_enc2','H03_enc3',
          'H04_bot','H08_kappa_out',
          'H09_poisson','H10_deflection','H11_inv_lens','H12_HANDOFF']:
    vals = pinn_r90[pinn_r90['hook'] == h]['delta_l2']
    print(f"  {h:25s}: {vals.mean():.4f} ± {vals.std():.4f}")

# ── 3. Rotation vs reflection asymmetry across all head hooks ─────────────────
print("\n" + "=" * 60)
print("Rotation vs Reflection delta — both models, all hooks")
print("=" * 60)
for model_name in ['D4LensPINN', 'ResNet18']:
    sub = df_prim[(df_prim['model'] == model_name) & (df_prim['group'] != 'e')]
    rot_mean  = sub[sub['group'].isin(ROTATIONS)]['delta_l2'].mean()
    refl_mean = sub[sub['group'].isin(REFLECTIONS)]['delta_l2'].mean()
    print(f"  {model_name}: rotation={rot_mean:.4f}  reflection={refl_mean:.4f}  ratio={refl_mean/rot_mean:.2f}x")

# ── 4. GAP drop using repr_dist (once that run finishes) ─────────────────────
# Placeholder — fill in after repr_dist sweep completes
print("\n" + "=" * 60)
print("GAP drop (repr_dist) — run after repr_dist sweep completes")
print("=" * 60)
_repr_path = os.path.join(OUT_DIR, 'repr_dist_results.csv')
if os.path.exists(_repr_path):
    rdf = pd.read_csv(_repr_path)
    for model_name, before_gap, after_gap in [
        ('D4LensPINN', 'H15_before_gap', 'H16_after_gap'),
        ('ResNet18',   'R04_before_gap', 'R05_after_gap'),
    ]:
        sub = rdf[(rdf['model'] == model_name) & (rdf['group'] != 'e')]
        before = sub[sub['hook'] == before_gap]['repr_dist'].mean()
        after  = sub[sub['hook'] == after_gap]['repr_dist'].mean()
        drop   = (before - after) / (before + 1e-8)
        print(f"  {model_name}: before_GAP={before:.4f}  after_GAP={after:.4f}  drop={drop:.2%}")
else:
    print("  repr_dist_results.csv not ready yet — rerun this cell after sweep completes")

In [ ]:
# Guard cell: ensure repr_dist file exists before downstream readers.
_repr_csv = os.path.join(OUT_DIR, 'repr_dist_results.csv')
if not os.path.exists(_repr_csv):
    print("repr_dist_results.csv not found.")
    print("Run the repr_dist sweep before executing the next cells.")
    raise FileNotFoundError(_repr_csv)
print(f"Found repr_dist results: {_repr_csv}")

In [ ]:
rdf = pd.read_csv(os.path.join(OUT_DIR, 'repr_dist_results.csv'))

print("Full repr_dist profile by hook (mean over N=200, non-identity group elements)")
for model_name, hook_order in [
    ('D4LensPINN', list(PINN_HOOKS_PRIMARY.keys())),
    ('ResNet18',   list(RESNET_HOOKS.keys())),
]:
    print(f"\n{model_name}:")
    sub = rdf[(rdf['model'] == model_name) & (rdf['group'] != 'e')]
    for h in hook_order:
        vals = sub[sub['hook'] == h]['repr_dist']
        if len(vals) > 0:
            print(f"  {h:30s}: {vals.mean():.4f} ± {vals.std():.4f}")

In [ ]:
# 1. The anomalous H14b_eff6 dip — is it group-specific or class-specific?
rdf = pd.read_csv(os.path.join(OUT_DIR, 'repr_dist_results.csv'))
h14b = rdf[(rdf['model'] == 'D4LensPINN') & 
           (rdf['hook'] == 'H14b_eff6') & 
           (rdf['group'] != 'e')]
print("H14b_eff6 by group:")
print(h14b.groupby('group')['repr_dist'].mean().sort_values())
print("\nH14b_eff6 by class:")
print(h14b.groupby('true_class')['repr_dist'].mean())

# 2. Full group-element breakdown at key hooks
key_hooks_pinn   = ['H09_poisson','H11_inv_lens','H12_HANDOFF',
                    'H15_before_gap','H16_after_gap']
key_hooks_resnet = ['R04_before_gap','R05_after_gap','R06_fc']
for model_name, hooks in [('D4LensPINN', key_hooks_pinn),
                           ('ResNet18',   key_hooks_resnet)]:
    print(f"\n{model_name} — per group element at key hooks:")
    sub = rdf[(rdf['model'] == model_name) & (rdf['group'] != 'e')]
    pivot = sub[sub['hook'].isin(hooks)].groupby(
        ['hook','group'])['repr_dist'].mean().unstack()
    print(pivot.to_string())

# 3. Vortex anomaly in repr_dist — does it match the delta_l2 pattern?
print("\nH11 repr_dist by class:")
h11r = rdf[(rdf['model'] == 'D4LensPINN') & 
           (rdf['hook'] == 'H11_inv_lens') & 
           (rdf['group'] != 'e')]
print(h11r.groupby('true_class')['repr_dist'].mean())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

rdf = pd.read_csv(os.path.join(OUT_DIR, 'repr_dist_results.csv'))

FAMILY_90  = ['r90', 'r270', 'flip_h_r90', 'flip_h_r270']
FAMILY_180 = ['r180', 'flip_h', 'flip_h_r180']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), dpi=150)

for ax, model_name, hook_order, short_labels in zip(
    axes,
    ['D4LensPINN', 'ResNet18'],
    [list(PINN_HOOKS_PRIMARY.keys()), list(RESNET_HOOKS.keys())],
    [SHORT_LABELS_PINN, SHORT_LABELS_RESNET],
):
    sub = rdf[(rdf['model'] == model_name) & (rdf['group'] != 'e')]
    xs  = list(range(len(hook_order)))

    for family, color, label in [
        (FAMILY_90,  '#1a4fa8', '90°-family (r90, r270, flip_h_r90, flip_h_r270)'),
        (FAMILY_180, '#a81a1a', '180°-family (r180, flip_h, flip_h_r180)'),
    ]:
        means, stds = [], []
        for h in hook_order:
            vals = sub[(sub['hook'] == h) & (sub['group'].isin(family))]['repr_dist']
            means.append(vals.mean() if len(vals) > 0 else np.nan)
            stds.append(vals.std()  if len(vals) > 0 else np.nan)
        ax.errorbar(xs, means, yerr=stds, color=color,
                    linewidth=2, marker='o', markersize=4,
                    capsize=3, label=label, alpha=0.9)

    # Vertical lines
    if model_name == 'D4LensPINN':
        for hook_name, color, text in [
            ('H09_poisson',   'green',      'Poisson\nbottleneck'),
            ('H12_HANDOFF',   'darkorange', 'Classifier\ninput'),
            ('H16_after_gap', 'navy',       'GAP'),
        ]:
            if hook_name in hook_order:
                xi = hook_order.index(hook_name)
                ax.axvline(xi, color=color, linestyle='--', linewidth=1.5)
                ax.text(xi + 0.1, ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 1.0,
                        text, color=color, fontsize=7)
    else:
        for hook_name, color, text in [
            ('R05_after_gap', 'navy', 'GAP'),
        ]:
            if hook_name in hook_order:
                xi = hook_order.index(hook_name)
                ax.axvline(xi, color=color, linestyle='--', linewidth=1.5)
                ax.text(xi + 0.1, 1.5, text, color=color, fontsize=7)

    ax.set_xticks(xs)
    ax.set_xticklabels([short_labels.get(h, h) for h in hook_order],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Repr. Distance (normalized)', fontsize=9)
    ax.set_title(model_name, fontsize=10, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(
    'Geometric Information Flow: 90°-family vs 180°-family\n'
    'D4LensPINN vs ResNet-18 (N=200, repr_dist)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure_family_split.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
# Full repr_dist regeneration
_repr_path = os.path.join(OUT_DIR, 'repr_dist_results.csv')

if os.path.exists(_repr_path):
    _rc = pd.read_csv(_repr_path)
    if _rc['hook'].nunique() >= 15:
        print(f"Full profile exists: {_rc['hook'].nunique()} hooks — skip")
    else:
        print(f"Only {_rc['hook'].nunique()} hooks — need full sweep")
        os.remove(_repr_path)

if not os.path.exists(_repr_path):
    _rows = []
    for _mn, _model, _hooks, _dev in [
        ('D4LensPINN', pinn_model, PINN_HOOKS, DEVICE_PINN),
        ('ResNet18', resnet_model, RESNET_HOOKS, DEVICE_RESNET),
    ]:
        _primary = {k:v for k,v in _hooks.items()
                    if k not in PINN_HOOKS_SECONDARY}
        _model.eval()
        for _ii, _sample in enumerate(MI_SUBSET):
            _x = _sample['img']
            _, _cx = cache_activations(_model, _x, _primary, _dev)
            for _g in GROUP_ELEMS:
                _gx = D4_TRANSFORMS[_g](_x.unsqueeze(0)).squeeze(0)
                _, _cgx = cache_activations(_model, _gx, _primary, _dev)
                for _h in _primary:
                    _ax = _cx.get(_h)
                    _ag = _cgx.get(_h)
                    if _ax is None or _ag is None: continue
                    if isinstance(_ax, tuple): _ax = _ax[0]
                    if isinstance(_ag, tuple): _ag = _ag[0]
                    if isinstance(_ax, torch.Tensor):
                        _af = _ax.float().flatten()
                        _gf = _ag.float().flatten()
                        _n  = _af.norm().item()
                        _d  = (_af - _gf).norm().item()
                        _rows.append({
                            'model':_mn, 'img_idx':_ii,
                            'true_class':_sample['label'],
                            'group':_g, 'hook':_h,
                            'repr_dist': _d/(_n+1e-8),
                        })
                del _cgx
            del _cx
            if _ii % 10 == 0:
                torch.cuda.empty_cache()
                print(f"[{_mn}] {_ii}/{N_SUBSET}")
    
    pd.DataFrame(_rows).to_csv(_repr_path, index=False)
    print("Saved full repr_dist_results.csv")

In [ ]:
# Cell B1B - PCA-equalized group probe (robust to cache schema)
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import os

# Load cached activations
cache_data = np.load(os.path.join(OUT_DIR, 'act_cache.npz'))
meta = pd.read_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'))

seed_local = SEED if 'SEED' in globals() else 42
CHANCE = 1.0 / 7.0
N_PCA = 50  # fixed representational capacity across hooks

# Canonical hook order for reporting; each maps to one or more possible cache keys.
HOOK_KEY_CANDIDATES = {
    'H08_kappa_out': ['H08_kappa_out', 'H08kappaout'],
    'H09_poisson': ['H09_poisson', 'H09poisson'],
    'H11_inv_lens': ['H11_inv_lens', 'H11invlens'],
    'H12_HANDOFF': ['H12_HANDOFF', 'H12HANDOFF'],
    'H13_eff3': ['H13_eff3', 'H13eff3'],
    'H13b_eff4': ['H13b_eff4', 'H13beff4'],
    'H15_before_gap': ['H15_before_gap', 'H15beforegap'],
    'H16_after_gap': ['H16_after_gap', 'H16aftergap'],
}
PROBE_HOOKS_ORDERED = list(HOOK_KEY_CANDIDATES.keys())

# Resolve metadata schema differences across cache versions.
if 'group' in meta.columns:
    group_col = 'group'
elif 'group_name' in meta.columns:
    group_col = 'group_name'
else:
    raise KeyError(f"Neither 'group' nor 'group_name' present in act_cache_meta.csv. Columns: {list(meta.columns)}")

# Use persisted numeric labels when available; otherwise derive them from group names.
if 'group_label' in meta.columns:
    group_labels_all = meta['group_label'].to_numpy()
else:
    non_identity_names = [g for g in sorted(meta[group_col].astype(str).unique()) if g != 'e']
    name_to_label = {g: i for i, g in enumerate(non_identity_names)}
    group_labels_all = meta[group_col].astype(str).map(name_to_label).to_numpy()

group_mask = meta[group_col].astype(str) != 'e'
group_labels = group_labels_all[group_mask.to_numpy()]

print(f"Probe samples: {int(group_mask.sum())} | Chance: {CHANCE:.4f}")
print(f"PCA components: {N_PCA} (fixed across all hooks)\n")

probe_rows = []
for canonical_hook in PROBE_HOOKS_ORDERED:
    # Pick the first cache key that exists for this canonical hook.
    key = next((k for k in HOOK_KEY_CANDIDATES[canonical_hook] if k in cache_data.files), None)
    if key is None:
        print(f"  {canonical_hook:30s}: NOT IN CACHE - skip")
        continue

    X_all = cache_data[key]
    # Keep rows aligned with metadata mask only when cache contains identity rows.
    if X_all.shape[0] == len(meta):
        X = X_all[group_mask.to_numpy()]
    else:
        X = X_all

    if X.shape[0] != group_labels.shape[0]:
        print(f"  {canonical_hook:30s}: SHAPE MISMATCH - skip (X={X.shape[0]}, y={group_labels.shape[0]})")
        continue

    D = X.shape[1]
    n_components = min(N_PCA, D, max(1, X.shape[0] - 1))
    if D > n_components:
        pca = PCA(n_components=n_components, random_state=seed_local)
        X_pca = pca.fit_transform(X)
        var_explained = float(pca.explained_variance_ratio_.sum())
    else:
        X_pca = X
        var_explained = 1.0

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_local)
    accs = []
    for tr, te in cv.split(X_pca, group_labels):
        clf = LogisticRegression(C=0.1, max_iter=500, random_state=seed_local)
        clf.fit(X_pca[tr], group_labels[tr])
        accs.append(accuracy_score(group_labels[te], clf.predict(X_pca[te])))

    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs))
    probe_rows.append({
        'model': 'D4LensPINN',
        'hook': canonical_hook,
        'cache_key': key,
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc,
        'acc': mean_acc,
        'std': std_acc,
        'chance': float(CHANCE),
        'delta_over_chance': float(mean_acc - CHANCE),
        'margin_over_chance': float(mean_acc - CHANCE),
        'n_samples': int(X.shape[0]),
        'raw_features': int(D),
        'raw_dim': int(D),
        'pca_components_used': int(X_pca.shape[1]),
        'pca_dim': int(X_pca.shape[1]),
        'var_explained': var_explained,
        'n_splits': 5,
    })
    above = 'ABOVE CHANCE' if mean_acc > CHANCE * 1.5 else 'near chance'
    print(
        f"  {canonical_hook:30s}: {mean_acc:.4f} +/- {std_acc:.4f} "
        f"({above}, raw_D={D}, pca_D={X_pca.shape[1]}, var={var_explained:.2f}, key={key})"
    )

probe_df = pd.DataFrame(probe_rows)
probe_path = os.path.join(OUT_DIR, 'probe_pca_equalized.csv')
probe_df.to_csv(probe_path, index=False)

print("\nSaved probe_pca_equalized.csv")
print(probe_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print("\nINTERPRETATION:")
print("  If H13 still jumps after PCA equalization:")
print("  -> The jump is REAL, not a capacity artifact")
print("  If H13 drops to near-chance after PCA:")
print("  -> The jump was capacity artifact, not information content")

In [ ]:
# After PCA-equalized probe runs:
probe_df = pd.read_csv(os.path.join(OUT_DIR, 'probe_pca_equalized.csv'))

fig, ax = plt.subplots(figsize=(10, 4), dpi=200)
hooks_ordered = [
    'H08_kappa_out', 'H09_poisson', 'H11_inv_lens',
    'H12_HANDOFF', 'H13_eff3', 'H13b_eff4',
    'H15_before_gap', 'H16_after_gap'
]
short = {
    'H08_kappa_out':'H08\nGroupPool',
    'H09_poisson':'H09\nPoisson',
    'H11_inv_lens':'H11\nInvLens',
    'H12_HANDOFF':'H12\nHANDOFF',
    'H13_eff3':'H13\neff3',
    'H13b_eff4':'H13b\neff4',
    'H15_before_gap':'H15\npre-GAP',
    'H16_after_gap':'H16\npost-GAP',
}

accs  = []
stds  = []
valid = []
for h in hooks_ordered:
    row = probe_df[probe_df['hook']==h]
    if len(row) == 0:
        continue
    acc_col = 'mean_accuracy' if 'mean_accuracy' in row.columns else 'acc'
    std_col = 'std_accuracy' if 'std_accuracy' in row.columns else 'std'
    accs.append(float(row[acc_col]))
    stds.append(float(row[std_col]))
    valid.append(short[h])

xs = np.arange(len(valid))
ax.errorbar(xs, accs, yerr=stds, marker='o', linewidth=2,
            color='#1a4fa8', capsize=4, markersize=6)
ax.axhline(1/7, color='red', linestyle='--', linewidth=1.5,
           label=f'Chance (1/7 = {1/7:.3f})')
ax.axvline(valid.index('H12\nHANDOFF') + 0.5, 
           color='darkorange', linestyle='--', linewidth=1.5,
           label='Classifier input boundary')
ax.set_xticks(xs)
ax.set_xticklabels(valid, fontsize=8)
ax.set_ylabel('Group element probe accuracy (5-fold CV)', fontsize=9)
ax.set_title(
    'Invariance Restoration Curve: Group Element Decodability Per Hook\n'
    '(PCA-equalized to 50 features — controls for capacity confound)',
    fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, 0.6)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure2_invariance_restoration.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved figure2_invariance_restoration.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ORBIT POLYTOPE GEOMETRY — chain-property-immune symmetry metric
# For each image, D4 action creates 8 activation vectors.
# SVD of these 8 vectors measures orbit dimensionality per hook.
# ═══════════════════════════════════════════════════════════════

import numpy as np
from scipy.linalg import svd as scipy_svd

cache_np = np.load(os.path.join(OUT_DIR, 'act_cache.npz'), allow_pickle=True)
meta_df  = pd.read_csv(os.path.join(OUT_DIR, 'act_cache_meta.csv'))

GROUP_ORDER = ['e','r90','r180','r270','flip_h','flip_h_r90','flip_h_r180','flip_h_r270']

HOOKS_POLYTOPE = [
    'H08kappaout','H09poisson','H11invlens',
    'H12HANDOFF','H13eff3','H15beforegap','H16aftergap'
]

polytope_rows = []

print(f"{'Hook':<20} {'mean_vol':>12} {'mean_rank':>10} "
      f"{'std_vol':>10} {'interpretation'}")
print("-" * 70)

for hk in HOOKS_POLYTOPE:
    if hk not in cache_np:
        print(f"{hk:<20}  NOT IN CACHE")
        continue
    
    X_all = cache_np[hk]  # shape (N_images*8, D)
    N_total = len(meta_df)
    N_imgs  = N_total // 8
    
    vols, ranks = [], []
    
    for img_i in range(N_imgs):
        # Get 8 activation vectors for this image (all D4 transforms)
        idx_start = img_i * 8
        idx_end   = idx_start + 8
        orbit = X_all[idx_start:idx_end].astype(np.float32)  # (8, D)
        
        # Center the orbit
        centered = orbit - orbit.mean(axis=0, keepdims=True)
        
        # SVD — singular values measure orbit extent in each direction
        try:
            U, S, Vt = scipy_svd(centered, full_matrices=False)
            # Intrinsic rank: number of significant singular values
            rank = int(np.sum(S > S[0] * 0.01))
            # Pseudo-volume: product of significant singular values
            sig_S = S[S > S[0] * 0.01]
            vol   = float(np.prod(sig_S)) if len(sig_S) > 0 else 0.0
        except Exception:
            rank, vol = 0, 0.0
        
        vols.append(vol)
        ranks.append(rank)
    
    mean_vol  = np.mean(vols)
    std_vol   = np.std(vols)
    mean_rank = np.mean(ranks)
    
    # Interpretation
    if mean_rank <= 1:
        interp = "COLLAPSED (invariant)"
    elif mean_rank <= 3:
        interp = "low-dim (partial)"
    else:
        interp = f"high-dim rank~{mean_rank:.0f}"
    
    print(f"{hk:<20} {mean_vol:>12.4e} {mean_rank:>10.2f} "
          f"{std_vol:>10.4e}  {interp}")
    
    polytope_rows.append({
        'model': 'D4LensPINN', 'hook': hk,
        'mean_vol': mean_vol, 'std_vol': std_vol,
        'mean_rank': mean_rank,
    })

pd.DataFrame(polytope_rows).to_csv(
    os.path.join(OUT_DIR, 'polytope_geometry.csv'), index=False)
print("\nSaved polytope_geometry.csv")
print("\nHYPOTHESIS CHECK:")
print("H08 (GroupPooling): expect mean_rank near 0 (invariant output)")
print("H09 (Poisson):      expect mean_rank near 0 (inherited)")
print("H13 (EfficientNet): expect mean_rank >> 0 (re-encodes geometry)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SPECTRAL ANALYSIS — which frequency bands carry D4 orbit info?
# Tests: does Poisson 1/k² destroy high-frequency orientation info?
# ═══════════════════════════════════════════════════════════════

import torch
import torch.fft as tfft

# We need SPATIAL activation maps — not pooled vectors.
# Load from act_cache.npz if spatial maps were saved,
# OR recompute for just H08 and H09.

# Check if spatial maps exist in cache
_hk_test = 'H08kappaout'
if _hk_test in cache_np:
    X_test = cache_np[_hk_test]
    print(f"H08 shape in cache: {X_test.shape}")
    # If shape is (N, C*H*W) — need to know original spatial dims
    # κ̂ shape: (1, 150, 150) → flattened to 22500
    # Attempt reshape back to spatial
    if X_test.shape[1] == 22500:
        H_sp, W_sp = 150, 150
        C_sp = 1
        print(f"Reshaping to ({C_sp}, {H_sp}, {W_sp}) spatial")
        _reshape_ok = True
    else:
        print(f"Cannot reshape — spatial dims unknown for D={X_test.shape[1]}")
        _reshape_ok = False
else:
    print("H08 not in cache — skip spectral analysis or recompute")
    _reshape_ok = False

if _reshape_ok:
    N_imgs = len(meta_df) // 8
    BANDS = {
        'low':  (0, 0.1),   # k < 10% of Nyquist
        'mid':  (0.1, 0.4), # 10-40%
        'high': (0.4, 1.0)  # 40-100%
    }
    
    spec_rows = []
    for hk, C, H, W in [('H08kappaout', 1, 150, 150),
                          ('H09poisson',  1, 150, 150)]:
        if hk not in cache_np: continue
        X_sp = torch.from_numpy(cache_np[hk].astype(np.float32))
        if X_sp.shape[1] != C*H*W: continue
        X_sp = X_sp.reshape(-1, 8, C, H, W)  # (N_imgs, 8, C, H, W)
        
        # Compute variance across 8 D4 group elements per image
        orbit_var = X_sp.var(dim=1)  # (N_imgs, C, H, W)
        
        # FFT of the variance map
        var_fft = tfft.fft2(orbit_var)           # (N_imgs, C, H, W) complex
        var_pow = var_fft.abs().pow(2)             # power spectrum
        
        # Build radial frequency mask
        ky = torch.fft.fftfreq(H).reshape(H, 1).expand(H, W)
        kx = torch.fft.fftfreq(W).reshape(1, W).expand(H, W)
        kr = (ky**2 + kx**2).sqrt()               # (H, W)
        
        total_pow = var_pow.mean().item()
        row = {'hook': hk, 'total_power': total_pow}
        
        for band_name, (lo, hi) in BANDS.items():
            mask = ((kr >= lo) & (kr < hi)).float()
            band_pow = (var_pow * mask.unsqueeze(0).unsqueeze(0)).mean().item()
            frac = band_pow / (total_pow + 1e-12)
            row[f'{band_name}_frac'] = frac
            print(f"  {hk} {band_name:6s}: {frac:.3f} of total variance power")
        
        spec_rows.append(row)
    
    spec_df = pd.DataFrame(spec_rows)
    spec_df.to_csv(os.path.join(OUT_DIR, 'spectral_band_analysis.csv'), index=False)
    print("\nSaved spectral_band_analysis.csv")
    print("\nINTERPRETATION:")
    print("If H08 high_frac > H09 high_frac:")
    print("  Poisson 1/k² DESTROYS high-frequency orientation info")
    print("  This explains why repr_dist collapses 93% at H09")
    print("If fractions are similar:")
    print("  Inherited equivariance explanation is correct (not spectral)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MECHANISTIC DIVERGENCE UNDER AUC PARITY
# Compare hook-level probe summaries: D4LensPINN vs VanillaLensPINN
# This is valid only when both CSVs expose the same summary schema.
# ═══════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd

# D4 summary from the PCA-equalized probe cell.
d4_candidates = [
    os.path.join('/kaggle/working/mi_experiment', 'probe_pca_equalized.csv'),
    os.path.join(OUT_DIR, 'probe_pca_equalized.csv'),
]
# Vanilla summary from the matching probe cell.
vanilla_candidates = [
    os.path.join('/kaggle/working/mi_experiment_vanilla', 'vanilla_probe_summary.csv'),
    os.path.join('/kaggle/working/mi_experiment', 'vanilla_probe_summary.csv'),
    os.path.join('/kaggle/input/datasets/[NAME]103856/vanilla-csv', 'vanilla_probe_summary.csv'),
]

d4_path = next((path for path in d4_candidates if os.path.exists(path)), None)
van_path = next((path for path in vanilla_candidates if os.path.exists(path)), None)

assert d4_path is not None, f"D4 summary not found. Tried: {d4_candidates}"
assert van_path is not None, f"Vanilla summary not found. Tried: {vanilla_candidates}"


def _normalize_probe_summary(frame: pd.DataFrame, model_name: str) -> pd.DataFrame:
    normalized = frame.copy()
    if 'mean_accuracy' not in normalized.columns and 'acc' in normalized.columns:
        normalized['mean_accuracy'] = normalized['acc']
    if 'std_accuracy' not in normalized.columns and 'std' in normalized.columns:
        normalized['std_accuracy'] = normalized['std']
    if 'chance' not in normalized.columns:
        normalized['chance'] = np.nan
    if 'delta_over_chance' not in normalized.columns and {'mean_accuracy', 'chance'}.issubset(normalized.columns):
        normalized['delta_over_chance'] = normalized['mean_accuracy'] - normalized['chance']
    if 'margin_over_chance' not in normalized.columns and 'delta_over_chance' in normalized.columns:
        normalized['margin_over_chance'] = normalized['delta_over_chance']
    if 'n_samples' not in normalized.columns:
        normalized['n_samples'] = np.nan
    if 'raw_features' not in normalized.columns and 'raw_dim' in normalized.columns:
        normalized['raw_features'] = normalized['raw_dim']
    if 'pca_components_used' not in normalized.columns and 'pca_dim' in normalized.columns:
        normalized['pca_components_used'] = normalized['pca_dim']
    if 'n_splits' not in normalized.columns:
        normalized['n_splits'] = np.nan
    normalized['model'] = model_name
    return normalized


def _canonicalize_hooks(frame: pd.DataFrame) -> pd.DataFrame:
    canonical = frame.copy()
    canonical['hook'] = canonical['hook'].replace({'H12_handoff': 'H12_HANDOFF'})
    return canonical


d4_df = _canonicalize_hooks(_normalize_probe_summary(pd.read_csv(d4_path), 'D4LensPINN'))
van_df = _canonicalize_hooks(_normalize_probe_summary(pd.read_csv(van_path), 'VanillaLensPINN'))

common = d4_df[['hook', 'mean_accuracy', 'std_accuracy', 'chance', 'delta_over_chance', 'n_samples', 'raw_features', 'pca_components_used', 'n_splits']].merge(
    van_df[['hook', 'mean_accuracy', 'std_accuracy', 'chance', 'delta_over_chance', 'n_samples', 'raw_features', 'pca_components_used', 'n_splits']],
    on='hook',
    how='inner',
    suffixes=('_d4', '_van'),
)

assert not common.empty, f"No overlapping hooks between {d4_path} and {van_path}"

common['delta_mean_accuracy'] = common['mean_accuracy_d4'] - common['mean_accuracy_van']
common['delta_std_accuracy'] = common['std_accuracy_d4'] - common['std_accuracy_van']
common['chance_match'] = np.isclose(common['chance_d4'], common['chance_van'], equal_nan=True)
common['n_samples_match'] = np.isclose(common['n_samples_d4'].fillna(-1), common['n_samples_van'].fillna(-1))
common['pca_components_match'] = np.isclose(common['pca_components_used_d4'].fillna(-1), common['pca_components_used_van'].fillna(-1))

focus_hooks = ['H09_poisson', 'H12_HANDOFF']
focus_common = common[common['hook'].isin(focus_hooks)].copy()
if not focus_common.empty:
    focus_common = focus_common.sort_values('hook')
    focus_common['delta_mean_accuracy'] = focus_common['mean_accuracy_d4'] - focus_common['mean_accuracy_van']
    focus_common['delta_std_accuracy'] = focus_common['std_accuracy_d4'] - focus_common['std_accuracy_van']

compare_path = os.path.join(OUT_DIR, 'mechanistic_probe_comparison.csv')
common.to_csv(compare_path, index=False)
if not focus_common.empty:
    focus_path = os.path.join(OUT_DIR, 'mechanistic_probe_comparison_focus.csv')
    focus_common.to_csv(focus_path, index=False)

print(f"D4 summary: {d4_path}")
print(f"Vanilla summary: {van_path}")
print(f"Saved comparison: {compare_path}\n")
print(f"{'Hook':<20} {'D4_acc':>10} {'Van_acc':>10} {'delta':>10} {'chance':>10}")
print('-' * 66)
display_df = focus_common if not focus_common.empty else common.sort_values('hook')
for _, row in display_df.iterrows():
    print(
        f"{row['hook']:<20} {row['mean_accuracy_d4']:>10.4f} {row['mean_accuracy_van']:>10.4f} "
        f"{row['delta_mean_accuracy']:>10.4f} {row['chance_d4']:>10.4f}"
    )

if not focus_common.empty:
    print('\nFocused H09/H12 delta summary:')
    print(focus_common[['hook', 'mean_accuracy_d4', 'mean_accuracy_van', 'delta_mean_accuracy', 'std_accuracy_d4', 'std_accuracy_van']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print("\nINTERPRETATION:")
print("  Compare mean_accuracy on the same hook names, not rank columns from unrelated CSVs.")
print("  If the hooks differ in raw_features or pca_components_used, that is a comparison warning.")
print("  The valid comparison here is D4 vs Vanilla probe summary at shared hooks.")
print("  H09/H12 are highlighted because they are the shared hooks used by the probe cells.")

In [ ]:
# Cell B1C - D4-only probe (H09/H12), Vanilla-comparable protocol
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import os

# Guards: this cell reads the shared activation cache from Cell 23.
if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working/mi_experiment'

cache_path = os.path.join(OUT_DIR, 'act_cache.npz')
meta_path = os.path.join(OUT_DIR, 'act_cache_meta.csv')
assert os.path.exists(cache_path), 'act_cache.npz not found - run Cell 23 first'
assert os.path.exists(meta_path), 'act_cache_meta.csv not found - run Cell 23 first'

cache_data = np.load(cache_path)
meta = pd.read_csv(meta_path)

N_PCA = 100
seed_local = SEED if 'SEED' in globals() else 42
CHANCE = 1.0 / 7.0

# Match D4 hooks to cache keys across naming variants.
HOOK_KEY_CANDIDATES = {
    'H09_poisson': ['H09_poisson', 'H09poisson'],
    'H12_HANDOFF': ['H12_HANDOFF', 'H12HANDOFF'],
}

# Metadata schema compatibility: some caches use group, others group_name.
if 'group' in meta.columns:
    group_col = 'group'
elif 'group_name' in meta.columns:
    group_col = 'group_name'
else:
    raise KeyError(
        "Neither 'group' nor 'group_name' exists in act_cache_meta.csv. "
        f"Columns: {list(meta.columns)}"
    )

if 'group_label' in meta.columns:
    y_all = meta['group_label'].to_numpy()
else:
    non_identity_names = [g for g in sorted(meta[group_col].astype(str).unique()) if g != 'e']
    name_to_label = {g: i for i, g in enumerate(non_identity_names)}
    y_all = meta[group_col].astype(str).map(name_to_label).to_numpy()

group_mask = meta[group_col].astype(str).to_numpy() != 'e'
y = y_all[group_mask]

print('D4 probe from cached activations (H09/H12)')
print(f'Samples: {len(y)} | Chance: {CHANCE:.4f} | PCA cap: {N_PCA}')
print(f"{'Hook':<16} {'Acc':>8} {'Std':>8} {'RawDim':>8} {'PCADim':>8} {'VarExp':>8}")
print('-' * 66)

rows = []
for canonical_hook, key_candidates in HOOK_KEY_CANDIDATES.items():
    key = next((k for k in key_candidates if k in cache_data.files), None)
    if key is None:
        print(f"{canonical_hook:<16} {'MISSING':>8} {'-':>8} {'-':>8} {'-':>8} {'-':>8}")
        continue

    X_all = cache_data[key]

    # If cache includes identity rows, align with metadata mask.
    if X_all.shape[0] == len(meta):
        X = X_all[group_mask]
    else:
        X = X_all

    if X.shape[0] != y.shape[0]:
        print(f"{canonical_hook:<16} {'SKIP':>8} {'-':>8} {'-':>8} {'-':>8} {'-':>8}  (shape mismatch)")
        continue

    raw_dim = int(X.shape[1])
    pca_dim = min(N_PCA, raw_dim, max(1, X.shape[0] - 1))

    if raw_dim > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=seed_local)
        X_use = pca.fit_transform(X)
        var_exp = float(pca.explained_variance_ratio_.sum())
    else:
        X_use = X
        var_exp = 1.0

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed_local)
    accs = []
    for tr, te in cv.split(X_use, y):
        clf = LogisticRegression(C=0.1, max_iter=500, random_state=seed_local)
        clf.fit(X_use[tr], y[tr])
        accs.append(accuracy_score(y[te], clf.predict(X_use[te])))

    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs))

    rows.append({
        'model': 'D4LensPINN',
        'hook': canonical_hook,
        'cache_key': key,
        'acc': mean_acc,
        'std': std_acc,
        'chance': CHANCE,
        'margin_over_chance': mean_acc - CHANCE,
        'n_samples': int(X.shape[0]),
        'raw_dim': raw_dim,
        'pca_dim': int(X_use.shape[1]),
        'var_explained': var_exp,
        'n_splits': 5,
        'seed': int(seed_local),
    })

    print(
        f"{canonical_hook:<16} {mean_acc:>8.4f} {std_acc:>8.4f} "
        f"{raw_dim:>8d} {int(X_use.shape[1]):>8d} {var_exp:>8.3f}"
    )

if len(rows) == 0:
    print('\nNo valid hooks were evaluated. Check cache keys and metadata alignment.')
else:
    d4_probe_df = pd.DataFrame(rows)
    out_csv = os.path.join(OUT_DIR, 'probe_d4_h09_h12_pca100.csv')
    d4_probe_df.to_csv(out_csv, index=False)
    print(f'\nSaved: {out_csv}')

    # Optional comparison if a Vanilla CSV already exists.
    vanilla_candidates = [
        os.path.join(OUT_DIR, 'probe_vanilla_h09_h12_pca100.csv'),
        os.path.join(OUT_DIR, 'vanilla_probe_summary.csv'),
        os.path.join('/kaggle/input/datasets/[NAME]103856/vanilla-csv', 'vanilla_probe_summary.csv'),
    ]
    vanilla_path = next((p for p in vanilla_candidates if os.path.exists(p)), None)

    if vanilla_path is not None:
        vdf = pd.read_csv(vanilla_path).copy()
        if 'hook' not in vdf.columns:
            print(f'Vanilla CSV found but missing hook column: {vanilla_path}')
        else:
            # Normalize Vanilla to the same schema as the D4 probe CSV.
            if 'mean_accuracy' in vdf.columns and 'acc' not in vdf.columns:
                vdf['acc'] = vdf['mean_accuracy']
            if 'std_accuracy' in vdf.columns and 'std' not in vdf.columns:
                vdf['std'] = vdf['std_accuracy']
            if 'delta_over_chance' in vdf.columns and 'margin_over_chance' not in vdf.columns:
                vdf['margin_over_chance'] = vdf['delta_over_chance']
            if 'raw_features' in vdf.columns and 'raw_dim' not in vdf.columns:
                vdf['raw_dim'] = vdf['raw_features']
            if 'pca_components_used' in vdf.columns and 'pca_dim' not in vdf.columns:
                vdf['pca_dim'] = vdf['pca_components_used']
            if 'n_splits' not in vdf.columns:
                vdf['n_splits'] = 5
            if 'chance' not in vdf.columns:
                vdf['chance'] = CHANCE
            if 'model' not in vdf.columns:
                vdf['model'] = 'VanillaLensPINN'
            if 'cache_key' not in vdf.columns:
                vdf['cache_key'] = pd.NA
            if 'var_explained' not in vdf.columns:
                vdf['var_explained'] = pd.NA

            vanilla_out = os.path.join(OUT_DIR, 'probe_vanilla_h09_h12_pca100.csv')
            vdf.to_csv(vanilla_out, index=False)

            merged = d4_probe_df[['hook', 'acc', 'std', 'chance', 'margin_over_chance', 'n_samples', 'raw_dim', 'pca_dim', 'var_explained', 'n_splits']].merge(
                vdf[['hook', 'acc', 'std', 'chance', 'margin_over_chance', 'n_samples', 'raw_dim', 'pca_dim', 'var_explained', 'n_splits']].rename(
                    columns={
                        'acc': 'vanilla_acc',
                        'std': 'vanilla_std',
                        'chance': 'vanilla_chance',
                        'margin_over_chance': 'vanilla_margin_over_chance',
                        'n_samples': 'vanilla_n_samples',
                        'raw_dim': 'vanilla_raw_dim',
                        'pca_dim': 'vanilla_pca_dim',
                        'var_explained': 'vanilla_var_explained',
                        'n_splits': 'vanilla_n_splits',
                    }
                ),
                on='hook',
                how='inner'
            )
            merged['d4_minus_vanilla'] = merged['acc'] - merged['vanilla_acc']
            print(f'\nComparison vs Vanilla from: {vanilla_path}')
            print(f'Saved normalized Vanilla CSV: {vanilla_out}')
            print(merged.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    else:
        print('\nNo Vanilla CSV found yet. Run the Vanilla probe and save to one of:')
        for p in vanilla_candidates:
            print(f'  - {p}')

## Cell 16 — Zip Experiment Outputs

In [ ]:
import zipfile, pathlib

# Zip everything produced by this experiment:
#   CSVs, figures (png/pdf), JSON indices, npy arrays
_out_zip = '/kaggle/working/mi_experiment_outputs3.zip'
with zipfile.ZipFile(_out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(pathlib.Path(OUT_DIR).rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))

_size_mb = pathlib.Path(_out_zip).stat().st_size / 1e6
print(f"Outputs zip: {_out_zip}  ({_size_mb:.1f} MB)")
print("Contents:")
with zipfile.ZipFile(_out_zip, 'r') as zf:
    for info in sorted(zf.infolist(), key=lambda x: x.filename):
        print(f"  {info.filename:<55s}  {info.file_size/1e3:>8.1f} KB")


## Cell 17 — Zip Kaggle Working Datasets

In [ ]:
import zipfile, pathlib

# Zip all dataset files in /kaggle/working/ that are NOT part of the
# mi_experiment outputs (checkpoint files, input data, etc.)
_ds_zip   = '/kaggle/working/mi_datasets.zip'
if '_out_zip' not in globals():
    _out_zip = '/kaggle/working/mi_experiment_outputs3.zip'
_excl_dir = pathlib.Path(OUT_DIR).resolve()
_excl_zip = pathlib.Path(_out_zip).resolve()

with zipfile.ZipFile(_ds_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(pathlib.Path('/kaggle/working').rglob('*')):
        if not f.is_file():
            continue
        fr = f.resolve()
        # Exclude experiment outputs dir and the zip we just made
        if str(fr).startswith(str(_excl_dir)):
            continue
        if fr == _excl_zip or fr == pathlib.Path(_ds_zip).resolve():
            continue
        zf.write(f, arcname=f.relative_to('/kaggle/working'))

_ds_size_mb = pathlib.Path(_ds_zip).stat().st_size / 1e6
print(f"Datasets zip: {_ds_zip}  ({_ds_size_mb:.1f} MB)")
print("Contents:")
with zipfile.ZipFile(_ds_zip, 'r') as zf:
    for info in sorted(zf.infolist(), key=lambda x: x.filename):
        print(f"  {info.filename:<55s}  {info.file_size/1e3:>8.1f} KB")
